# RSNA Knee — Fixed GPU Candidate Lab

**English · Same reference model stages · CPU-only replay of saved outputs**

This notebook turns the supplied public 0.942 model family into a reproducible,
fixed-compute experiment. **0.990 and rank 2–3 are goals, not verified results.**
This file has no measured Kaggle score. Its original blend remains the default
because no independent full-ensemble evidence establishes a better alternative.

### What the GPU constraint means

- In `inference` mode, the 22 model-stage source blocks are identical, in order,
  to `RSNA_Knee_Audited_Ensemble.ipynb` with Global96 disabled. Checkpoints,
  preprocessing, inference views, resolution and existing precision are preserved.
- The four candidate CSVs use outputs from that **one existing inference run**.
  They add **zero model forward passes**, no training and no extra model.
- In `replay_cpu` mode, the model blocks are skipped. Set Kaggle Accelerator to
  **None** and reuse a saved prediction bundle for the **same cohort**.
- Saving and sorting arrays still take CPU time and disk I/O. Identical billed
  GPU-hours, peak system memory or end-to-end runtime are **not measured or promised**.

### Run inference

1. Copy the supplied fast/restructured Kaggle notebook to retain its inputs,
   import this file, and verify the Input panel using the final checklist.
2. Select **T4 ×2**, Internet **OFF**. Leave `mode='inference'`.
3. Leave `submission_candidate='reference'` for the inherited blend. To test the
   new hypothesis explicitly, use `'reference_probability_family'`. Its gain is unmeasured.
4. Run all cells in a clean session. Submit `/kaggle/working/submission.csv` only
   after `run_complete.json` confirms its final SHA256 and all required families.
5. Save the small `fixed_compute_bundle/<run-id>/` output for subsequent CPU analysis.

### Replay saved scores without GPU

Set `mode='replay_cpu'`, `replay_bundle` to the saved `predictions.npz`, and
`competition_root` to its matching current cohort metadata directory. Keep the
adjacent `predictions.json`. Set Accelerator **None**, then Run All.
The copied `metadata/` directory is sufficient for historical same-cohort analysis.
Replay writes candidate CSVs and `replay_complete.json`; it does not publish a
competition `submission.csv`. No MRI images or checkpoint weights are required.

**Kaggle replaces the three visible test studies during scoring. A saved preview
bundle cannot substitute for the hidden test. Kaggle must still run the base
inference pipeline for each scored submission.** Generating four CSVs during one
run does not make four separately scored submissions free of Kaggle reruns.

Sources: [official overview](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/overview),
[official data](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data),
[public original V32](https://www.kaggle.com/code/mattiaangeli/bend-the-knee-to-speedy-raptors-the-original?scriptVersionId=351525366).


## 1. Mode, candidate and hard limits


In [ ]:
FIXED_RUN = {
    'mode': 'inference',                  # 'inference' or 'replay_cpu'
    'competition_root': '',                # blank = official Kaggle competition mount
    'replay_bundle': '',                   # required only for replay_cpu
    'output_root': '/kaggle/working',
    'submission_candidate': 'reference',   # see the four frozen candidates below
    'fusion_recipe': '',                   # optional independently validated recipe JSON
    'enable_global96': False,              # must remain False
    'run_training_audit': False,            # must remain False
    'add_tta': False,                       # must remain False
    'add_training': False,                  # must remain False
}


In [ ]:
NOTEBOOK_SUPPORT_SOURCE = r'''"""CPU-only notebook control for the fixed-compute RSNA candidate experiment.

The stage runner executes only the explicitly pinned reference stages, in
order, in inference mode. Replay mode does not execute their source strings.
This is a source budget contract, not a measured wall-clock-time guarantee.
"""
from __future__ import annotations

import hashlib
import linecache
from pathlib import Path
import uuid


ALLOWED_CANDIDATES = (
    'reference', 'reference_probability_family',
    'reference_no_calibrator', 'reference_rad_fracture',
)


def validate_fixed_config(config):
    expected = {
        'mode', 'competition_root', 'replay_bundle', 'output_root',
        'submission_candidate', 'fusion_recipe', 'enable_global96',
        'run_training_audit', 'add_tta', 'add_training',
    }
    if set(config) != expected:
        raise ValueError(f'Unexpected or missing configuration fields: {set(config) ^ expected}')
    if config['mode'] not in {'inference', 'replay_cpu'}:
        raise ValueError('mode must be inference or replay_cpu')
    for key in ('enable_global96', 'run_training_audit', 'add_tta', 'add_training'):
        if config[key] is not False:
            raise ValueError(f'{key} must remain False in this fixed-compute notebook')
    if config['submission_candidate'] not in ALLOWED_CANDIDATES:
        raise ValueError('Choose one of the four frozen candidates')
    for key in ('competition_root', 'replay_bundle', 'output_root', 'fusion_recipe'):
        if not isinstance(config[key], str):
            raise ValueError(f'{key} must be a path string')
    if not config['output_root'].strip():
        raise ValueError('output_root is required')
    if config['mode'] == 'inference' and Path(config['output_root']) != Path('/kaggle/working'):
        raise ValueError('The pinned inference stages use /kaggle/working')
    if config['mode'] == 'replay_cpu' and not Path(config['replay_bundle']).is_file():
        raise FileNotFoundError('CPU replay needs a previously exported predictions.npz and its JSON sidecar')
    if config['fusion_recipe'] and not Path(config['fusion_recipe']).is_file():
        raise FileNotFoundError('The selected independently validated fusion recipe is absent')
    return dict(config)


def resolve_metadata_root(explicit=''):
    roots = [Path(explicit)] if explicit else [
        Path('/kaggle/input/rsna-knee-abnormality-detection'),
        Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
    ]
    found = {str(root.resolve()): root.resolve() for root in roots
             if all((root / name).is_file() for name in
                    ('test.csv', 'test_series.csv', 'sample_submission.csv'))}
    if len(found) != 1:
        raise FileNotFoundError('Supply one current cohort metadata directory with all three competition CSVs')
    return next(iter(found.values()))


def metadata_paths(root):
    return {name: Path(root) / name for name in
            ('test.csv', 'test_series.csv', 'sample_submission.csv')}


def make_stage_runner(namespace, config, reference_manifest):
    """Pin both source and execution order while preserving inspect.getsource."""
    expected = reference_manifest['model_code_cells']
    completed = []

    def execute_pinned_stage(source, original_index, expected_sha):
        if config['mode'] != 'inference':
            raise RuntimeError('A model stage cannot execute in replay_cpu mode')
        if len(completed) >= len(expected):
            raise RuntimeError('A reference stage was executed more than once')
        contract = expected[len(completed)]
        digest = hashlib.sha256(source.encode('utf-8')).hexdigest()
        if (original_index != contract['reference_cell']
                or expected_sha != contract['sha256'] or digest != expected_sha):
            raise RuntimeError('Reference model stage changed, repeated, or executed out of order')
        # Child-worker builders call inspect.getsource on inherited functions.
        # A unique linecache entry makes that work for compiled literal source.
        filename = f'<rsna_fixed_stage_{original_index:03d}_{digest[:12]}>'
        linecache.cache[filename] = (len(source), None, source.splitlines(keepends=True), filename)
        exec(compile(source, filename, 'exec', dont_inherit=True), namespace)
        completed.append({'reference_cell': original_index, 'sha256': digest})

    return execute_pinned_stage, completed


def new_output_directory(root):
    directory = Path(root) / 'fixed_compute_bundle' / uuid.uuid4().hex[:16]
    directory.mkdir(parents=True, exist_ok=False)
    return directory


def candidate_differences(frames, labels):
    """Measure changed ranks, without interpreting unlabeled scores as AUC."""
    import numpy as np
    base = frames['reference'][labels].to_numpy(dtype=np.float64)
    records = []
    for name, frame in frames.items():
        values = frame[labels].to_numpy(dtype=np.float64)
        for j, label in enumerate(labels):
            records.append({
                'candidate': name, 'label': label,
                'changed_study_scores': int(np.count_nonzero(base[:, j] != values[:, j])),
                'max_abs_score_change': float(np.max(np.abs(base[:, j] - values[:, j]))),
                'auc': None, 'auc_status': 'unavailable_without_truth',
            })
    return records
'''
import importlib.util as _fixed_importlib
import sys as _fixed_sys
from pathlib import Path
_fixed_module_path = Path(FIXED_RUN['output_root']) / 'fixed_compute_notebook_support.py'
_fixed_module_path.parent.mkdir(parents=True, exist_ok=True)
_fixed_module_path.write_text(NOTEBOOK_SUPPORT_SOURCE)
_fixed_spec = _fixed_importlib.spec_from_file_location('fixed_compute_notebook_support', _fixed_module_path)
_fixed_module = _fixed_importlib.module_from_spec(_fixed_spec)
_fixed_sys.modules[_fixed_spec.name] = _fixed_module
_fixed_spec.loader.exec_module(_fixed_module)


In [ ]:
from fixed_compute_notebook_support import (validate_fixed_config, resolve_metadata_root,
    metadata_paths, make_stage_runner, new_output_directory, candidate_differences)
FIXED_RUN = validate_fixed_config(FIXED_RUN)
FIXED_BUDGET = {'schema_version': 1, 'reference_notebook': 'RSNA_Knee_Audited_Ensemble.ipynb', 'reference_notebook_sha256': '0ba98b5cd5cda1566ee12f967fd5b83d9484f263d8800d9da8183a7634ca2b93', 'reference_configuration': {'mode': 'submission', 'enable_global96': False, 'parallel_coat_readers': True}, 'scope': 'ordered source parity; no empirical timing or performance claim', 'model_code_cells': [{'reference_cell': 6, 'sha256': '789f88e22c78c5953428ef0885ff70b7107d06169ca861bf09408c3e13e5c443', 'bytes': 23257, 'first_line': 'import os'}, {'reference_cell': 8, 'sha256': '5815eed58685ad17392c9c5b13dfb0fb43fb54ec32b7834f4c2c66439cd3ea5a', 'bytes': 7450, 'first_line': 'import numpy as np'}, {'reference_cell': 10, 'sha256': 'ed919119ee53bac62f8fcabbe0f5b71ab5e42dddb638a20bd7de7f19070d99bb', 'bytes': 16266, 'first_line': 'from __future__ import annotations'}, {'reference_cell': 12, 'sha256': '2fcf3a2e0a3d7e52ac09e60015be1e5a0ee45e147935c43fe848f5a10e8eeab5', 'bytes': 4327, 'first_line': 'def log(msg):'}, {'reference_cell': 13, 'sha256': 'a773890aca786656f025af0f008fd97bcd0736eb143698d71cd16448bb8fce76', 'bytes': 5560, 'first_line': "HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']"}, {'reference_cell': 14, 'sha256': 'a554f7d0f44a197e371a0ce3ce734466c2658bae27c09d6ffd7855cb4210f1a5', 'bytes': 740, 'first_line': 'def pick_slots(series_df, plane_map):'}, {'reference_cell': 15, 'sha256': 'a9ca46d25764c895248403e162605da09d9aeae03aad342058571b31005705e7', 'bytes': 6793, 'first_line': 'ORDER_TAGS = [(32, 50), (32, 55), (32, 19), (40, 48)]'}, {'reference_cell': 16, 'sha256': 'ba17bd3298fc57f07ef581794ee274b5d4990b62a999086881ed6f57bcd882aa', 'bytes': 198, 'first_line': 'def normalise_laterality(img, plane, lat):'}, {'reference_cell': 17, 'sha256': 'a1334e4b2174f1512d3dcd7f46ba257d698b83a6e399b8c1e6c2fa0901f29b8a', 'bytes': 3831, 'first_line': 'def build_cache(slot_map, plane_map, lat_map, tag):'}, {'reference_cell': 18, 'sha256': '6878773f8ffa7883670b9bfceed886f80465514cea3ab30057c7378406531507', 'bytes': 1281, 'first_line': 'class SlotHead(nn.Module):'}, {'reference_cell': 19, 'sha256': '17434f44700a37e344f33040cf12618806370b438c0f2b2f8fe3caa8b42af065', 'bytes': 1170, 'first_line': 'class Model(nn.Module):'}, {'reference_cell': 20, 'sha256': '0cee81943e6bfbb987380cf9c88b2ed961acde926d96211154c4b2946f6e849e', 'bytes': 1890, 'first_line': "def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):"}, {'reference_cell': 21, 'sha256': 'a816bbcb91c3f8645f847d921a7e387df9ba7301c546ed151299a050c6e1e56c', 'bytes': 34056, 'first_line': 'SHARED_DINO_PREFIX_LAYERS = 6'}, {'reference_cell': 22, 'sha256': '9006955666c6c5f89bd87d606b83d5fba2d446c42168e223491e11b8b8f95102', 'bytes': 1526, 'first_line': 'def write_submission(pred, studies, test_df, path):'}, {'reference_cell': 23, 'sha256': 'cd7a8762ef90ccf507eaf4588e350402a4f8b1f0f58b7f6687b135eee684d357', 'bytes': 127, 'first_line': "rsna_phase('dinov2', 'START')"}, {'reference_cell': 25, 'sha256': '6fffd33cf27daf23afcae0616f5fac9ffc7e869e08cd7827f3d4f141276f2611', 'bytes': 1631, 'first_line': "if globals().get('_DINOV2_MATCHED_MEMBERS') != 20:"}, {'reference_cell': 26, 'sha256': 'f1fa6aa2130ce82d2efb84793f47c50c97b71e3e61c68ffc5840c4431e7a5f5e', 'bytes': 5151, 'first_line': "SERIES_ROOT = COMP / 'test_series'"}, {'reference_cell': 27, 'sha256': 'a0f7ec9a6a26310a246fef4d3349a1af1e853e99cd2ecbc2b819d96aa9071a1e', 'bytes': 17971, 'first_line': 'N_SLOT_TYPES, MASK_IDX = (6, 0)'}, {'reference_cell': 28, 'sha256': 'f9c7c32140b051df0fab9511bffe97abd7ed806770cc320ee1ad2fd5def40e7f', 'bytes': 10711, 'first_line': "AMP_PREF = 'bf16'  # BTKD precision retained, including T4 execution; no unvalidated AMP change."}, {'reference_cell': 29, 'sha256': '5c31acb1019dd9de97103d9a68cb0fc0db6aa1032fb00af5b483b3a67ed1b08b', 'bytes': 640, 'first_line': "_a5_sub = pd.read_csv('/kaggle/working/_pipeline_stage.csv', dtype={'StudyInstanceUID':str})"}, {'reference_cell': 31, 'sha256': 'cebe98aefa3d4bd90770a2711f93b48012e1fb56ea401e9b1ba3e58863097bf6', 'bytes': 52977, 'first_line': 'from __future__ import annotations'}, {'reference_cell': 33, 'sha256': 'feba3fd71eb0f6a45f2c996ce3958228ba0b15c6935e447418cbebf9642668d3', 'bytes': 39587, 'first_line': "if globals().get('_DINOV2_MATCHED_MEMBERS') != 20:"}], 'ordered_cell_hashes_sha256': '84241366427ec267642a513ce6215e47bd22e8ae9fd3fb647f95396738ae1cce', 'embedded_raptor_sha256': 'a04887b868614fa5204784bd82d813d2744cf227bc5951b39526b1b0180c811f'}
execute_pinned_stage, EXECUTED_REFERENCE_STAGES = make_stage_runner(globals(), FIXED_RUN, FIXED_BUDGET)
FIXED_METADATA_ROOT = resolve_metadata_root(FIXED_RUN['competition_root'])
FIXED_OUTPUT = new_output_directory(FIXED_RUN['output_root'])
print('Mode:', FIXED_RUN['mode'], '| Candidate:', FIXED_RUN['submission_candidate'])
print('Output directory:', FIXED_OUTPUT)


In [ ]:
"""Transparent orchestration and auditing helpers for the supplied RSNA ensemble.

This module never supplies hidden labels, fits on test data, or downloads models.
It uses the checkpoint-specific image readers in the accompanying notebook.
"""
from pathlib import Path
import hashlib
import json
import os
import shutil
import time

import numpy as np
import pandas as pd

RSNA_TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture',
]
RSNA_ID = 'StudyInstanceUID'


def delivery_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()


def delivery_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w') as handle:
        json.dump(obj, handle, indent=2, sort_keys=True, allow_nan=False)
        handle.write('\n')
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def delivery_frame(frame, ids, name, probability=True):
    """Require exact cohort/label coverage, then align by ID, never row position."""
    if frame.columns.tolist() != [RSNA_ID] + RSNA_TARGETS:
        raise ValueError(f'{name}: expected the exact official 13-column schema')
    frame = frame.copy()
    if frame[RSNA_ID].isna().any():
        raise ValueError(f'{name}: missing prediction ID')
    frame[RSNA_ID] = frame[RSNA_ID].astype(str)
    if any(pd.isna(i) for i in ids):
        raise ValueError(f'{name}: missing expected ID')
    ids = [str(i) for i in ids]
    if not ids or len(ids) != len(set(ids)) or any(not i.strip() for i in ids):
        raise ValueError(f'{name}: empty/duplicate expected IDs')
    if frame[RSNA_ID].duplicated().any():
        raise ValueError(f'{name}: duplicate prediction IDs')
    if set(frame[RSNA_ID]) != set(ids):
        raise ValueError(f'{name}: missing or unexpected study IDs')
    frame = frame.set_index(RSNA_ID).loc[ids].reset_index()
    values = frame[RSNA_TARGETS].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all():
        raise ValueError(f'{name}: nonfinite predictions')
    if probability and ((values < 0).any() or (values > 1).any()):
        raise ValueError(f'{name}: predictions outside [0, 1]')
    return frame


def delivery_rank(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or not len(values) or not np.isfinite(values).all():
        raise ValueError('Expected finite, nonempty [studies, labels] scores')
    return pd.DataFrame(values).rank(method='average', pct=True).to_numpy()


def delivery_from_array(ids, values):
    frame = pd.DataFrame(values, columns=RSNA_TARGETS)
    frame.insert(0, RSNA_ID, [str(i) for i in ids])
    return delivery_frame(frame, ids, 'array conversion')


def delivery_prepare_run(config, work='/kaggle/working'):
    """Create a separate, explicit training-audit view when requested."""
    mode = config['mode']
    if mode not in {'submission', 'audit_train'}:
        raise ValueError('mode must be submission or audit_train')
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    # Remove only this notebook's own previous publication/receipt. A failed run
    # must not leave a stale submission or a stale success marker behind.
    for name in ('submission.csv', 'run_complete.json', 'audit_predictions.csv'):
        (work / name).unlink(missing_ok=True)
    explicit = config.get('competition_root', '')
    roots = [Path(explicit)] if explicit else [
        Path('/kaggle/input/rsna-knee-abnormality-detection'),
        Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
    ]
    existing = {str(p.resolve()): p for p in roots if (p / 'test.csv').is_file()}
    if len(existing) != 1:
        raise FileNotFoundError(f'Expected one competition input; found {list(existing)}')
    original = next(iter(existing.values())).resolve()
    if config.get('fusion_recipe') and not Path(config['fusion_recipe']).is_file():
        raise FileNotFoundError('The requested fusion_recipe file is not attached')
    if config.get('enable_global96', False):
        g96_roots = [
            Path('/kaggle/input/datasets/mattiaangeli/rsna-knee-coatnet-global96-top3'),
            Path('/kaggle/input/rsna-knee-coatnet-global96-top3'),
        ]
        manifests = [p / 'coatnet_global96_top3_manifest.json' for p in g96_roots]
        valid = [p for p in manifests if p.is_file() and delivery_sha(p) ==
                 '015f09030e76f86274cadae40777b3c3cf5c19f8c835d5ad88e15db41903779e']
        if not valid:
            raise FileNotFoundError('enable_global96=True requires the pinned mattiaangeli/rsna-knee-coatnet-global96-top3 input')
    root = original
    if mode == 'audit_train':
        train = pd.read_csv(original / 'train.csv', dtype={RSNA_ID: str})
        if not set([RSNA_ID] + RSNA_TARGETS).issubset(train.columns):
            raise ValueError('Training labels do not match the official schema')
        labels = train[RSNA_TARGETS].apply(pd.to_numeric, errors='raise')
        invalid = labels.notna() & ~labels.isin([0, 1, -1])
        if invalid.any().any():
            raise ValueError('Training targets must be binary, missing, or -1')
        eligible = labels.isin([0, 1]).any(axis=1)
        selected = train.loc[eligible].copy()
        requested = config.get('audit_ids_csv', '')
        if requested:
            requested_ids = pd.read_csv(requested, dtype={RSNA_ID: str})[RSNA_ID].tolist()
            if len(requested_ids) != len(set(requested_ids)):
                raise ValueError('audit_ids_csv contains duplicates')
            if not set(requested_ids).issubset(set(selected[RSNA_ID])):
                raise ValueError('Requested audit IDs must have observed training labels')
            selected = selected.set_index(RSNA_ID).loc[requested_ids].reset_index()
        else:
            selected = selected.sort_values(RSNA_ID, kind='stable')
        limit = int(config.get('audit_limit', 0))
        if limit < 0:
            raise ValueError('audit_limit must be nonnegative; 0 means all')
        if limit:
            selected = selected.iloc[:limit]
        if selected.empty or selected[RSNA_ID].duplicated().any():
            raise ValueError('Empty or duplicate training-audit cohort')
        root = work / 'rsna_train_audit_input'
        root.mkdir(exist_ok=True)
        ids = selected[RSNA_ID].tolist()
        selected[[RSNA_ID]].to_csv(root / 'test.csv', index=False)
        series = pd.read_csv(original / 'train_series.csv', dtype={RSNA_ID: str, 'SeriesInstanceUID': str})
        series = series.loc[series[RSNA_ID].isin(ids)]
        if set(series[RSNA_ID]) != set(ids):
            raise ValueError('An audit study has no training series')
        series.to_csv(root / 'test_series.csv', index=False)
        delivery_from_array(ids, np.full((len(ids), 12), .5)).to_csv(root / 'sample_submission.csv', index=False)
        for name, source in [('test_series', original / 'train_series'), ('train.csv', original / 'train.csv')]:
            target = root / name
            if target.is_symlink():
                target.unlink()
            elif target.exists():
                raise FileExistsError(f'Refusing to overwrite non-symlink: {target}')
            target.symlink_to(source, target_is_directory=source.is_dir())
        selected[[RSNA_ID] + RSNA_TARGETS].to_csv(work / 'audit_truth.csv', index=False)
        print(f'TRAINING AUDIT: {len(ids)} labeled studies. Training overlap is UNKNOWN; this is not OOF validation.')
    os.environ['RSNA_COMP_ROOT'] = str(root)
    os.environ['RSNA_PARALLEL_COAT_READERS'] = '1' if config['parallel_coat_readers'] else '0'
    os.environ['PIP_NO_INDEX'] = '1'
    os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    os.environ.pop('RSNA_LOCAL_ALLOW_V100', None)
    context = {'mode': mode, 'original_root': str(original), 'inference_root': str(root),
               'started_unix': time.time(), 'target_auc': .990, 'target_auc_verified': False}
    delivery_json(work / 'run_configuration.json', {'config': config, 'context': context})
    return context


def delivery_require_full_ensemble(audit, work, expected_ids, prepared_ids):
    """Reject the inherited catch-and-continue paths before publication."""
    forbidden = {
        'raptor_study_failed', 'raptor_neutral_fill', 'a5_study_failed',
        'dino_member_nonfinite_neutral', 'dino_member_public_nonfinite_neutral',
        'empty_study_rows',
        'rad_nonfinite_zeroed', 'rad_head_nonfinite_neutral', 'rad_invalid_values_repaired',
        'coat_resgated_child_failed', 'coat_d4_child_failed',
        'coat_resgated_output_rejected', 'coat_d4_output_rejected',
        'coat_family_unavailable', 'coat_family_partial', 'coat_resgated_fallback_studies',
    }
    bad = [e for e in audit.get('events', []) if e.get('kind') in forbidden]
    if bad:
        raise RuntimeError(f'Full ensemble failed; submission withheld. First events: {bad[:4]}')
    required = {(arm, str(uid)) for arm in ('maxspan-v5', 'native384dense-v10', 'native384-v8') for uid in expected_ids}
    if set(prepared_ids) != required:
        raise RuntimeError('Raptor preparation coverage differs from the requested cohort')
    work = Path(work)
    blend = json.loads((work / '_coat_raptor_blend_receipt.json').read_text())
    if set(blend.get('members', [])) != {'resgated_top3', 'd4_swa3'}:
        raise RuntimeError('Both complementary CoAtNet readers must contribute')
    if int(blend.get('study_count', -1)) != len(expected_ids):
        raise RuntimeError('CoAtNet receipt cohort count mismatch')
    for path, models in [(work / '_coat_arm_receipt.json', 3), (work / 'd4_input/d4.receipt.json', 1)]:
        receipt = json.loads(path.read_text())
        if receipt.get('delivery_contract_verified') is not True or receipt.get('models') != models:
            raise RuntimeError(f'Missing verified child execution/model contract: {path.name}')
        if 'fallback_studies' not in receipt or int(receipt['fallback_studies']) != 0:
            raise RuntimeError(f'Fallback study found in {path.name}')
        if 'studies' in receipt and int(receipt['studies']) != len(expected_ids):
            raise RuntimeError(f'Child cohort size mismatch in {path.name}')


def delivery_export_candidates(work, ids, outer_weights):
    """Export distinct families and two explicit, UNVALIDATED ablations."""
    work = Path(work)
    output = work / 'candidates'
    output.mkdir(exist_ok=True)
    paths = {
        'reference': work / '_pipeline_stage.csv',
        'raptor_public': work / '_raptor_public.csv',
        'coat_family': work / '_coat_family_rank.csv',
    }
    diagnostic_names = {
        'transformer': 'btkd_parent_after_rad',
        'transformer_no_calibrator': 'transformer_no_calibrator',
        'transformer_rad_fracture': 'transformer_rad_fracture',
        'coat_resgated': 'coat_resgated_rank',
        'coat_d4': 'coat_d4_rank',
    }
    frames = {}
    for name, path in paths.items():
        frames[name] = delivery_frame(pd.read_csv(path, dtype={RSNA_ID: str}), ids, name)
    for name, tag in diagnostic_names.items():
        path = work / 'diagnostics' / f'{tag}.npz'
        with np.load(path, allow_pickle=False) as data:
            labels = data['labels'].astype(str).tolist()
            if set(labels) != set(RSNA_TARGETS) or len(labels) != len(RSNA_TARGETS):
                raise ValueError(f'{tag}: unexpected labels')
            frame = pd.DataFrame(data['values'], columns=labels)[RSNA_TARGETS]
            frame.insert(0, RSNA_ID, data['study_uids'].astype(str))
        frames[name] = delivery_frame(frame, ids, name)
    # Use exactly the existing hybrid CoAt branch before the reference outer blend.
    hybrid = delivery_frame(pd.read_csv(work / '_raptor.csv', dtype={RSNA_ID: str}), ids, 'hybrid')
    coat_values = hybrid[RSNA_TARGETS].to_numpy(dtype=np.float64)
    w = np.array([outer_weights[label] for label in RSNA_TARGETS])
    tr = delivery_rank(frames['transformer_no_calibrator'][RSNA_TARGETS])
    frames['reference_no_calibrator'] = delivery_from_array(ids, delivery_rank((1-w)*tr + w*coat_values))
    tr = delivery_rank(frames['transformer'][RSNA_TARGETS])
    frames['uniform_outer_060'] = delivery_from_array(ids, delivery_rank(.4*tr + .6*coat_values))
    tr = delivery_rank(frames['transformer_rad_fracture'][RSNA_TARGETS])
    frames['reference_rad_fracture'] = delivery_from_array(ids, delivery_rank((1-w)*tr + w*coat_values))
    manifest = {'cohort_size': len(ids), 'prediction_role': 'inference',
                'held_out_from_checkpoint_training': 'unknown',
                'note': 'Correlated stage outputs are not independent model runs. Ablations are not validated improvements.',
                'sources': []}
    for name, frame in frames.items():
        path = output / f'{name}.csv'
        frame.to_csv(path, index=False)
        values = frame[RSNA_TARGETS].to_numpy()
        manifest['sources'].append({'name': name, 'path': str(path), 'sha256': delivery_sha(path),
                                    'unique_scores': {lab: int(len(np.unique(values[:, j]))) for j, lab in enumerate(RSNA_TARGETS)}})
    delivery_json(output / 'manifest.json', manifest)
    return frames


def delivery_audit_scores(truth_csv, frames, work):
    """Diagnostic only: never promote an in-sample/training-audit score to CV."""
    from sklearn.metrics import roc_auc_score
    truth = pd.read_csv(truth_csv, dtype={RSNA_ID: str}).set_index(RSNA_ID)
    rows = []
    for name, frame in frames.items():
        truth_aligned = truth.loc[frame[RSNA_ID]]
        for label in RSNA_TARGETS:
            y = pd.to_numeric(truth_aligned[label], errors='raise').to_numpy(dtype=float)
            p = frame[label].to_numpy(dtype=float)
            known = np.isin(y, [0, 1])
            positive = int(np.sum(y[known] == 1))
            negative = int(np.sum(y[known] == 0))
            value = float(roc_auc_score(y[known], p[known])) if positive and negative else None
            rows.append({'candidate': name, 'label': label, 'auc': value,
                         'positive': positive, 'negative': negative,
                         'status': 'TRAINING_OVERLAP_UNKNOWN_NOT_OOF'})
    result = pd.DataFrame(rows)
    result.to_csv(Path(work) / 'audit_per_class_auc.csv', index=False)
    summary = result.groupby('candidate', sort=False).agg(mean_defined_auc=('auc', 'mean'), defined_labels=('auc', 'count'))
    summary['official_macro_auc_if_all_12_defined'] = summary.mean_defined_auc.where(summary.defined_labels == 12)
    summary['status'] = 'DIAGNOSTIC_ONLY_TRAINING_OVERLAP_UNKNOWN'
    summary.to_csv(Path(work) / 'audit_summary.csv')
    print(summary.to_string())
    return summary


def delivery_publish(frame, ids, context, receipt, work='/kaggle/working'):
    work = Path(work)
    frame = delivery_frame(frame, ids, 'final publication')
    filename = 'submission.csv' if context['mode'] == 'submission' else 'audit_predictions.csv'
    path = work / filename
    tmp = path.with_suffix('.csv.tmp')
    with tmp.open('w') as handle:
        frame.to_csv(handle, index=False)
        handle.flush()
        os.fsync(handle.fileno())
    receipt = dict(receipt, file=filename, cohort_size=len(ids), mode=context['mode'],
                   sha256=delivery_sha(tmp), kaggle_score=None, target_auc_verified=False)
    # Validate and stage the complete receipt before publishing the CSV. The
    # success receipt is committed last, so a failed JSON conversion never
    # leaves a seemingly deliverable new prediction file.
    encoded = json.dumps(receipt, indent=2, sort_keys=True, allow_nan=False) + '\n'
    receipt_path = work / 'run_complete.json'
    receipt_tmp = work / 'run_complete.json.tmp'
    with receipt_tmp.open('w') as handle:
        handle.write(encoded)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)
    os.replace(receipt_tmp, receipt_path)
    print(f'{filename}: {len(ids)} studies, {len(RSNA_TARGETS)} labels; SHA256={receipt["sha256"]}')
    return path


In [ ]:
if FIXED_RUN['mode'] == 'inference':
    RSNA_RUN = {
        'mode': 'submission', 'competition_root': str(FIXED_METADATA_ROOT),
        'audit_ids_csv': '', 'audit_limit': 0, 'parallel_coat_readers': True,
        'enable_global96': False, 'submission_candidate': FIXED_RUN['submission_candidate'],
        'fusion_recipe': FIXED_RUN['fusion_recipe'],
    }
    DELIVERY_CONTEXT = delivery_prepare_run(RSNA_RUN)
else:
    print('CPU replay: all reference model stages below will be skipped.')


## 2. Preserved reference inference stages

Each visible source string below is checked against its original SHA256 and
executed once, in the original order, only in inference mode. `linecache` keeps
source inspection available for the inherited child-worker builders. Model
errors stop publication. The independent source-budget report lists all hashes.

The model workload remains DINO 20 member tails/shared prefix, A5 five folds,
the original Rad layouts/heads, Raptor four views from three checkpoints,
residual CoAt three checkpoints, and one effective D4 SWA model plus adapter.


### Pinned workers and preprocessing


In [ ]:
GPU_STAGE_006 = r'''import os
for _thread_env in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_thread_env, '4')
os.environ.setdefault('CUDNN_CONV_WSCAP_DBG', '1024')

def _d4_check_runtime(rt, artifact_root):
    import hashlib
    import json
    from pathlib import Path
    import torch
    root = Path(artifact_root)
    manifest_path = root / 'coatnet_pairfilm_manifest.json'
    digest = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    if digest != '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d':
        raise RuntimeError('D4 reference manifest identity changed')
    expected = json.loads(manifest_path.read_text())
    manifest, paths = rt.validate_artifact_root(root, verify_hashes=True)
    if manifest != expected or len(paths) != 1 or rt.N_MODELS != 1:
        raise RuntimeError('D4 must load exactly the reference SWA model')
    if Path(paths[0]).name != 'coatnet_d4_parent_rank8_swa2.pt':
        raise RuntimeError('D4 must use its reference rank-8 parent checkpoint')
    if manifest['slice_depth']['adapter_sha256'] != 'b7c48b19997bc7ecb7e997f8dcf491ec5701ff917d1dac4b369eb85ac97ae89a':
        raise RuntimeError('D4 reference adapter identity changed')
    if rt.GPU_BATCH_STUDIES != 2 or rt.BACKBONE_MICRO_IMAGES != 8:
        raise RuntimeError('D4 reference batching changed')
    if rt.CUDNN_BENCHMARK is not False:
        raise RuntimeError('D4 reference convolution policy changed')
    if rt.base.cv2.__version__ != '4.12.0' or rt.parent.timm.__version__ != '1.0.22':
        raise RuntimeError('D4 reference dependency versions changed')
    if torch.cuda.device_count() < 1:
        raise RuntimeError('D4 requires at least one GPU')
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    return (manifest, paths)

def _d4_check_outputs(rt, manifest, receipt, output, competition):
    import hashlib
    from pathlib import Path
    import numpy as np
    import pandas as pd

    def need(condition, message):
        if not condition:
            raise RuntimeError('D4 release check: ' + message)
    need(receipt['status'] == rt.SUBMISSION_STATUS, 'unsuccessful status')
    expected_fields = {'models': 1, 'fallback_studies': 0, 'dicom_preparations_per_study': 1, 'model_passes_per_study': 1, 'rank_after_probability_average': True, 'canonical_sagittal': True, 'common_physical_triplet_warp': False, 'resolution': 384, 'slice_depth_arm': 'd4_zones', 'slice_depth_state_elements': 3255, 'slice_depth_parent_checkpoint_sha256': '5de38e333e3184d7b52fb068183f0cc198d6cbabb6ffbe36b26338622e82d297', 'fsx_attention_heads': 2, 'fsx_attention_width': 128, 'fsx_serving_delta_cap': 0.3, 'fsx_training_delta_cap': 0.5, 'head': 'rank8_swa2_plus_d4_three_zone_expanded_fsx', 'slice_depth_swa_member_epochs': [11, 9, 5], 'slice_depth_checkpoint_sha256': manifest['slice_depth']['adapter_sha256']}
    for key, value in expected_fields.items():
        need(receipt[key] == value, key)
    need(1 <= int(receipt['maximum_eval_windows']) <= 94, 'window limit')
    checkpoints = receipt['checkpoints']
    need(len(checkpoints) == 1, 'parent checkpoint count')
    need(checkpoints[0]['sha256'] == '5de38e333e3184d7b52fb068183f0cc198d6cbabb6ffbe36b26338622e82d297', 'parent checkpoint hash')
    need(checkpoints[0]['member_epochs'] == [15, 16], 'parent SWA member epochs')
    need(receipt['slice_depth_checkpoint_sha256'] == 'b7c48b19997bc7ecb7e997f8dcf491ec5701ff917d1dac4b369eb85ac97ae89a', 'adapter checkpoint hash')
    processes = receipt['processes']
    need(len(processes) >= 1 and all((int(p['returncode']) == 0 for p in processes)), 'worker process failure')
    shards = receipt['shards']
    need(len(shards) >= 1, 'shard count')
    for s in shards:
        need(s['device'] == 'cuda', 'non-CUDA shard')
        need(int(s['fallback_studies']) == 0 and int(s['preparation_warnings']) == 0, 'failed study/preparation')
        need(int(s['models_resident']) == 1, 'resident model count')
        need(int(s['peak_reserved_bytes']) < int(15.5 * 1024 ** 3), 'reference memory gate')
        need(s['specialized_worker_loader_contract'] == manifest['worker_loader_contract'], 'specialized loader')
    output = Path(output)
    prediction_path = output.parent / rt.PREDICTIONS_NAME
    with np.load(prediction_path, allow_pickle=False) as payload:
        uids = payload['study_uids'].astype(str).tolist()
        raw = np.asarray(payload['raw_probabilities'], dtype=np.float32)
        mean = np.asarray(payload['probability_mean'], dtype=np.float32)
        ranked = np.asarray(payload['submission_percentile_rank'], dtype=np.float64)
    ids = pd.read_csv(Path(competition) / 'test.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()
    need(len(ids) > 0 and len(ids) == len(set(ids)), 'test identity')
    need(uids == ids and int(receipt['studies']) == len(ids), 'prediction identity/count')
    need(raw.shape == (1, len(ids), 12), 'raw probability shape')
    need(np.isfinite(raw).all() and ((raw >= 0) & (raw <= 1)).all(), 'raw probabilities')
    need(np.array_equal(raw.astype(np.float64).mean(0).astype(np.float32), mean), 'probability mean')
    need(np.array_equal(rt.percentile_rank64(mean), ranked), 'global percentile ranks')
    frame = pd.read_csv(output, dtype={'StudyInstanceUID': str}, float_precision='round_trip')
    need(frame.columns.tolist() == ['StudyInstanceUID', *rt.base.LABELS], 'CSV labels')
    need(frame.StudyInstanceUID.tolist() == ids, 'CSV identity')
    need(np.array_equal(frame.iloc[:, 1:].to_numpy(np.float64), ranked), 'CSV numeric round trip')
    actual_hash = hashlib.sha256(output.read_bytes()).hexdigest()
    need(actual_hash == receipt['output_sha256'], 'CSV hash')
    receipt['reference_wrapper_checks'] = {'export_sha256': '7f8410fdf1b60cd3831d50f517bb25798a4e4a3404c59dc269193d5fe7cce729', 'validated_manifest_sha256': '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d', 'probability_mean_exact': True, 'ranks_exact': True, 'csv_roundtrip_exact': True, 'source_and_model_hashes_validated': True, 'input_identity_equals_original_grid': False}
    receipt['delivery_contract_verified'] = True
    return receipt

def _run_required_child(command, environment, logfile):
    import os
    import signal
    import subprocess
    import time
    from pathlib import Path
    remaining = TIME_BUDGET - (time.time() - T0)
    if remaining <= 0:
        raise TimeoutError('No time remains for a required model branch')
    path = Path(logfile)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as log_handle:
        process = subprocess.Popen(command, env=environment, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            returncode = process.wait(timeout=remaining)
        except BaseException:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait()
            raise
    if returncode != 0:
        with path.open('rb') as handle:
            handle.seek(max(0, path.stat().st_size - 5000))
            tail = handle.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Required model branch failed ({returncode}); {path}\n{tail}')
    return returncode
ASSET_ROOTS = ['/kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan', '/kaggle/input/raptor-knee-maxspan', '/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384', '/kaggle/input/raptor-knee-native384', '/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384dense', '/kaggle/input/raptor-knee-native384dense', '/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights', '/kaggle/input/knee-mri-fold-weights', '/kaggle/input/datasets/mattiaangeli/opencv-python-headless-4120088-x86', '/kaggle/input/opencv-python-headless-4120088-x86', '/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan', '/kaggle/input/resnet-50-radimagenet-marwan', '/kaggle/input/datasets/mattiaangeli/rsna-knee-coat-resgated-ep10-top3', '/kaggle/input/rsna-knee-coat-resgated-ep10-top3', '/kaggle/input/datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20', '/kaggle/input/rsna-knee-e11-diverse-heads-v20', '/kaggle/input/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15', '/kaggle/input/rsna-knee-e9-radimagenet-heads-v15', '/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels', '/kaggle/input/rsna-knee-llm-labels', '/kaggle/input/datasets/prvsiyan/rsna-knee-v52-radimagenet-heads-20260812', '/kaggle/input/rsna-knee-v52-radimagenet-heads-20260812', '/kaggle/input/datasets/pilkwang/rsna-knee-weights', '/kaggle/input/rsna-knee-weights', '/kaggle/input/datasets/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2', '/kaggle/input/rsna-knee-coatnet-d4-depthzone-swa3-b2', '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e11-train', '/kaggle/input/rsna-knee-e11-train', '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e13-train', '/kaggle/input/rsna-knee-e13-train', '/kaggle/input/models/metaresearch/dinov2/pytorch/small/1', '/kaggle/input/dinov2/pytorch/small/1']

def _asset_walk():
    from pathlib import Path
    seen = set()
    for root in ASSET_ROOTS:
        root = Path(root).resolve()
        if not root.is_dir():
            continue
        level = [root]
        for _depth in range(6):
            following = []
            for directory in sorted(level):
                if str(directory) in seen:
                    continue
                seen.add(str(directory))
                children = sorted(directory.iterdir())
                dirs = [p for p in children if p.is_dir() and p.name not in ('train_series', 'test_series', 'train_images', 'test_images')]
                files = [p.name for p in children if p.is_file()]
                yield (str(directory), [p.name for p in dirs], files)
                following.extend(dirs)
            level = following

def _asset_find_asset(name, digest=None):
    import fnmatch, hashlib
    from pathlib import Path
    hits = []
    for root, _, files in _asset_walk():
        for file in files:
            if fnmatch.fnmatchcase(file, name):
                p = Path(root) / file
                if digest:
                    h = hashlib.sha256()
                    with p.open('rb') as handle:
                        for block in iter(lambda: handle.read(8 << 20), b''):
                            h.update(block)
                    if h.hexdigest() != digest:
                        continue
                if p.resolve() not in [h.resolve() for h in hits]:
                    hits.append(p)
    if len(hits) != 1:
        raise RuntimeError(f'expected one pinned asset {name}, got {hits}')
    return hits[0]

def _asset_half_rank_mix(top3, swa3):
    if top3.shape != swa3.shape or not top3.index.equals(swa3.index):
        raise ValueError('CoAt family alignment changed')
    if top3.isna().any().any() or swa3.isna().any().any():
        raise ValueError('missing CoAt family prediction')
    units = top3.rank(method='average') + swa3.rank(method='average')
    return units.rank(method='average', pct=True)

def _asset_write_coat_runtime(artifact, output, helper_source):
    from pathlib import Path
    import hashlib
    source = Path(artifact) / 'coatnet_resgated_ep10_top3_inference.py'
    text = source.read_text()
    if hashlib.sha256(source.read_bytes()).hexdigest() != 'b11e58f8d7cabe9e264ac70b01e811dc6a846aa01248ace1f0e6536d0d4094d9':
        raise RuntimeError('unexpected parent CoAt runtime')
    edits = {'if not 1 <= windows <= MAX_EVAL_WINDOWS:': 'if not 1 <= windows <= 94:', 'or int(counts.max()) > MAX_EVAL_WINDOWS:': 'or int(counts.max()) > 94:', '"runtime_contract": RUNTIME_CONTRACT,': '"runtime_contract": RUNTIME_CONTRACT, "input_max_windows": 94,', '"eval_grid": "saved_training_gold_unique_center_slot_budgets",': '"eval_grid": "input_unique_native_centers_six_slot_v1",'}
    for old, new in edits.items():
        if old not in text:
            raise RuntimeError('CoAt patch target missing: ' + old)
        text = text.replace(old, new)
    text = text.replace('"gold58_rank_ensemble_auc": manifest["selection"]', '"historical_parent_gold58_rank_ensemble_auc": manifest["selection"]')
    text = text.replace('"t4_gold58_rank_ensemble_auc": manifest["t4_numerical_portability"]', '"historical_parent_t4_gold58_rank_ensemble_auc": manifest["t4_numerical_portability"]')
    marker = 'if __name__ == "__main__":'
    if text.count(marker) != 1:
        raise RuntimeError('CoAt entrypoint changed')
    patch = helper_source + ('\n_original_train_faithful_eval_specs = train_faithful_eval_specs\n'
                             'def _tolerant_coat_specs(study):\n    try:\n        return _dense_coat_specs(study)\n'
                             '    except Exception as exc:\n        print("[dense-fallback] " + str(getattr(study, "study_uid", "?")) + ": " + type(exc).__name__ + ": " + str(exc), flush=True)\n'
                             '        return _original_train_faithful_eval_specs(study)\n'
                             'train_faithful_eval_specs = _tolerant_coat_specs\nRUNTIME_CONTRACT = "resgated_ep10_input_nativecenters_v1"\n')
    text = text.replace(marker, patch + '\n' + marker)
    compile(text, str(output), 'exec')
    Path(output).write_text(text)
    return hashlib.sha256(text.encode()).hexdigest()

def _asset_patch_d4(rt, grid_source):
    import ast
    import hashlib
    import inspect
    import json
    import os
    import textwrap
    from pathlib import Path
    audited = rt.parent.audited
    original = audited.prepare_global96_bag
    original_shard = rt._PARENT_INFER_SHARD
    if getattr(original, '_input_patch_installed', False):
        raise RuntimeError('D4 input preparation was already patched')
    src = textwrap.dedent(inspect.getsource(original))
    tree = ast.parse(src)
    fn = next((n for n in tree.body if isinstance(n, ast.FunctionDef)), None)
    if fn is None or fn.name != 'prepare_global96_bag':
        raise RuntimeError('D4 preparation function identity changed')
    if 'study_uid' not in inspect.signature(original).parameters:
        raise RuntimeError('D4 preparation study-UID contract changed')
    candidates = [n for n in ast.walk(fn) if isinstance(n, ast.Assign) and any((isinstance(t, ast.Name) and t.id == 'specs' for t in n.targets)) and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Attribute) and isinstance(n.value.func.value, ast.Name) and (n.value.func.value.id == 'global_stack') and (n.value.func.attr == 'sample_global_windows')]
    if len(candidates) != 1 or candidates[0] not in fn.body:
        raise RuntimeError('D4 expected one top-level Global96 specs constructor')
    call = candidates[0]
    keywords = {k.arg: k.value for k in call.value.keywords}
    if not (isinstance(keywords.get('count'), ast.Constant) and keywords['count'].value is None and isinstance(keywords.get('train'), ast.Constant) and (keywords['train'].value is False)):
        raise RuntimeError('D4 original evaluation must use count=None, train=False')
    exec(compile(grid_source, '<input-grid>', 'exec'), audited.__dict__)
    audit_dir = Path(os.environ['RSNA_D4_INPUT_AUDIT_DIR'])
    audit_dir.mkdir(parents=True, exist_ok=True)

    def record_input(study_uid, stack, audit):
        import numpy as np
        arrays = ('global_indices', 'nominal_slots', 'canonical_depths', 'nominal_steps', 'source_series_rows', 'source_slot_ids')
        digest = hashlib.sha256()
        for name in arrays:
            value = np.ascontiguousarray(getattr(stack, name))
            digest.update(name.encode())
            digest.update(str(value.dtype).encode())
            digest.update(str(value.shape).encode())
            digest.update(value.tobytes())
        record = {'study_uid': str(study_uid), 'pid': os.getpid(), 'positions': int(len(stack.global_indices)), 'windows_expected': int(len(stack.global_indices) - 2), 'slot_widths': list(map(int, stack.slot_widths)), 'stack_sidecars_sha256': digest.hexdigest(), 'slots': audit}
        with (audit_dir / f'input_{os.getpid()}.jsonl').open('a') as handle:
            handle.write(json.dumps(record, sort_keys=True, allow_nan=False) + '\n')
    audited.__dict__['_record_input'] = record_input
    injection = ast.parse('try:\n    stack, input_audit = _dense_stack(stack, study.offsets, study.valid, study.source_depth)\n'
                          'except Exception as _dense_exc:\n    print("[dense-fallback] " + str(study_uid) + ": " + type(_dense_exc).__name__ + ": " + str(_dense_exc), flush=True)\n'
                          '    input_audit = [{"fallback": type(_dense_exc).__name__ + ": " + str(_dense_exc)}]\n'
                          '_record_input(study_uid, stack, input_audit)\n').body
    where = fn.body.index(call)
    fn.body[where:where] = injection
    ast.fix_missing_locations(tree)
    exec(compile(tree, '<D4-input-preparation-only>', 'exec'), audited.__dict__)
    patched = audited.prepare_global96_bag
    patched._input_patch_installed = True
    for module in (rt, rt.parent):
        if getattr(module, 'prepare_global96_bag', None) is original:
            module.prepare_global96_bag = patched
    if rt._PARENT_INFER_SHARD is not original_shard:
        raise RuntimeError('D4 parent shard must remain the official callable')
    files = {}
    for name, module in [('d4_runtime', rt), ('parent_runtime', rt.parent), ('audited_runtime', audited)]:
        path = Path(module.__file__)
        if path.is_file():
            files[name] = {'file': path.name, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    receipt = {'contract': 'd4-original-runtime-input-preparation-only-v2', 'reference_script_version_id': 348764100, 'reference_export_available': True, 'original_prepare_source_sha256': hashlib.sha256(src.encode()).hexdigest(), 'patched_prepare_source_sha256': hashlib.sha256(ast.unparse(tree).encode()).hexdigest(), 'model_loader_modified': False, 'zone_head_modified': False, 'parent_infer_shard_replaced': False, 'grid_source_sha256': hashlib.sha256(grid_source.encode()).hexdigest(), 'artifact_source_files': files}
    (audit_dir / f'patch_{os.getpid()}.json').write_text(json.dumps(receipt, indent=2))
    return receipt

def _asset_run_d4(artifact, environment_dir, competition, output):
    import hashlib, inspect, json, os, sys
    from pathlib import Path
    import pandas as pd
    artifact, output = (Path(artifact), Path(output))
    manifest = artifact / 'coatnet_pairfilm_manifest.json'
    if hashlib.sha256(manifest.read_bytes()).hexdigest() != '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d':
        raise RuntimeError('D4 SWA3 manifest changed')
    grid = 'import numpy as np\nfrom raptor_global_stack_smart336 import FixedGlobalStack, INFERRED96_WIDTHS\n' + '\n'.join((inspect.getsource(f) for f in (_dense_quotas, _dense_stack)))
    audit_dir = output.parent / 'study_input_receipts'
    audit_dir.mkdir(parents=True, exist_ok=True)
    for stale in audit_dir.glob('*.json*'):
        stale.unlink()
    output.unlink(missing_ok=True)
    output.with_suffix('.receipt.json').unlink(missing_ok=True)
    wrapper = output.parent / 'd4_worker_entry.py'
    code = f'import sys\nsys.path.insert(0,{str(environment_dir)!r})\nsys.path.insert(0,{str(artifact)!r})\nfrom pathlib import Path\nimport json\nimport coatnet_d4_depthzone_swa_inference as rt\nimport os\n' + inspect.getsource(_asset_patch_d4) + '\n' + inspect.getsource(_d4_check_runtime) + '\n' + inspect.getsource(_d4_check_outputs) + '\n' + f"if __name__ == '__main__' and '--worker-output' not in sys.argv:\n    _checked_manifest, _checked_paths = _d4_check_runtime(rt, Path({str(artifact)!r}))\n" + f'_asset_patch_d4(rt,{grid!r})\n' + 'rt.parent.audited.__file__ = __file__\n' + "if __name__ == '__main__':\n" + "    if '--worker-output' in sys.argv:\n        raise SystemExit(rt.main())\n" + '    manifest, paths = _checked_manifest, _checked_paths\n' + '    r = rt.run_submission(\n' + f'        competition_root=Path({str(competition)!r}), artifact_root=Path({str(artifact)!r}),\n' + f'        output_path=Path({str(output)!r}), gpu_batch_studies=2, backbone_micro_images=8)\n' + f'    r = _d4_check_outputs(rt, manifest, r, Path({str(output)!r}), Path({str(competition)!r}))\n' + f"    Path({str(output.with_suffix('.receipt.json'))!r}).write_text(json.dumps(r,indent=2,allow_nan=False))\n"
    compile(code, str(wrapper), 'exec')
    wrapper.write_text(code)
    env = dict(os.environ)
    env['RSNA_D4_INPUT_AUDIT_DIR'] = str(audit_dir)
    env['PYTHONPATH'] = f'{environment_dir}:/kaggle/working/_coat_env:{artifact}:' + env.get('PYTHONPATH', '')
    env.pop('CUDNN_CONV_WSCAP_DBG', None)
    _run_required_child([sys.executable, str(wrapper)], env, output.with_suffix('.log'))
    receipt = json.loads(output.with_suffix('.receipt.json').read_text())
    expected = pd.read_csv(Path(competition) / 'test.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()
    records = []
    for path in sorted(audit_dir.glob('input_*.jsonl')):
        records.extend((json.loads(line) for line in path.read_text().splitlines() if line.strip()))
    ids = [r['study_uid'] for r in records]
    if len(ids) != len(expected) or len(ids) != len(set(ids)) or set(ids) != set(expected):
        raise RuntimeError('D4 input hook must execute exactly once for every test study')
    if not all((1 <= r['windows_expected'] <= 94 and r['positions'] == r['windows_expected'] + 2 for r in records)):
        raise RuntimeError('D4 input record cardinality drift')
    receipt['input_coverage'] = {'studies': len(records), 'folder': str(audit_dir), 'min_windows': min((r['windows_expected'] for r in records)), 'max_windows': max((r['windows_expected'] for r in records)), 'original_model_loader_retained': True, 'original_shard_scheduler_retained': True}
    output.with_suffix('.receipt.json').write_text(json.dumps(receipt, indent=2, allow_nan=False))
    return receipt
_speed_original_asset_walk = _asset_walk
_speed_original_find_asset = _asset_find_asset
'Catalogue named immutable attachments once; no competition-tree search.'
import threading as _speed_threading
_speed_asset_lock = _speed_threading.RLock()
_speed_asset_catalogue = None
_speed_asset_hits = {}

def _asset_walk():
    global _speed_asset_catalogue
    with _speed_asset_lock:
        if _speed_asset_catalogue is None:
            _speed_asset_catalogue = tuple(((root, tuple(dirs), tuple(files)) for root, dirs, files in _speed_original_asset_walk()))
        catalogue = _speed_asset_catalogue
    for root, dirs, files in catalogue:
        yield (root, list(dirs), list(files))

def _asset_find_asset(name, digest=None):
    key = (str(name), digest)
    with _speed_asset_lock:
        if key not in _speed_asset_hits:
            _speed_asset_hits[key] = _speed_original_find_asset(name, digest)
        return _speed_asset_hits[key]
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_006, 6, '789f88e22c78c5953428ef0885ff70b7107d06169ca861bf09408c3e13e5c443')


In [ ]:
GPU_STAGE_008 = r'''import numpy as np

def _dense_allocate(capacities, proportions, target):
    capacities = np.asarray(capacities, dtype=np.int64)
    proportions = np.asarray(proportions, dtype=np.float64)
    if capacities.ndim != 1 or capacities.shape != proportions.shape or np.any(capacities < 0) or (not np.isfinite(proportions).all()) or np.any(proportions <= 0) or (int(target) < 1):
        raise ValueError('invalid capacity-aware sampling allocation')
    ideal = int(target) * proportions / proportions.sum()
    initial = np.floor(ideal).astype(np.int64)
    for i in np.argsort(-(ideal - initial), kind='stable')[:int(target) - int(initial.sum())]:
        initial[i] += 1
    quotas = np.minimum(initial, capacities)
    wanted = min(int(target), int(capacities.sum()))
    deficit = np.zeros(len(quotas), dtype=np.float64)
    while int(quotas.sum()) < wanted:
        donors = quotas < capacities
        weights = np.where(donors, proportions, 0.0)
        deficit[donors] += weights[donors] / weights.sum()
        chosen = int(np.argmax(np.where(donors, deficit, -np.inf)))
        quotas[chosen] += 1
        deficit[chosen] -= 1
    assert int(quotas.sum()) == wanted and np.all(quotas <= capacities)
    return quotas

def _dense_unique_linspace(capacity, count):
    if not 0 <= count <= capacity:
        raise ValueError('cannot create unique picks beyond physical capacity')
    result = np.linspace(0, capacity - 1, count).round().astype(np.int64)
    if count and (result.min() < 0 or result.max() >= capacity):
        raise AssertionError('out-of-range source')
    assert len(np.unique(result)) == count
    return result

def _dense_coat_specs(study):
    import raptor_light224 as sampler
    capacities = []
    for row in study.series_rows:
        if int(row) < 0:
            capacities.append(0)
        else:
            centers, _, _, _ = sampler._candidate_centers(int(row), study.offsets, study.valid, study.source_depth)
            capacities.append(len(centers))
    if sum(capacities) < 1:
        raise ValueError('no usable CoAt triplets')
    quotas = _dense_allocate(capacities, (20, 12, 12, 8, 16, 8), 94)
    specs = sampler.sample_eval_windows(study.series_rows, study.offsets, study.valid, study.source_depth, count=None, slot_budgets=np.maximum(quotas, 1).tolist())
    if len(specs) != min(94, sum(capacities)):
        raise RuntimeError('capacity-aware sampling window-count drift')
    keys = [(s.series_row, s.center_local) for s in specs]
    if len(keys) != len(set(keys)):
        raise RuntimeError('capacity-aware sampling repeated a native center')
    for s in specs:
        start = int(study.offsets[s.series_row])
        if not np.all(study.valid[start + np.asarray(s.local_indices)]):
            raise RuntimeError('capacity-aware sampling invalid source channel')
    return specs
INFERRED96_WIDTHS = (27, 21, 18, 12, 18)

def _dense_quotas(capacities) -> np.ndarray:
    capacities = np.asarray(capacities, dtype=np.int64)
    base = np.asarray(INFERRED96_WIDTHS, dtype=np.int64)
    if capacities.shape != (5,) or bool((capacities < 0).any()):
        raise ValueError('expected five nonnegative source capacities')
    quotas = np.minimum(base, capacities)
    target = min(96, int(capacities.sum()))
    deficit = np.zeros(5, dtype=np.float64)
    while int(quotas.sum()) < target:
        donors = quotas < capacities
        weights = np.where(donors, base, 0).astype(np.float64)
        deficit[donors] += weights[donors] / weights.sum()
        selected = int(np.argmax(np.where(donors, deficit, -np.inf)))
        quotas[selected] += 1
        deficit[selected] -= 1
    if int(quotas.sum()) != target or bool((quotas > capacities).any()):
        raise RuntimeError('additional quota allocation changed')
    return quotas

def _dense_stack(reference, offsets, valid, source_depth):
    from types import SimpleNamespace
    from raptor_global_stack_smart336 import FixedGlobalStack
    if tuple(reference.slot_widths) != tuple(INFERRED96_WIDTHS):
        raise ValueError('capacity-aware sampling is pinned to the Global96 reference layout')
    pools = []
    audit = []
    begin = 0
    for slot, width in enumerate(reference.slot_widths, 1):
        end = begin + width
        old = reference.global_indices[begin:end]
        row = int(reference.source_series_rows[begin])
        info = {'slot': slot, 'original_width': width, 'source_row': row}
        audit.append(info)
        if row < 0 or bool((old < 0).all()):
            pools.append((np.empty(0, dtype=np.int64), np.empty(0, dtype=np.float32)))
            begin = end
            continue
        start, stop = map(int, offsets[row:row + 2])
        if not 0 <= start < stop <= len(valid) == len(source_depth):
            raise ValueError('source offset drift')
        depth = np.asarray(source_depth[start:stop], dtype=np.float32)
        keep = np.asarray(valid[start:stop], dtype=bool) & np.isfinite(depth) & (depth >= 0.02) & (depth <= 0.98)
        eligible = np.flatnonzero(keep).astype(np.int64) + start
        reverse = bool(old[0] > old[-1] or (old[0] == old[-1] and np.float32(source_depth[old[0]]) != reference.canonical_depths[begin]))
        if reverse:
            eligible = eligible[::-1].copy()
        depth = np.asarray(source_depth[eligible], dtype=np.float32)
        if reverse:
            depth = (1.0 - depth).astype(np.float32)
        pools.append((eligible, depth))
        begin = end
    quotas = _dense_quotas([len(pool[0]) for pool in pools])
    if int(quotas.sum()) < 3:
        raise ValueError('study has fewer than three usable unique source slices')
    parts = {key: [] for key in ('global_indices', 'nominal_slots', 'canonical_depths', 'nominal_steps', 'source_series_rows', 'source_slot_ids')}
    for slot, ((eligible, depth), quota, info) in enumerate(zip(pools, quotas, audit), 1):
        quota = int(quota)
        info.update(eligible=len(eligible), allocated_width=quota, additional=max(0, quota - info['original_width']))
        info['reason'] = 'missing_no_padding' if not quota else 'short_slot_all_unique' if len(eligible) < info['original_width'] else 'uniform_native_centers' if info['additional'] else 'uniform_native_centers'
        positions = np.linspace(0, len(eligible) - 1, quota).round().astype(np.int64)
        if len(positions) != quota or not bool((np.diff(positions) > 0).all()):
            raise RuntimeError('capacity-aware sampling slot selection repeated a source')
        mask = reference.nominal_slots == slot
        source_slot = int(reference.source_slot_ids[mask][0])
        parts['global_indices'].append(eligible[positions])
        parts['canonical_depths'].append(depth[positions])
        parts['nominal_slots'].append(np.full(quota, slot, dtype=np.int8))
        parts['nominal_steps'].append(np.full(quota, 0.96 / max(quota - 1, 1), dtype=np.float32))
        parts['source_series_rows'].append(np.full(quota, info['source_row'], dtype=np.int32))
        parts['source_slot_ids'].append(np.full(quota, source_slot, dtype=np.int8))
    result = FixedGlobalStack(**{key: np.concatenate(value) for key, value in parts.items()}, slot_widths=tuple((int(value) for value in quotas)))
    if len(np.unique(result.global_indices)) != len(result.global_indices) or bool((result.global_indices < 0).any()):
        raise RuntimeError('capacity-aware sampling output contains a repeated/missing source')
    return (result, audit)
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_008, 8, '5815eed58685ad17392c9c5b13dfb0fb43fb54ec32b7834f4c2c66439cd3ea5a')


### Runtime and input checks


In [ ]:
GPU_STAGE_010 = r'''from __future__ import annotations
import time as _startup_time
_STARTUP_T0 = _startup_time.perf_counter()
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '4')
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

def _cuda_execution_probe(index):
    dev = torch.device(f'cuda:{index}')
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f'unexpected CUDA probe shape {tuple(out.shape)}')
        torch.cuda.synchronize(index)
        print(f'cuda:{index} probe PASS (compute {major}.{minor})')
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        import sys; sys.stderr.write('CUDA PROBE FAIL cuda:' + str(index) + ': ' + repr(exc) + '\n'); sys.stderr.flush()
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False
DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]
import sys; sys.stderr.write('DEBUG: cuda_avail=' + str(torch.cuda.is_available()) + ', count=' + str(torch.cuda.device_count()) + '\n'); sys.stderr.flush()
if len(DEVS) != 2 or torch.cuda.device_count() != 2:
    raise RuntimeError('This reference pipeline requires exactly two working GPUs. Select Kaggle T4 x2.')
if not all('T4' in torch.cuda.get_device_name(i) for i in range(2)):
    raise RuntimeError('This notebook preserves the T4 x2 runtime contract. Select T4 x2; do not impersonate another GPU.')
print(f'devices: {[str(d) for d in DEVS]}')
_STARTUP_GPU_IMPORT_S = _startup_time.perf_counter() - _STARTUP_T0
print(f'[startup] imports+GPU preflight: {_STARTUP_GPU_IMPORT_S:.2f}s', flush=True)
T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
RUNS = [{'name': 'r224', 'img': 224}, {'name': 'r336', 'img': 336}]
EPOCHS = 10
BATCH_STUDIES = 8
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b|smart fat|\\bwater\\b')  # 2026-09-17: GE Dixon water images ('SMART FAT', 'Water: SMART FAT', seq FSEfw) are fat-suppressed; 37/4407 training studies had no fat-sat series detected before
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

# Runtime integrity: no partial ensemble is ever published as submission.csv.
import os, json, hashlib, time, threading, tempfile
from pathlib import Path
import numpy as np
import pandas as pd

_RSNA_AUDIT_LOCK = threading.RLock()
_RSNA_AUDIT = {'contract': 'btkd_speedy_v558_input_uniform', 'events': [], 'phases': {}}
_RSNA_ORDER_MEMO = {}
_RSNA_HEADERS_MEMO = {}
_RSNA_CACHE_FILES = []
_RSNA_TEST_IDS = None
_RSNA_LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

def rsna_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

def rsna_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name('.' + path.name + '.tmp')
    with tmp.open('w') as f:
        json.dump(value, f, indent=2, sort_keys=True, allow_nan=False, default=str)
        f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def rsna_event(kind, **details):
    with _RSNA_AUDIT_LOCK:
        _RSNA_AUDIT['events'].append({'kind':kind, **details})

def rsna_finite(values, tag, probability=False):
    x = np.asarray(values)
    if not np.issubdtype(x.dtype, np.number) or not np.isfinite(x).all():
        raise RuntimeError(f'{tag}: nonfinite/non-numeric values BEFORE ranking')
    if probability and x.size and (x.min() < 0 or x.max() > 1):
        raise RuntimeError(f'{tag}: probability outside [0,1]')
    return x

def rsna_rank01(values):
    """Parent endpoint scale; tied values share their average rank."""
    x = np.asarray(rsna_finite(values, 'rank01'), dtype=np.float64)
    if x.ndim != 2 or not len(x):
        raise ValueError('rank01 requires a nonempty [study,finding] matrix')
    if len(x) == 1:
        return np.full_like(x, .5)
    return (pd.DataFrame(x).rank(method='average').to_numpy() - 1) / (len(x)-1)

def rsna_rankpct(values):
    x = np.asarray(rsna_finite(values, 'rankpct'), dtype=np.float64)
    if x.ndim != 2 or not len(x):
        raise ValueError('rankpct requires a nonempty matrix')
    return pd.DataFrame(x).rank(method='average', pct=True).to_numpy(np.float64)

def rsna_frame(frame, ids, labels, tag):
    ids = [str(u) for u in ids]; labels = list(labels)
    if len(ids) != len(set(ids)):
        raise RuntimeError(f'{tag}: duplicate expected UID')
    if frame.columns.tolist() != ['StudyInstanceUID', *labels]:
        raise RuntimeError(f'{tag}: label order/schema mismatch')
    frame = frame.copy(); frame['StudyInstanceUID'] = frame['StudyInstanceUID'].astype(str)
    if frame.StudyInstanceUID.duplicated().any() or set(frame.StudyInstanceUID) != set(ids):
        raise RuntimeError(f'{tag}: missing/extra/duplicate UID')
    frame = frame.set_index('StudyInstanceUID').loc[ids].reset_index()
    rsna_finite(frame[labels].to_numpy(), tag, probability=True)
    return frame

def rsna_save_predictions(tag, ids, values, labels=None):
    x = rsna_finite(values, tag)
    target = Path('/kaggle/working/diagnostics'); target.mkdir(parents=True, exist_ok=True)
    path = target/(tag+'.npz')
    np.savez_compressed(path, study_uids=np.asarray(ids,dtype=str), labels=np.asarray(labels or _RSNA_LABELS,dtype=str), values=x)
    with _RSNA_AUDIT_LOCK:
        _RSNA_AUDIT['phases'][tag]={'path':str(path),'sha256':rsna_sha(path),'shape':list(x.shape)}

def rsna_strict_load(model, state, tag):
    """A random frozen encoder is never an acceptable missing-state fallback."""
    expected = model.state_dict()
    missing = sorted(set(expected)-set(state)); extra = sorted(set(state)-set(expected))
    bad_shape = [k for k in set(expected)&set(state) if tuple(expected[k].shape)!=tuple(state[k].shape)]
    if missing or extra or bad_shape:
        raise RuntimeError(f'{tag}: incomplete/mismatched checkpoint; missing={missing[:20]}, extra={extra[:20]}, shapes={bad_shape[:20]}. A partial encoder needs its exact pinned source weights; random initialization is forbidden.')
    for k,t in state.items():
        if t.is_floating_point() and not bool(t.isfinite().all()):
            raise RuntimeError(f'{tag}: nonfinite checkpoint tensor {k}')
    return model.load_state_dict(state, strict=True)

def rsna_deadline(tag):
    if time.time() - T0 > TIME_BUDGET:
        raise TimeoutError(f'{tag}: full-run budget exceeded; no incomplete output may be submitted')

def rsna_array(shape, tag):
    """Large temporary pixels belong in scratch space, not saved notebook outputs."""
    size = int(np.prod(shape))
    if size <= float(os.environ.get('RSNA_CACHE_RAM_GIB', '1')) * (1 << 30):
        return np.zeros(shape, np.uint8)
    directory = Path(os.environ.get('RSNA_PIXEL_SCRATCH', '/kaggle/temp/rsna_pixels_v559'))
    try:
        directory.mkdir(parents=True, exist_ok=True)
    except OSError as scratch_error:  # /kaggle/temp is not guaranteed; /tmp is non-persisted scratch with ~1 TiB free
        rsna_event('scratch_fallback', tag=tag, requested=str(directory), error=f'{type(scratch_error).__name__}: {scratch_error}')
        directory = Path('/tmp/rsna_pixels_v559')
        directory.mkdir(parents=True, exist_ok=True)
    import shutil
    free = shutil.disk_usage(directory).free
    rsna_event('scratch_allocation', tag=tag, bytes=size, directory=str(directory), free_bytes=int(free))
    if free < size + (1 << 30) and str(directory) != '/tmp/rsna_pixels_v559':
        rsna_event('scratch_fallback', tag=tag, requested=str(directory), error=f'insufficient free space ({free} bytes)')
        directory = Path('/tmp/rsna_pixels_v559'); directory.mkdir(parents=True, exist_ok=True); free = shutil.disk_usage(directory).free
    if free < size + (1 << 30):
        raise RuntimeError(f'{tag}: not enough scratch disk for complete cache ({size} bytes; {free} free)')
    fd, path = tempfile.mkstemp(prefix='pixels_', suffix='.npy', dir=directory)
    os.close(fd)
    try:
        a = np.lib.format.open_memmap(path, mode='w+', dtype=np.uint8, shape=tuple(shape))
        a[:] = 0
    except BaseException:
        Path(path).unlink(missing_ok=True)
        raise
    _RSNA_CACHE_FILES.append(path)
    return a

def rsna_release_pixels(array):
    if isinstance(array,np.memmap):
        path=str(array.filename)
        array.flush()
        # POSIX unlink releases the file after the last view is closed.
        Path(path).unlink(missing_ok=True)
        if path in _RSNA_CACHE_FILES: _RSNA_CACHE_FILES.remove(path)

def rsna_initialize(ids):
    global _RSNA_TEST_IDS
    _RSNA_TEST_IDS=[str(u) for u in ids]
    if not _RSNA_TEST_IDS or len(set(_RSNA_TEST_IDS))!=len(_RSNA_TEST_IDS):
        raise RuntimeError('empty or duplicate competition study IDs')
    work=Path('/kaggle/working'); work.mkdir(parents=True,exist_ok=True)
    os.chdir(work)
    for f in ['submission.csv','_pipeline_stage.csv','btkd_v559_complete.json','btkd_v559_complete.json']:
        (work/f).unlink(missing_ok=True)
    (work/'diagnostics').mkdir(exist_ok=True)
    _RSNA_AUDIT['cohort']=len(_RSNA_TEST_IDS)
    _RSNA_AUDIT['environment']={'python':__import__('sys').version,'torch':str(torch.__version__),'cuda':torch.version.cuda,'gpu_names':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    rsna_json(work/'diagnostics/runtime_started.json',_RSNA_AUDIT)

def rsna_asset_preflight():
    """Locate required additions before spending time on the public parent."""
    names=['raptor_ft_coatnet_v5_full_swa.pt','raptor_ft_coatnet_v10_full.pt','raptor_ft_coatnet_v8_full_swa.pt']
    receipt={name:str(_asset_find_asset(name)) for name in names}
    for name,digest in [
        ('coat_resgated_ep10_top3_manifest.json','98511a8fdeb9da0e6e70c78d013ff636e1476f31c80b5dc134d294b18c3f284e'),
        ('coatnet_pairfilm_manifest.json','7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d'),
        ('opencv_python_headless-4.12.0.88-*.whl','236c8df54a90f4d02076e6f9c1cc763d794542e886c576a6fee46ec8ff75a7a9')]:
        receipt[name]={'path':str(_asset_find_asset(name,digest)),'sha256':digest}
    roots=[Path(p) for p in ['/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights','/kaggle/input/knee-mri-fold-weights'] if (Path(p)/'m_f0.pt').is_file()]
    roots=list({str(p.resolve()):p for p in roots}.values())
    if len(roots)!=1:raise RuntimeError(f'A5: expected one complete attached root, found {roots}')
    missing=[str(roots[0]/f'm_f{i}.pt') for i in range(5) if not (roots[0]/f'm_f{i}.pt').is_file()]
    if missing:raise FileNotFoundError(f'A5 missing folds: {missing}')
    receipt['A5_root']=str(roots[0]);receipt['tensor_loading_and_fingerprints']='validated by mandatory component loaders before predictions'
    d4_manifest_path = _asset_find_asset('coatnet_pairfilm_manifest.json',
        '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d')
    timm_wheel = d4_manifest_path.parent / 'timm-1.0.22-py3-none-any.whl'
    if not timm_wheel.is_file() or rsna_sha(timm_wheel) != '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0':
        raise RuntimeError('D4 pinned timm wheel missing or changed')
    receipt['D4_timm_wheel_sha256'] = rsna_sha(timm_wheel)
    rsna_json('/kaggle/working/diagnostics/asset_preflight.json',receipt)


def rsna_phase(stage, status='START', **extra):
    """Local phase evidence; this does not expose Kaggle's hidden rerun logs."""
    import json, os, resource, time
    from pathlib import Path
    work = Path('/kaggle/working/diagnostics')
    work.mkdir(parents=True, exist_ok=True)
    current = {'stage': stage, 'status': status,
               'elapsed_seconds': time.time()-T0,
               'max_rss_kib': int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss), **extra}
    for path in ['/sys/fs/cgroup/memory.current', '/sys/fs/cgroup/memory.max']:
        try: current[path.rsplit('/', 1)[-1]] = Path(path).read_text().strip()
        except OSError: pass
    if torch.cuda.is_initialized():
        current['gpus'] = [{'id': i, 'allocated_bytes': torch.cuda.memory_allocated(i),
                            'reserved_bytes': torch.cuda.memory_reserved(i)}
                           for i in range(torch.cuda.device_count())]
    print('[phase] '+json.dumps(current, sort_keys=True), flush=True)
    rsna_json(work/'current_phase.json', current)
    with (work/'phase_events.jsonl').open('a') as f:
        f.write(json.dumps(current, sort_keys=True)+'\n')
    return current

# Record uncaught cell errors locally; Kaggle may still withhold hidden-run logs.
def _rsna_capture_cell_error(result):
    error = getattr(result, 'error_in_exec', None) or getattr(result, 'error_before_exec', None)
    if error is None:
        return
    try:
        path = Path('/kaggle/working/diagnostics/current_phase.json')
        previous = json.loads(path.read_text()) if path.is_file() else {}
        rsna_phase(previous.get('stage', 'unknown'), 'FAILED', error_type=type(error).__name__, error=str(error))
    except Exception as logging_error:
        print('[diagnostic-write-failed]', type(logging_error).__name__, flush=True)
try:
    _rsna_ip = get_ipython()
    if _rsna_ip is not None:
        _rsna_ip.events.register('post_run_cell', _rsna_capture_cell_error)
except NameError:
    pass
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_010, 10, 'ed919119ee53bac62f8fcabbe0f5b71ab5e42dddb638a20bd7de7f19070d99bb')


In [ ]:
GPU_STAGE_012 = r'''def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def _one_direct(tag, candidates, required):
    matches = []
    for candidate in map(Path, candidates):
        if candidate.is_dir() and all((candidate / item).exists() for item in required):
            matches.append(candidate)
    unique = []
    for match in matches:
        if str(match.resolve()) not in {str(item.resolve()) for item in unique}:
            unique.append(match)
    if len(unique) != 1:
        raise FileNotFoundError(
            f'{tag}: expected exactly one direct mounted artifact, found '
            f'{[str(item) for item in unique]}; checked {[str(Path(item)) for item in candidates]}'
        )
    return unique[0]

def find_root():
    explicit = os.environ.get('RSNA_COMP_ROOT', '')
    if explicit:
        return _one_direct('explicit competition', [explicit], ['test.csv', 'test_series.csv', 'test_series'])
    return _one_direct('competition', [
        os.environ.get('RSNA_COMP_ROOT', '/kaggle/input/competitions/rsna-knee-abnormality-detection'),
        '/kaggle/input/rsna-knee-abnormality-detection',
    ], ['test.csv', 'test_series.csv', 'test_series'])

def find_dinov2(variant='small'):
    if variant != 'small':
        raise ValueError(f'unpinned DINOv2 variant: {variant}')
    return _one_direct('DINOv2-S model', [
        '/kaggle/input/models/metaresearch/dinov2/PyTorch/small/1',
        '/kaggle/input/models/metaresearch/dinov2/pytorch/small/1',
        '/kaggle/input/dinov2/PyTorch/small/1',
        '/kaggle/input/dinov2/pytorch/small/1',
    ], ['config.json', 'pytorch_model.bin'])
ROOT = find_root()
DINOV2_SOURCE = find_dinov2('small')
log(f'input root: {ROOT}')
_STARTUP_PATH_S = _startup_time.perf_counter() - _STARTUP_T0 - _STARTUP_GPU_IMPORT_S
print(f'[startup] competition-root preflight: {_STARTUP_PATH_S:.2f}s', flush=True)
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
log(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')
rsna_initialize(pd.read_csv(ROOT/'test.csv',dtype={'StudyInstanceUID':str}).StudyInstanceUID.tolist())
os.environ['RSNA_COMP_ROOT']=str(ROOT)
rsna_asset_preflight()

# Resolve unsupported A5 metadata before long model inference, without inventing labels.
_meta = pd.read_csv(ROOT/'test_series.csv', dtype={'StudyInstanceUID':str})
_required = {'StudyInstanceUID','Anatomical_Plane','Fat_Suppression'}
if not _required.issubset(_meta.columns):
    raise RuntimeError('Missing required series metadata columns: '+str(sorted(_required-set(_meta.columns))))
_a5_matches = _meta.Anatomical_Plane.isin(['Sagittal','Coronal','Axial']) & _meta.Fat_Suppression.isin([0,1])
_a5_ids = set(_meta.loc[_a5_matches,'StudyInstanceUID'])
_a5_unavailable = [u for u in _RSNA_TEST_IDS if u not in _a5_ids]
rsna_json('/kaggle/working/diagnostics/a5_metadata_applicability.json', {
    'studies':len(_RSNA_TEST_IDS), 'no_matching_acquisition_uids':_a5_unavailable,
    'policy':'base-only contribution for explicit A5 absence; model failures remain fatal'})
rsna_phase('input_metadata', 'COMPLETE', a5_without_matching_acquisition=len(_a5_unavailable),
           missing_fluid_flags=int(_meta.Fluid_Sensitive.isna().sum()) if 'Fluid_Sensitive' in _meta else len(_meta))
del _meta, _a5_matches, _a5_ids, _a5_unavailable
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_012, 12, '2fcf3a2e0a3d7e52ac09e60015be1e5a0ee45e147935c43fe848f5a10e8eeab5')


In [ ]:
GPU_STAGE_013 = r'''HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    key=(str(ROOT.resolve()),str(split))
    if key in _RSNA_HEADERS_MEMO:
        return _RSNA_HEADERS_MEMO[key].copy(deep=True)
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    result=pd.DataFrame(rows)
    _RSNA_HEADERS_MEMO[key]=result.copy(deep=True)
    return result

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_013, 13, 'a773890aca786656f025af0f008fd97bcd0736eb143698d71cd16448bb8fce76')


In [ ]:
GPU_STAGE_014 = r'''def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_014, 14, 'a554f7d0f44a197e371a0ce3ce734466c2658bae27c09d6ffd7855cb4210f1a5')


In [ ]:
GPU_STAGE_015 = r'''ORDER_TAGS = [(32, 50), (32, 55), (32, 19), (40, 48)]
DECODE_FAILED = []
RAPTOR_HEADER_CACHE = {}


def _natural_key(name):
    return tuple(int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', str(name)))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    raptor_records = []
    raptor_safe = True
    for f in files:
        k = None
        ds = None
        file_raptor_safe = True
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            file_raptor_safe = False
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        # Reuse only geometry that the original Raptor reader can interpret
        # unambiguously. Malformed geometry keeps the original reader fallback.
        if file_raptor_safe:
            file_raptor_safe = bool(np.isfinite(k) and np.isfinite(iop).all() and np.isfinite(ipp).all())
        if file_raptor_safe:
            try:
                spacing = getattr(ds, 'PixelSpacing', None)
                spacing = float(spacing[0]) if spacing is not None else 0.5
                if not np.isfinite(spacing) or spacing <= 0:
                    file_raptor_safe = False
                else:
                    raptor_records.append((k, os.path.join(d, f), spacing))
            except Exception:
                file_raptor_safe = False
        raptor_safe &= file_raptor_safe
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (sorted(files,key=_natural_key), False)
    # Cache only the unambiguous geometric order. Every fallback still uses
    # Raptor's original reader, so slice selection cannot change.
    if raptor_safe and len({item[0] for item in raptor_records}) == len(files):
        raptor_records.sort(key=lambda item: item[0])
        spacings = [item[2] for item in raptor_records]
        RAPTOR_HEADER_CACHE[str(d)] = (
            [(item[1], item[2]) for item in raptor_records],
            float(np.median(spacings)) if spacings else 0.5,
        )
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    """Same healthy pixels as BTKD; failed slices cannot erase good native planes."""
    n_slice = GROUP if n_slice is None else int(n_slice)
    out_size = IMG if out_size is None else int(out_size)
    files, d, px = rec.get('ordered') or rec['files'], rec['dir'], rec['px']
    if not files:
        rsna_event('slot_no_files', series=str(rec.get('SeriesInstanceUID', d)), dir=str(d))
        return None
    lo, hi = int(SLICE_BAND[0]*(len(files)-1)), int(SLICE_BAND[1]*(len(files)-1))
    idx = np.unique(np.linspace(lo,hi,n_slice).astype(int)) if hi>lo else np.array([len(files)//2])
    while len(idx)<n_slice: idx=np.append(idx,idx[-1])
    decoded, errors = {}, []
    for i in sorted(set(int(v) for v in idx[:n_slice])):
        path=os.path.join(d,files[i])
        try:
            ds=pydicom.dcmread(path,force=True)
            a=ds.pixel_array.astype(np.float32)
            a=a*float(getattr(ds,'RescaleSlope',1) or 1)+float(getattr(ds,'RescaleIntercept',0) or 0)
            if a.ndim!=2 or not np.isfinite(a).all(): raise ValueError('expected finite 2-D pixels')
            decoded[i]=a
        except Exception as exc:
            decoded[i]=None; errors.append({'path':path,'error':f'{type(exc).__name__}: {exc}'})
    planes=[decoded[int(i)] for i in idx[:n_slice]]
    got=[i for i,p in enumerate(planes) if p is not None]
    if errors:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID',d))
        rsna_event('partial_slice_decode',series=str(rec.get('SeriesInstanceUID',d)),errors=errors)
    if not got:
        rsna_event('slot_all_decode_failed', series=str(rec.get('SeriesInstanceUID', d)), dir=str(d), errors=errors[:5])
        return None
    from collections import Counter
    shape=Counter(planes[i].shape for i in got).most_common(1)[0][0]
    odd=[i for i in got if planes[i].shape!=shape]
    if odd:
        rsna_event('slot_shape_mismatch', series=str(rec.get('SeriesInstanceUID', d)), dir=str(d), majority=[int(v) for v in shape], dropped=len(odd))
        for i in odd: planes[i]=None
        got=[i for i in got if planes[i] is not None]
    for i,p in enumerate(planes):
        if p is None:
            planes[i]=np.zeros(shape,np.float32) if RULES['decode_fill']=='zero' else planes[min(got,key=lambda j:abs(j-i))]
    vol=np.stack(planes)
    if px and np.isfinite(px) and px>0:
        want=int(round(CROP_MM/px)); h,w=shape
        if 16<want<min(h,w):
            cy,cx=h//2,w//2; half=want//2
            vol=vol[:,max(0,cy-half):cy+half,max(0,cx-half):cx+half]
    lo_v,hi_v=np.percentile(vol,[1,99])
    vol=np.clip((vol-lo_v)/max(hi_v-lo_v,1e-6),0,1)
    t=torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t=F.interpolate(t,size=(out_size,out_size),mode='bilinear',align_corners=False)
    return (t.squeeze(0)*255).round().clamp(0,255).to(torch.uint8)
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_015, 15, 'a9ca46d25764c895248403e162605da09d9aeae03aad342058571b31005705e7')


In [ ]:
GPU_STAGE_016 = r'''def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_016, 16, 'ba17bd3298fc57f07ef581794ee274b5d4990b62a999086881ed6f57bcd882aa')


In [ ]:
GPU_STAGE_017 = r'''def build_cache(slot_map, plane_map, lat_map, tag):
    studies = list(_RSNA_TEST_IDS) if _RSNA_TEST_IDS is not None else sorted(slot_map)
    if set(studies)!=set(slot_map):
        rsna_event('coverage_mismatch',tag=tag,missing=[str(s) for s in studies if s not in slot_map][:20],extra=len(set(slot_map)-set(studies)))
        slot_map={s:slot_map.get(s,{}) for s in studies}
    sidx={s:i for i,s in enumerate(studies)}
    cache=rsna_array((len(studies),N_SLOT,CACHE_SLICES,IMG,IMG),tag)
    mask=np.zeros((len(studies),N_SLOT),np.float32)
    jobs=[(st,k,plane,slot_map[st][name]) for st in studies for k,(name,plane,_,_) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: {len(studies)} studies, {len(jobs)} slot-series, {cache.nbytes/2**30:.2f} GiB pixels')
    missing=[]; reused=0
    for job in jobs:
        rec=job[3]
        key=(str(Path(rec['dir']).resolve()),RULES['order'],tuple(sorted(rec['files'])))
        saved=_RSNA_ORDER_MEMO.get(key)
        if saved is not None:
            rec['ordered']=list(saved[0]);reused+=1
        else: missing.append((job,key))
    def ordered(item):
        job,key=item
        try:
            files,good=order_slices(job[3])
            if len(files)!=len(job[3]['files']) or set(files)!=set(job[3]['files']):
                raise ValueError(f'ordering lost/added files ({len(files)} vs {len(job[3]["files"])})')
        except Exception as exc:
            rsna_event('ordering_fallback',tag=tag,series=str(job[3].get('SeriesInstanceUID')),error=f'{type(exc).__name__}: {exc}')
            return key,sorted(job[3]['files']),False
        return key,files,bool(good)
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for start in range(0,len(missing),256):
            rsna_deadline(tag+' ordering')
            block=missing[start:start+256]
            for (job,_),(key,files,good) in zip(block,pool.map(ordered,block)):
                job[3]['ordered']=files;_RSNA_ORDER_MEMO[key]=(tuple(files),good)
                if not good: rsna_event('deterministic_order_fallback',series=str(job[3].get('SeriesInstanceUID')))
    done=0; unfilled=0
    def _tolerant_read_slot(j):
        try: return read_slot(j[3],CACHE_SLICES,IMG)
        except Exception as exc:
            rsna_event('slot_read_error',tag=tag,study=str(j[0]),series=str(j[3].get('SeriesInstanceUID')),error=f'{type(exc).__name__}: {exc}'); return None
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for start in range(0,len(jobs),128):
            rsna_deadline(tag+' decoding')
            block=jobs[start:start+128]
            for (st,k,plane,rec),img in zip(block,pool.map(_tolerant_read_slot,block)):
                done+=1
                if img is None:
                    unfilled+=1; rsna_event('slot_unfilled',tag=tag,study=str(st),series=str(rec.get('SeriesInstanceUID')),slot=int(k)); continue
                cache[sidx[st],k]=normalise_laterality(img,plane,lat_map.get(st)).numpy()
                mask[sidx[st],k]=1
    if unfilled: log(f'{tag}: {unfilled} slot(s) left empty and masked (flagged)')
    _empty=[studies[i] for i in np.flatnonzero(mask.sum(1)==0)]
    if _empty:
        rsna_event('empty_study_rows',tag=tag,count=len(_empty),studies=[str(s) for s in _empty])
        log(f'{tag}: {len(_empty)} study(ies) with no series matching this layout kept as all-zero masked rows (parent policy): {_empty[:5]}')
    if done!=len(jobs):
        raise RuntimeError(f'{tag}: incomplete cache; {done}/{len(jobs)} slots')
    if isinstance(cache,np.memmap): cache.flush()
    rsna_event('cache_complete',tag=tag,studies=len(studies),slot_jobs=len(jobs),filled=done,order_cache_hits=reused,bytes=int(cache.nbytes))
    log(f'{tag}: COMPLETE {done}/{len(jobs)}; reused ordering for {reused} series')
    return studies,cache,mask
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_017, 17, 'a1334e4b2174f1512d3dcd7f46ba257d698b83a6e399b8c1e6c2fa0901f29b8a')


In [ ]:
GPU_STAGE_018 = r'''class SlotHead(nn.Module):

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_018, 18, '6878773f8ffa7883670b9bfceed886f80465514cea3ab30057c7378406531507')


In [ ]:
GPU_STAGE_019 = r'''class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_019, 19, '17434f44700a37e344f33040cf12618806370b438c0f2b2f8fe3caa8b42af065')


In [ ]:
GPU_STAGE_020 = r'''def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    from transformers import AutoModel
    p = source if source is not None else DINOV2_SOURCE
    if p is None:
        raise FileNotFoundError('DINOv2 weights not attached')
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))
    log(f'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}')
    return Model(bb, dim, pool=pool, prior=prior)
_speed_original_build_model = build_model
"""Reuse a CPU architecture template; every member still strict-loads and fingerprints."""
import copy as _speed_copy
import threading as _speed_dino_threading

_speed_dino_templates = {}
_speed_dino_lock = _speed_dino_threading.RLock()


def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    key = (int(unfreeze_last), None if source is None else str(source), variant,
           pool, bool(prior), N_SLOT, tuple(TARGETS))
    with _speed_dino_lock:
        if key not in _speed_dino_templates:
            template = _speed_original_build_model(unfreeze_last, source, variant, pool, prior)
            assert all(p.device.type == 'cpu' for p in template.parameters())
            _speed_dino_templates[key] = template
        # No mutable state or GPU tensors shared between independently trained models.
        return _speed_copy.deepcopy(_speed_dino_templates[key])
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_020, 20, '0cee81943e6bfbb987380cf9c88b2ed961acde926d96211154c4b2946f6e849e')


### DINO shared-prefix ensemble


In [ ]:
GPU_STAGE_021 = r'''SHARED_DINO_PREFIX_LAYERS = 6
SHARED_DINO_PREFIX_SHA256 = '0a55b893bde971c864ea5aeea443f075b17c084d13aa36f30bd1c7863f655cf9'
def _shared_prefix_state_sha256(state):
    digest = hashlib.sha256()
    prefixes = (
        'backbone.embeddings.',
        *(
            f'backbone.encoder.layer.{index}.'
            for index in range(
                SHARED_DINO_PREFIX_LAYERS
            )
        ),
    )
    keys = sorted(
        key
        for key in state
        if key.startswith(prefixes)
    )
    if len(keys) != 113:
        raise WeightsError(
            'shared DINOv2 prefix key-count drift: '
            f'{len(keys)} != 113'
        )
    for key in keys:
        tensor = state[key].detach().cpu().contiguous()
        digest.update(key.encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

def _shared_prefix_eligible(members):
    if not members or not DEVS:
        return False
    return all(
        'state' not in member
        and int(member['config']['unfreeze_last'])
        == SHARED_DINO_PREFIX_LAYERS
        and member['config']['variant'] == 'small'
        and member['config'].get('pool', 'cls_mean')
        in POOL_PARTS
        for member in members
    )

def _shared_dino_prefix(model, images, img_size):
    batch, slots = images.shape[:2]
    values = images.reshape(
        batch * slots,
        *images.shape[2:],
    ).float().div_(255.0)
    if (
        img_size is not None
        and img_size != values.shape[-1]
    ):
        values = F.interpolate(
            values,
            size=(img_size, img_size),
            mode='bilinear',
            align_corners=False,
        )
    values = (
        values - model.mean
    ) / model.std
    hidden = model.backbone.embeddings(values)
    for layer in model.backbone.encoder.layer[
        :SHARED_DINO_PREFIX_LAYERS
    ]:
        hidden = layer(hidden)
        hidden = hidden[0] if isinstance(hidden, (tuple, list)) else hidden
    return hidden

def _shared_dino_tail(
    model,
    hidden,
    batch,
    slots,
    slot_mask,
):
    value = hidden
    for layer in model.backbone.encoder.layer[
        SHARED_DINO_PREFIX_LAYERS:
    ]:
        value = layer(value)
        value = value[0] if isinstance(value, (tuple, list)) else value
    value = model.backbone.layernorm(value)
    patch = value[:, 1:]
    parts = [
        value[:, 0],
        patch.mean(1),
    ]
    if model.pool == 'cls_mean_focal':
        count = max(
            1,
            patch.shape[1] // 8,
        )
        parts.append(
            patch.topk(
                count,
                dim=1,
            ).values.mean(1)
        )
    feature = torch.cat(
        parts,
        dim=1,
    ).reshape(
        batch,
        slots,
        -1,
    )
    return model.head(feature, slot_mask)

@torch.no_grad()
def _predict_shared_dino_members(
    entries,
    cache,
    mask,
    idx,
    dev,
    img_size,
    group,
    starts,
):
    target_idx = {
        target: index
        for index, target in enumerate(TARGETS)
    }
    states = {}
    for member, model, jitter in entries:
        generator = torch.Generator(device=dev)
        jitter_seed = (
            SEED
            + int(
                hashlib.sha256(
                    str(member['id']).encode()
                ).hexdigest()[:8],
                16,
            )
        )
        generator.manual_seed(
            int(jitter_seed) % (2 ** 63 - 1)
        )
        states[member['id']] = {
            'out': [],
            'public': [],
            'soft': [],
            'generator': generator,
            'jitter': jitter,
        }
        model.eval()

    reference = entries[0][1]
    for batch_start in range(
        0,
        len(idx),
        EVAL_BATCH,
    ):
        selected = idx[
            batch_start:
            batch_start + EVAL_BATCH
        ]
        slot_mask = torch.from_numpy(
            mask[selected]
        ).to(dev)
        batch = len(selected)
        slots = cache.shape[1]
        member_windows = {
            member['id']: (
                [],
                [],
                [],
            )
            for member, _, _ in entries
        }

        for start in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(
                    cache[
                        selected,
                        :,
                        start:start + group,
                    ]
                )
            ).to(dev)
            def _dino_shared_forward(_amp):
                with torch.autocast('cuda', enabled=_amp):
                    _common = _shared_dino_prefix(reference, rows, img_size)
                    return {member['id']: _shared_dino_tail(model, _common, batch, slots, slot_mask).float() for member, model, _ in entries}
            original_logits = _dino_shared_forward(dev.type == 'cuda')
            if dev.type == 'cuda' and not all(bool(torch.isfinite(v).all()) for v in original_logits.values()):
                rsna_event('dino_fp16_nonfinite_retry_fp32', window_start=int(start))
                original_logits = _dino_shared_forward(False)

            for member, model, jitter in entries:
                member_id = member['id']
                view_logits = [
                    original_logits[member_id]
                ]
                if jitter:
                    jittered = augment(
                        rows,
                        generator=states[
                            member_id
                        ]['generator'],
                    )
                    with torch.autocast(
                        'cuda',
                        enabled=dev.type == 'cuda',
                    ):
                        jitter_hidden = (
                            _shared_dino_prefix(
                                reference,
                                jittered,
                                img_size,
                            )
                        )
                        view_logits.append(
                            _shared_dino_tail(
                                model,
                                jitter_hidden,
                                batch,
                                slots,
                                slot_mask,
                            ).float()
                        )
                    if dev.type == 'cuda' and not bool(torch.isfinite(view_logits[-1]).all()):
                        rsna_event('dino_jitter_fp16_nonfinite_dropped', member=str(member_id))
                        view_logits.pop()
                view_probs = [
                    torch.sigmoid(value)
                    for value in view_logits
                ]
                probabilities, logits, originals = (
                    member_windows[member_id]
                )
                logits.append(
                    torch.stack(
                        view_logits
                    ).mean(0)
                )
                probabilities.append(
                    torch.stack(
                        view_probs
                    ).mean(0)
                )
                originals.append(
                    view_probs[0]
                )

        for member, _, _ in entries:
            member_id = member['id']
            win_probs, win_logits, win_originals = (
                member_windows[member_id]
            )
            probabilities = torch.stack(win_probs)
            logits = torch.stack(win_logits)
            originals = torch.stack(win_originals)
            value = (
                torch.sigmoid(logits.mean(0))
                if TTA_POOL == 'logit'
                else probabilities.mean(0)
            )
            value = apply_target_window_pool(
                value,
                probabilities,
                logits,
                originals,
                TTA_TARGET_POOL,
                target_idx,
            )
            states[member_id]['out'].append(
                value.cpu().numpy()
            )
            public_value = apply_target_window_pool(
                originals.mean(0),
                originals,
                logits,
                originals,
                PUBLIC_FRONTIER_TARGET_POOL,
                target_idx,
            )
            states[member_id]['public'].append(
                public_value.cpu().numpy()
            )
            states[member_id]['soft'].append(
                legacy_fold_soft_window_pool(
                    originals,
                    target_idx,
                ).cpu().numpy()
            )

    return {
        member['id']: (
            np.concatenate(states[member['id']]['out']),
            np.concatenate(states[member['id']]['public']),
            np.concatenate(states[member['id']]['soft']),
        )
        for member, _, _ in entries
    }

@torch.inference_mode()
def _check_shared_dino_path(model,dev,img_size,tag):
    model.eval()
    g=torch.Generator().manual_seed(SEED)
    im=torch.randint(0,256,(2,N_SLOT,GROUP,img_size,img_size),generator=g,dtype=torch.uint8).to(dev)
    mask=torch.ones(2,N_SLOT,device=dev);mask[1,-1]=0
    direct=model(im,mask,img_size).float()
    split=_shared_dino_tail(model,_shared_dino_prefix(model,im,img_size),2,N_SLOT,mask).float()
    if not torch.isfinite(split).all() or not torch.allclose(split,direct,rtol=1e-5,atol=2e-5):
        raise RuntimeError(f'{tag}: shared-prefix execution differs from full forward')
    rsna_event('dino_shared_path_parity',member=tag,max_abs=float((split-direct).abs().max()))

def _run_shared_dino_group(
    path,
    members,
    cache,
    mask,
    idx,
    starts,
):
    ordered = sorted(
        members,
        key=lambda member: -(
            member.get('holdout') or 0
        ),
    )
    # run_dinov2 retains only submission_public_0899.csv, whose member
    # predictions are built exclusively from the unaugmented views.
    # Jittering here affected only the intermediate submission.csv that the
    # retained public-frontier file replaces at the end of this stage.
    plans = [
        (
            member,
            False,
        )
        for member in ordered
    ]
    assignments = [
        plans[index::len(DEVS)]
        for index in range(len(DEVS))
    ]
    results = {}
    result_lock = threading.Lock()

    def worker(dev, plans_for_device):
        global _DINOV2_MATCHED_MEMBERS
        loaded = []
        for member, jitter in plans_for_device:
            with BUILD_LOCK:
                checkpoint = torch.load(
                    Path(path) / member['file'],
                    map_location='cpu',
                    weights_only=False,
                )
                state = checkpoint['model']
                prefix_sha256 = (
                    _shared_prefix_state_sha256(
                        state
                    )
                )
                if (
                    prefix_sha256
                    != SHARED_DINO_PREFIX_SHA256
                ):
                    raise WeightsError(
                        f"{member['id']}: frozen prefix "
                        'hash mismatch: '
                        f'{prefix_sha256}'
                    )
                model = build_model(
                    int(
                        member['config'][
                            'unfreeze_last'
                        ]
                    ),
                    variant=member['config'][
                        'variant'
                    ],
                    pool=member['config'].get(
                        'pool',
                        'cls_mean',
                    ),
                    prior=bool(
                        member['config'].get(
                            'prior',
                            False,
                        )
                    ),
                ).to(dev)
                model.load_state_dict(state)
                check_fingerprint(
                    model,
                    dev,
                    IMG,
                    checkpoint.get('fingerprint'),
                    tag=f"{member['id']}: ",
                )
                _check_shared_dino_path(model, dev, IMG, member['id'])
                _DINOV2_MATCHED_MEMBERS += 1
                del checkpoint, state
            loaded.append(
                (member, model, jitter)
            )

        predicted = _predict_shared_dino_members(
            loaded,
            cache,
            mask,
            idx,
            dev,
            IMG,
            GROUP,
            starts,
        )
        with result_lock:
            results.update(predicted)
        del loaded
        gc.collect()
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()

    with ThreadPoolExecutor(max_workers=len(DEVS)) as pool:
        futures=[pool.submit(worker,dev,plan) for dev,plan in zip(DEVS,assignments)]
        for future in futures: future.result()
    if len(results) != len(ordered):
        raise WeightsError(
            'shared DINOv2 worker did not return '
            f'all members: {len(results)} / '
            f'{len(ordered)}'
        )
    log(
        'shared first 6 DINOv2 blocks for '
        f'{len(ordered)} members; original views '
        f'computed once per {len(DEVS)} device(s)'
    )
    return [
        (
            member,
            results[member['id']],
            jitter,
        )
        for member, jitter in plans
    ]

FINGERPRINT_TOL = 0.002
EXPECTED_DINOV2_MEMBERS = 20
_DINOV2_MATCHED_MEMBERS = 0

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=''):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if not np.isfinite(got).all() or not np.isfinite(exp).all():
        raise WeightsError(f'{tag}: nonfinite fingerprint')
    if got.shape != exp.shape:
        raise WeightsError(f'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to')
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.')
    log(f'{tag}fingerprint matches within {d:.2g}')
    return d

class WeightsError(RuntimeError):
    pass

def find_weights(name='manifest.json'):
    import json
    roots = [
        Path('/kaggle/input/datasets/pilkwang/rsna-knee-weights'),
        Path('/kaggle/input/rsna-knee-weights'),
    ]
    valid = []
    for root in roots:
        path = root / name
        if not path.is_file():
            continue
        try:
            man = json.loads(path.read_text())
        except (OSError, ValueError) as exc:
            raise WeightsError(f'invalid pinned weights manifest at {path}: {exc}') from exc
        if not isinstance(man.get('members'), list) or len(man['members']) != EXPECTED_DINOV2_MEMBERS:
            raise WeightsError(f'{path} must list exactly {EXPECTED_DINOV2_MEMBERS} members')
        if len({m.get('id') for m in man['members']}) != EXPECTED_DINOV2_MEMBERS:
            raise WeightsError(f'{path} has duplicate or missing member ids')
        missing = [m['file'] for m in man['members'] if not (root / m['file']).is_file()]
        if missing:
            raise WeightsError(
                f'{root} lists {len(man["members"])} members but misses {missing[0]!r}'
            )
        valid.append(root)
    if len(valid) != 1:
        raise WeightsError(f'expected one pinned rsna-knee-weights root, found {valid}')
    return valid[0]
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
# No-extra-pass diversity branch: smooth focal pooling is evaluated from
# the same no-jitter public-member windows already used by the parent.
LEGACY_FOLD_SOFTPOOL_BETA = {
    'ACL': 6.0, 'MCL': 6.0,
    'Medial Meniscus': 8.0, 'Lateral Meniscus': 8.0,
    "Baker's": 8.0, 'Contusion': 8.0, 'Fracture': 10.0,
}
LEGACY_FOLD_SOFTPOOL_ALPHA = {
    'ACL': 0.20, 'MCL': 0.20,
    'Medial Meniscus': 0.25, 'Lateral Meniscus': 0.25,
    "Baker's": 0.20, 'Contusion': 0.20, 'Fracture': 0.15,
}
LEGACY_MEMBER_WEIGHT_BY_TARGET = {'Lateral Meniscus': 15.0, 'Medial OA': 2.5, 'Lateral OA': 15.0, 'Contusion': 5.0}

def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]

def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == 'max':
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == 'mean':
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == 'logit_mean':
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == 'original_mean':
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ('top2', 'top3'):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f'unknown TTA pooling mode for {target}: {mode}')
    return values

def legacy_fold_soft_window_pool(original_probs, target_idx):
    values = original_probs.mean(0).clone()
    for target, beta in LEGACY_FOLD_SOFTPOOL_BETA.items():
        j = target_idx[target]
        x = original_probs[:, :, j]
        weight = torch.softmax(float(beta) * x, dim=0)
        values[:, j] = (weight * x).sum(0)
    return values

@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError('predict_member was given no windows to average over')
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f'unknown target(s) in TTA_TARGET_POOL: {unknown}')
    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    model.eval()
    out, public_frontier_out, public_soft_out = ([], [], [])
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = ([], [], [])
        for st in starts:
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = ([], [])
            for view in views:
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    z = model(view, m, img_size).float()
                if dev.type == 'cuda' and not bool(torch.isfinite(z).all()):
                    rsna_event('dino_legacy_fp16_nonfinite_retry_fp32')
                    with torch.autocast('cuda', enabled=False):
                        z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            win_original_probs.append(view_probs[0])
        probs = torch.stack(win_probs)
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = torch.sigmoid(logits.mean(0)) if pool == 'logit' else probs.mean(0)
        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            public_frontier_out.append(public_v.cpu().numpy())
            public_soft = legacy_fold_soft_window_pool(original_probs, target_idx)
            public_soft_out.append(public_soft.cpu().numpy())
    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)
    if not return_public_frontier:
        return primary
    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)
    public_soft = np.concatenate(public_soft_out) if public_soft_out else np.zeros((0, len(TARGETS)), np.float32)
    return (primary, public_frontier, public_soft)
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()
LEGACY_BUNDLE_FILE = 'rsna_20260807_v1.pt'
LEGACY_WEIGHT = 0.5

def find_legacy_bundle():
    candidates = [
        Path('/kaggle/input/datasets/pilkwang/rsna-knee-weights') / LEGACY_BUNDLE_FILE,
        Path('/kaggle/input/rsna-knee-weights') / LEGACY_BUNDLE_FILE,
    ]
    hits = [path for path in candidates if path.is_file()]
    if len(hits) > 1:
        raise WeightsError(f'ambiguous legacy bundle: {hits}')
    return hits[0] if hits else None

def legacy_group_members():
    return {}

def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    t0 = time.time()
    with BUILD_LOCK:
        if 'state' in m:
            raise WeightsError(f"{m['id']}: inline legacy state is forbidden")
        ck = torch.load(Path(path) / m['file'], map_location='cpu', weights_only=False)
        state, fp = (ck['model'], ck.get('fingerprint'))
        if fp is None:
            raise WeightsError(f"{m['id']}: stored fingerprint is required")
        model = build_model(int(m['config']['unfreeze_last']), variant=m['config']['variant'], pool=m['config'].get('pool', 'cls_mean'), prior=bool(m['config'].get('prior', False))).to(dev)
        model.load_state_dict(state)
        check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        global _DINOV2_MATCHED_MEMBERS
        _DINOV2_MATCHED_MEMBERS += 1
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m['id']).encode()).hexdigest()[:8], 16)
    public_member = 'state' not in m
    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)
    if public_member:
        p, public_p, public_soft = predicted
    else:
        p, public_p, public_soft = (predicted, None, None)
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == 'cuda':
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return (p, public_p, public_soft, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))

def _combine(per_member):
    all_ids = sorted({s for m in per_member for s in m['ids']})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get('target_weight')
        w = np.asarray(target_weight if target_weight is not None else [float(m.get('weight', 1.0))] * len(TARGETS), dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        rsna_finite(m['pred'],str(m['id'])+' retained raw',probability=True)
        r = pd.DataFrame(m['pred']).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m['ids']]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f'at least one target has no ensemble vote: {tot}')
    return (all_ids, acc / tot[None, :])

def combine_public_members_by_fold(per_member, pred_key='pred'):
    # Raw-average the four members within each fold, rank each fold,
    # then give all five folds equal weight.
    all_ids = sorted({study for member in per_member for study in member['ids']})
    position = {study: i for i, study in enumerate(all_ids)}
    groups = {}
    for i, member in enumerate(per_member):
        fold = member.get('fold')
        key = f'fold_{fold}' if fold is not None else f'member_{i}'
        groups.setdefault(key, []).append(member)
    fold_ranks, diagnostics = ([], [])
    for key, members_in_fold in sorted(groups.items()):
        matrices = []
        for member in members_in_fold:
            values = np.full((len(all_ids), len(TARGETS)), np.nan, np.float64)
            values[[position[study] for study in member['ids']]] = np.asarray(member[pred_key], np.float64)
            if np.isnan(values).any():
                raise WeightsError(f"{member.get('id')}: incomplete {pred_key} coverage")
            matrices.append(values)
        raw_fold_mean = np.mean(matrices, axis=0)
        fold_ranks.append(pd.DataFrame(raw_fold_mean).rank(method='average', pct=True).to_numpy(np.float64))
        diagnostics.append({'ensemble_group': key, 'members': len(members_in_fold)})
    if len(fold_ranks) != 5:
        raise WeightsError(f'legacy branch requires five folds, found {len(fold_ranks)}')
    return all_ids, np.mean(fold_ranks, axis=0), pd.DataFrame(diagnostics)

def blend_legacy_frontier_and_soft(frontier_rank, soft_rank):
    output = np.asarray(frontier_rank, np.float64).copy()
    for j, target in enumerate(TARGETS):
        alpha = float(LEGACY_FOLD_SOFTPOOL_ALPHA.get(target, 0.0))
        if alpha:
            output[:, j] = (1.0 - alpha) * frontier_rank[:, j] + alpha * soft_rank[:, j]
    return output

def infer_from_package(path, dev=None):
    man = json.loads((Path(path) / 'manifest.json').read_text())
    members = man['members']
    if len(members) != EXPECTED_DINOV2_MEMBERS:
        raise WeightsError(f'expected {EXPECTED_DINOV2_MEMBERS} DINOv2 members, found {len(members)}')
    log(f'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)')
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
    hte = annotate(walk('test_series'))
    log(f'test header pass: {len(hte)} series')
    groups = {}
    for m in members:
        groups.setdefault(m['pixel_group'], []).append(m)
    groups.update(legacy_group_members())
    per_member, public_frontier_members = ([], [])
    failures = []
    abort = threading.Event()
    est = {'fixed': None, 'win': None}

    def bank(m, ids, pred, starts, jitter, public_pred=None, public_soft=None):
        if not np.isfinite(pred).all():
            rsna_event('dino_member_nonfinite_neutral', member=str(m['id']), rows=int((~np.isfinite(pred)).reshape(len(pred), -1).any(axis=1).sum()))
            pred = np.where(np.isfinite(pred), pred, 0.5)
        if public_pred is not None and not np.isfinite(public_pred).all():
            rsna_event('dino_member_public_nonfinite_neutral', member=str(m['id']))
            public_pred = np.where(np.isfinite(public_pred), public_pred, 0.5)
        rsna_finite(pred, m['id']+' DINO raw', probability=True)
        if public_pred is not None: rsna_finite(public_pred,m['id']+' DINO retained raw',probability=True)
        rsna_save_predictions('dino_'+str(m['id']),ids,public_pred if public_pred is not None else pred,TARGETS)
        if float(np.std(pred)) < 1e-09:
            raise WeightsError(f"{m['id']}: degenerate predictions")
        with STATE_LOCK:
            per_member.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': pred, 'weight': m.get('weight', 1.0), 'target_weight': m.get('target_weight'), 'holdout': m.get('holdout')})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-09:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': public_pred, 'soft_pred': public_soft})
            elif public_pred is not None:
                raise WeightsError(f"{m['id']}: incomplete public-frontier windows")
            log(f"  banked {m['id']} fold {m.get('fold', '?')} ({len(starts)} window(s); {len(per_member)} member(s)")
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map, lat_of(hte, 'test '), f'test g{gi}')
        idx = np.arange(len(st_te))
        starts_full = window_starts(Cte.shape[2], GROUP)
        if len(groups) == 1 and _shared_prefix_eligible(gm):
            shared_results = _run_shared_dino_group(
                path,
                gm,
                Cte,
                Mte,
                idx,
                starts_full,
            )
            for member, predicted, jitter in shared_results:
                prediction, public, soft = predicted
                bank(
                    member,
                    st_te,
                    prediction,
                    starts_full,
                    jitter,
                    public,
                    soft,
                )
            rsna_release_pixels(Cte)
            del Cte, Mte, shared_results
            gc.collect()
            continue
        pending = sorted(gm, key=lambda m: -(m.get('holdout') or 0))
        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))

        def pop_next():
            with STATE_LOCK:
                if not pending:
                    return (None, None, False)
                rsna_deadline('DINO complete-member inference')
                starts, jit = starts_full, False
                return (pending.pop(0), starts, jit)

        def worker(dev):
            while not abort.is_set():
                m, starts, jit = pop_next()
                if m is None:
                    return
                try:
                    p, public_p, public_soft, (fs, ws) = _run_member(path, m, dev, Cte, Mte, idx, starts, jit)
                    with STATE_LOCK:
                        est['fixed'], est['win'] = (fs, ws)
                    bank(m, st_te, p, starts, jit, public_p, public_soft)
                except Exception as exc:
                    with STATE_LOCK:
                        failures.append((m['id'], str(dev), type(exc).__name__, str(exc)))
                    abort.set()
                    return
        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        if failures:
            raise WeightsError(f'DINOv2 member failure; no fallback permitted: {failures[0]}')
        rsna_release_pixels(Cte)
        del Cte, Mte
        gc.collect()
    if _DINOV2_MATCHED_MEMBERS != EXPECTED_DINOV2_MEMBERS:
        raise WeightsError(f'fingerprint gate failed: {_DINOV2_MATCHED_MEMBERS} / {EXPECTED_DINOV2_MEMBERS}')
    if len(public_frontier_members) != EXPECTED_DINOV2_MEMBERS:
        raise WeightsError(f'public-frontier inference incomplete: {len(public_frontier_members)} / {EXPECTED_DINOV2_MEMBERS} members')
    log(f'DINOv2 fail-closed gate PASS: {_DINOV2_MATCHED_MEMBERS}/{EXPECTED_DINOV2_MEMBERS} fingerprints and members')
    frontier_ids, frontier_acc = _combine(public_frontier_members)
    sub = write_submission(frontier_acc, frontier_ids, test_df, '_pipeline_stage.csv')
    log(f'submission.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {sub.shape}')
    return sub

def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg['img'])
    GROUP = int(cfg['group'])
    CACHE_SLICES = int(cfg['slices'])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg['crop_mm'])
    SLICE_BAND = tuple((float(x) for x in cfg['band']))
    rules = cfg.get('rules') or RULES_NATIVE
    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f'the members record pixel rules this pipeline cannot reproduce: {unknown}')
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg['slots']):
        raise WeightsError(f"the members were fitted on slots {cfg['slots']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_021, 21, 'a816bbcb91c3f8645f847d921a7e387df9ba7301c546ed151299a050c6e1e56c')


In [ ]:
GPU_STAGE_022 = r'''def write_submission(pred, studies, test_df, path):
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    if sub[TARGETS].isna().any().any():
        rsna_event('submission_neutral_fill', studies=int(sub[TARGETS].isna().any(axis=1).sum()))
        sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub

def _v37_validate_submission(path, test_df, tag):
    frame = pd.read_csv(path)
    expected = ['StudyInstanceUID'] + TARGETS
    if list(frame.columns) != expected:
        raise ValueError(f'{tag}: columns differ from the competition contract')
    if len(frame) != len(test_df) or not frame['StudyInstanceUID'].is_unique:
        raise ValueError(f'{tag}: row count or StudyInstanceUID uniqueness failed')
    if set(frame['StudyInstanceUID'].astype(str)) != set(test_df['StudyInstanceUID'].astype(str)):
        raise ValueError(f'{tag}: StudyInstanceUID set differs from test.csv')
    values = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(values).all():
        raise ValueError(f'{tag}: non-finite prediction')
    return frame

def main():
    pkg = find_weights()
    if pkg is None:
        raise WeightsError('required public checkpoint manifest not found')
    infer_from_package(pkg, DEVS[0])
    _v37_validate_submission('_pipeline_stage.csv', pd.read_csv(ROOT / 'test.csv'), 'public frontier')
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_022, 22, '9006955666c6c5f89bd87d606b83d5fba2d446c42168e223491e11b8b8f95102')


In [ ]:
GPU_STAGE_023 = r'''rsna_phase('dinov2', 'START')
main()
log('done')

_speed_dino_templates.clear()
gc.collect()

rsna_phase('dinov2', 'COMPLETE')
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_023, 23, 'cd7a8762ef90ccf507eaf4588e350402a4f8b1f0f58b7f6687b135eee684d357')


### A5 five-fold ensemble


In [ ]:
GPU_STAGE_025 = r'''if globals().get('_DINOV2_MATCHED_MEMBERS') != 20:
    raise RuntimeError('DINOv2 20/20 fingerprint gate did not pass')
rsna_phase('a5', 'START')
_A5_SAVED = dict(globals())
import gc, os, time, warnings
from concurrent.futures import ThreadPoolExecutor as A5DecodePool, as_completed
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)
CROP_MM = 130.0
SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICE = 16
INTENSITY = 'slice'
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

def _one_a5_root(tag, candidates, marker):
    hits = [Path(path) for path in candidates if Path(path).is_dir() and (Path(path) / marker).exists()]
    if len(hits) != 1:
        raise RuntimeError(f'{tag}: expected one direct root, found {hits}')
    return hits[0]

COMP = Path(ROOT)
CKPT = _one_a5_root('A5 weights', [
    '/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights',
    '/kaggle/input/knee-mri-fold-weights',
], 'm_f0.pt')
assert COMP is not None, 'competition data not attached'
assert CKPT is not None, 'fold weights not attached'
assert (COMP / 'sample_submission.csv').exists(), f'no competition data at {COMP}'
assert list(CKPT.glob('*_f*.pt')), f'no checkpoints at {CKPT}'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_025, 25, '6fffd33cf27daf23afcae0616f5fac9ffc7e869e08cd7827f3d4f141276f2611')


In [ ]:
GPU_STAGE_026 = r'''SERIES_ROOT = COMP / 'test_series'
if not SERIES_ROOT.is_dir():
    raise FileNotFoundError('Missing test_series; train-series fallback is forbidden')
print('series root:', SERIES_ROOT)

def ordered_files(sdir, cap=64):
    # Preserve the parent's InstanceNumber ordering; remove pre-sort truncation.
    # The unused cap argument remains only for call compatibility.
    keyed=[]; unreadable=[]
    for f in sorted(sdir.glob('*.dcm')):
        try:
            ds=pydicom.dcmread(str(f),stop_before_pixels=True)
            key=int(ds.InstanceNumber)
        except Exception as exc:
            unreadable.append(str(f)); continue
        keyed.append((key,str(f)))
    if unreadable:
        rsna_event('a5_order_unreadable_dropped',series=str(sdir),unreadable=len(unreadable),readable=len(keyed),examples=unreadable[:3])
    return [f for _,f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(args):
    idx,study,recs=args
    out=np.zeros((N_SLOT,N_SLICE,SIZE,SIZE),np.uint8);mask=np.zeros(N_SLOT,np.uint8)
    rows=pd.DataFrame(recs)
    if not len(rows):
        rsna_event('a5_no_metadata',study=str(study)); return idx,out,mask
    for s_i,(plane,fs) in enumerate(SLOTS):
        sub=rows[(rows.Anatomical_Plane==plane)&(rows.Fat_Suppression==fs)]
        if sub.empty: continue  # Actual acquisition absence, not a decoding failure.
        series=str(sub.iloc[0].SeriesInstanceUID)
        files=ordered_files(SERIES_ROOT/str(study)/series)
        if not files:
            rsna_event('a5_slot_no_files',study=str(study),series=series); continue
        flip=plane!='Sagittal' and series_side(files[0])<0
        i0=int(round(SLICE_BAND[0]*(len(files)-1)));i1=int(round(SLICE_BAND[1]*(len(files)-1)))
        avail=list(range(i0,i1+1))
        if len(avail)>=N_SLICE:
            picks=[avail[int(round(t))] for t in np.linspace(0,len(avail)-1,N_SLICE)];off=0
        else:picks,off=avail,(N_SLICE-len(avail))//2
        rendered=0;errors=[]
        if INTENSITY=='series':
            crops=[read_crop(files[p]) for p in picks];got=[x for x in crops if x is not None]
            if not got:
                rsna_event('a5_slot_all_crops_failed',study=str(study),series=series); continue
            samp=np.concatenate([x[::4,::4].ravel() for x in got]);lo_,hi_=np.percentile(samp,[1,99])
            for c,x in enumerate(crops):
                if x is None:x=read_crop(files[min(len(files)-1,picks[c]+1)])
                if x is None:errors.append(files[picks[c]]);continue
                img=window(x,lo_,hi_,flip)
                rsna_finite(img,'A5 rendered pixels')
                out[s_i,off+c]=(img*255).astype(np.uint8);rendered+=1
        else:
            for c,p in enumerate(picks):
                img=render(files[p],flip)
                if img is None:img=render(files[min(len(files)-1,p+1)],flip)
                if img is None:errors.append(files[p]);continue
                rsna_finite(img,'A5 rendered pixels')
                out[s_i,off+c]=(img*255).astype(np.uint8);rendered+=1
        if errors or rendered!=len(picks):
            rsna_event('a5_decode_failure',study=str(study),series=series,failed=errors,rendered=int(rendered),wanted=int(len(picks)))
            if not rendered: continue
        mask[s_i]=rendered
    if not mask.any():
        rsna_event('a5_empty_study',study=str(study))
        print(f'[a5] {study}: no acquired slots; all-zero mask kept (parent policy)', flush=True)
    return idx,out,mask
sub_df = pd.read_csv(COMP / 'sample_submission.csv',dtype={'StudyInstanceUID':str})
ser_csv = pd.read_csv(COMP / 'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
if set(sub_df.StudyInstanceUID)!=set(_RSNA_TEST_IDS):
    raise RuntimeError('A5 test/sample UID mismatch')
ser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]
studies = sub_df.StudyInstanceUID.tolist()
by = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}
print(f'{len(studies):,} test studies, {len(by):,} with series metadata')
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_026, 26, 'f1fa6aa2130ce82d2efb84793f47c50c97b71e3e61c68ffc5840c4431e7a5f5e')


In [ ]:
GPU_STAGE_027 = r'''N_SLOT_TYPES, MASK_IDX = (6, 0)

def segment_softmax(scores, sidx, B):
    T, K = scores.shape
    idx = sidx.unsqueeze(1).expand(-1, K)
    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
    e = (scores - m[sidx]).exp()
    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
    return e / s[sidx].clamp(min=1e-06)

class MeanMaxPool(nn.Module):

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        D = f.shape[1]
        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
        mean = mean / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
        return (torch.cat([mean, mx], 1), None)

class LabelAttentionPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = (d, n_labels, n_heads)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        scores = self.key(f) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        a = segment_softmax(scores, sidx, B)
        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
        return (out, a)

class TokenXAttnPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = (d, n_labels)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tok, sidx, B, slot=None, return_attn=False):
        T, N, D = tok.shape
        cnt = torch.bincount(sidx, minlength=B)
        S = int(cnt.max().item())
        starts = torch.cumsum(cnt, 0) - cnt
        pos = torch.arange(T, device=tok.device) - starts[sidx]
        kv = tok + self.slot_emb(slot).unsqueeze(1)
        pad = tok.new_zeros(B, S, N, D)
        pad[sidx, pos] = kv
        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
        keep[sidx, pos] = True
        kpm = ~keep.repeat_interleave(N, dim=1)
        pad = self.kv_norm(pad.reshape(B, S * N, D))
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        cls = tok[:, 0]
        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
        return (torch.cat([att, base], -1), w)

class ViTSlotToken(nn.Module):

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        d = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for blk in vit.blocks:
            a = getattr(blk, 'attn', None)
            if a is not None and hasattr(a, 'num_prefix_tokens'):
                a.num_prefix_tokens = a.num_prefix_tokens + 1

    @staticmethod
    def _maybe(mod, x):
        return x if mod is None else mod(x)

    def forward_features(self, x, cat):
        v = self.vit
        x = v.patch_embed(x)
        pos = v._pos_embed(x)
        rope = None
        if isinstance(pos, tuple):
            x, rope = pos
        else:
            x = pos
        x = self._maybe(getattr(v, 'patch_drop', None), x)
        x = self._maybe(getattr(v, 'norm_pre', None), x)
        npt = self._orig_prefix
        tok = self.tok(cat).unsqueeze(1)
        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
        if rope is not None:
            if getattr(v, 'rope_mixed', False):
                for i, blk in enumerate(v.blocks):
                    x = blk(x, rope=rope[i])
            else:
                for blk in v.blocks:
                    x = blk(x, rope=rope)
        else:
            x = v.blocks(x)
        return v.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class _GatedDepthBlock(nn.Module):

    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

class DepthCompress(nn.Module):

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for b in self.blocks:
            z = b(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep
N_PLANE, N_CONTRAST = (3, 2)
_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

class SlotDepthMixer(nn.Module):

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
        self.alpha_max = alpha_max
        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer('base', b.log()[self.r:])
        n_u = self.r + 1
        self.shared = nn.Parameter(torch.zeros(n_u))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        idx = torch.arange(n_slice)
        self.register_buffer('off', idx[None, :] - idx[:, None])

    def kernel(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

    def forward(self, x, slot, vmask):
        T, S, H, W = x.shape
        if vmask is None:
            raise ValueError('stem=mixer requires the padding mask')
        k = self.kernel(slot)
        v = vmask.to(k.dtype)
        d = self.off + self.r
        inb = (d >= 0) & (d < self.ksize)
        kk = k[:, d.clamp(0, self.ksize - 1)] * inb
        M = kk * v[:, None, :]
        den = M.sum(-1, keepdim=True)
        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
        ok = (den > 1e-06) & v[:, :, None].bool()
        M = torch.where(ok, M / den.clamp(min=1e-06), eye)
        a = self.alpha(slot)[:, None, None]
        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

def _seg_mean_max(v, sidx, B):
    D = v.shape[1]
    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
    mean = mean / cnt.clamp(min=1).unsqueeze(1)
    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
    return torch.cat([mean, mx], 1)

def _pad_kv(x, sidx, B, norm):
    T, P, D = x.shape
    cnt = torch.bincount(sidx, minlength=B)
    S = int(cnt.max().item())
    starts = torch.cumsum(cnt, 0) - cnt
    pos = torch.arange(T, device=x.device) - starts[sidx]
    pad = x.new_zeros(B, S, P, D)
    pad[sidx, pos] = x
    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
    keep[sidx, pos] = True
    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

class _GatedDelta(nn.Module):

    def __init__(self, d, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(d)
        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, pat, sidx, B, return_attn):
        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

class TokenResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class CodexResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class ClsAddPool(nn.Module):

    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

class Readout(nn.Module):

    def __init__(self, pool, d, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = (pool, n_labels)
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ('xres', 'clsadd', 'xcodex'):
            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
        elif pool in ('attn', 'xattn'):
            if pool == 'xattn':
                self.pool = TokenXAttnPool(d, n_labels)
                wd = 3 * d + pe
            else:
                self.pool = LabelAttentionPool(d, n_labels)
                wd = d + pe
            self.norm = nn.LayerNorm(wd)
            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, f, slot, sidx, B, return_attn=False):
        pe = self.pres_emb(slot)
        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
            return self.pool(f, slot, sidx, B, pres)[0]
        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
        if self.pool_kind in ('attn', 'xattn'):
            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, pres], 1))

class Net(nn.Module):

    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
        super().__init__()
        self.enc, self.cond = (enc, cond)
        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
        D = enc.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
        self.readout = Readout(pool, D)
        if cond == 'post':
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

    def forward(self, im, slot, smeta, sidx, B, vm=None):
        if self.mixer is not None:
            im = self.mixer(im, slot, vm)
        if self.compress is not None:
            im = self.compress(im)
        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
        if self.tokens:
            inner = getattr(self.enc, 'vit', self.enc)
            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
            f = torch.cat([f[:, :1], f[:, orig:]], 1)
        else:
            f = self.enc.forward_head(f, pre_logits=True)
            if f.dim() > 2:
                f = f.flatten(1)
        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
        if self.cond == 'post':
            f = f + ex(self.slot_emb(slot))
        if self.meta_mlp is not None and smeta.shape[1] > 0:
            mt = self.meta_mlp(smeta)
            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
        return self.readout(f, slot, sidx, B)
models=[]
_A5_LOAD_RECEIPT=[]
_a5_paths=[CKPT/f'm_f{i}.pt' for i in range(5)]
if not all(p.is_file() for p in _a5_paths):
    raise RuntimeError(f'A5 requires m_f0.pt..m_f4.pt; missing {[str(p) for p in _a5_paths if not p.is_file()]}')
_a5_config=None
for ckpt_path in _a5_paths:
    z=torch.load(ckpt_path,map_location='cpu',weights_only=False)
    cfg=z['cfg'];fold=int(z['fold'])
    if ckpt_path.name!=f'm_f{fold}.pt': raise RuntimeError('A5 fold/file identity mismatch')
    fields={'backbone':cfg['backbone'],'cond':cfg['cond'],'pool':cfg['pool'],'img':int(cfg['img']),'n_slice':int(cfg.get('n_slice',16)),'n_meta':int(cfg.get('n_meta',0)),'meta':cfg.get('meta','none'),'stem':cfg.get('stem','native'),'norm':cfg.get('norm','none')}
    if _a5_config is None:_a5_config=fields
    if fields!=_a5_config:raise RuntimeError(f'A5 per-fold input/model contract differs: {ckpt_path}')
    if fields['img']!=SIZE or fields['n_slice']!=N_SLICE or fields['n_meta']!=0:
        raise RuntimeError(f'A5 checkpoint input contract incompatible with loader: {fields}')
    if 'labels' in z and list(z['labels'])!=list(LABELS):raise RuntimeError('A5 label order differs')
    _stem=fields['stem'];_in=3 if _stem=='compress' else fields['n_slice']
    enc=timm.create_model(cfg['backbone'],pretrained=False,num_classes=0,in_chans=_in,**({'img_size':cfg['img']} if 'vit_' in cfg['backbone'] else {}))
    if cfg['cond']=='token':enc=ViTSlotToken(enc,N_SLOT_TYPES)
    m=Net(enc,cfg['cond'],cfg.get('n_meta',0),cfg['pool'],stem=_stem,n_slice=cfg.get('n_slice',16))
    rsna_strict_load(m,z['state_dict'],f'A5 fold {fold}')
    models.append(m.eval());_A5_LOAD_RECEIPT.append({'fold':fold,'file':str(ckpt_path),'sha256':rsna_sha(ckpt_path),'tensor_keys':len(z['state_dict']),'strict':True,'config':fields})
    print(f'A5 strict-loaded {ckpt_path.name}; all encoder and head tensors present')
CFG=dict(cfg)
rsna_json('/kaggle/working/diagnostics/a5_loads.json',_A5_LOAD_RECEIPT)
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_027, 27, 'a0f7ec9a6a26310a246fef4d3349a1af1e853e99cd2ecbc2b819d96aa9071a1e')


In [ ]:
GPU_STAGE_028 = r'''AMP_PREF = 'bf16'  # BTKD precision retained, including T4 execution; no unvalidated AMP change.

def amp_for(dev):
    if not str(dev).startswith('cuda'):
        return (torch.float32, False)
    cc = torch.cuda.get_device_capability(dev)
    if AMP_PREF == 'bf16':
        return (torch.bfloat16, True)
    if AMP_PREF == 'fp16':
        return (torch.float16, True)
    if AMP_PREF == 'fp32':
        return (torch.float32, False)
    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)
AMP_DT, AMP_ON = amp_for(DEV)
WORKERS = max(1, min(4, os.cpu_count() or 4))
CHUNK = 48
MICRO = 8
import copy as _speed_a5_copy
from concurrent.futures import ThreadPoolExecutor as _SpeedA5Pool
if torch.cuda.device_count() >= 2:
    _speed_a5_second = [_speed_a5_copy.deepcopy(m).to('cuda:1').eval() for m in models]
    _speed_a5_models = [[m.to('cuda:0').eval() for m in models], _speed_a5_second]
    del _speed_a5_second
    _speed_a5_pool = _SpeedA5Pool(max_workers=2)
else:
    _speed_a5_models = [[m.to('cuda:0').eval() for m in models]]
    _speed_a5_pool = _SpeedA5Pool(max_workers=1)
models = _speed_a5_models[0]
print(f"device {DEV} | amp {str(AMP_DT).split('.')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")

def _norm_(im):
    k = CFG.get('norm', 'none')
    if k == 'zscore':
        m = (im > 0).float()
        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
        return (im - mu) / (var.sqrt() + 1e-06) * m
    if k == 'imagenet':
        m = (im > 0).float()
        return (im - 0.485) / 0.229 * m
    return im

@torch.no_grad()
def _speed_a5_micro(images, masks, *, dev, model_group, amp_dtype):
    """Predict acquired rows only; NaN is a sentinel exclusively for absent inputs."""
    active = np.flatnonzero((masks > 0).any(axis=1))
    out = np.full((len(model_group), len(masks), len(LABELS)), np.nan, np.float32)
    if not len(active):
        return out
    ims, slots, sidx, vms = [], [], [], []
    for compact_index, original_index in enumerate(active):
        present = np.flatnonzero(masks[original_index] > 0)
        blk = images[original_index][present]
        ims.append(torch.from_numpy(blk))
        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))
        slots.append(torch.from_numpy(present + 1).long())
        sidx.append(torch.full((len(present),), compact_index, dtype=torch.long))
    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))
    sl = torch.cat(slots).to(dev)
    si = torch.cat(sidx).to(dev)
    vm = torch.cat(vms).to(dev)
    sm = torch.zeros(len(sl), CFG.get('n_meta', 0), device=dev)
    per = torch.zeros(len(model_group), len(active), len(LABELS), device=dev, dtype=torch.float32)
    def _a5_forward(enabled):
        with torch.autocast('cuda' if str(dev).startswith('cuda') else 'cpu',
                            dtype=amp_dtype, enabled=enabled):
            for fold_index, model in enumerate(model_group):
                per[fold_index] = torch.sigmoid(model(im, sl, sm, si, len(active), vm=vm).float())
        return per.cpu().numpy()
    got = _a5_forward(AMP_ON)
    if AMP_ON and not np.isfinite(got).all():
        rsna_event('a5_fp16_nonfinite_retry_fp32', rows=int(len(active)))
        got = _a5_forward(False)
    _bad = ~np.isfinite(got)
    if _bad.any():
        rsna_event('a5_nonfinite_neutral', rows=int(_bad.any(axis=(0, 2)).sum()))
        got = np.where(_bad, 0.5, got)
    rsna_finite(got, 'A5 acquired-row probabilities', probability=True)
    out[:, active] = got
    return out

"""Two GPU owners, preserving every original eight-study microbatch."""


def speed_a5_ranges(n, micro=8, devices=2):
    if n < 0 or micro < 1 or devices < 1:
        raise ValueError((n, micro, devices))
    return [[(start, min(start + micro, n))
             for i, start in enumerate(range(0, n, micro)) if i % devices == d]
            for d in range(devices)]


def speed_a5_predict(images, masks, models_by_device, executor, dtype=None):
    """No shard-local ranks; output remains [fold, original study, finding]."""
    count = len(masks)
    assert len(images) == count
    output = np.full((len(models_by_device[0]), count, len(LABELS)), np.nan, np.float32)
    plans = speed_a5_ranges(count, MICRO, len(models_by_device))

    def owner(device_index, ranges):
        device = torch.device(f"cuda:{device_index}")
        result = []
        with torch.cuda.device(device):
            for start, stop in ranges:
                value = _speed_a5_micro(images[start:stop], masks[start:stop],
                    dev=device, model_group=models_by_device[device_index],
                    amp_dtype=AMP_DT if dtype is None else dtype)
                result.append((start, stop, value))
        return result

    tasks = [executor.submit(owner, d, ranges) for d, ranges in enumerate(plans) if ranges]
    for future in tasks:
        for start, stop, value in future.result():
            output[:, start:stop] = value
    return output

def predict(images, masks):
    return speed_a5_predict(images, masks, _speed_a5_models, _speed_a5_pool)

def _a5_validate_applicability(raw, eligible):
    """Never confuse an absent input with failed inference on an acquired input."""
    x = np.asarray(raw)
    m = np.asarray(eligible, dtype=bool)
    if x.ndim != 3 or x.shape[1] != len(m):
        raise ValueError('A5 eligibility/raw shape mismatch')
    if m.any():
        rsna_finite(x[:, m], 'A5 acquired-row complete tensor', probability=True)
    if (~m).any() and not np.isnan(x[:, ~m]).all():
        raise RuntimeError('A5 absent rows must retain explicit NaN sentinels')
    return m

def _a5_rank_available(raw, eligible):
    eligible = _a5_validate_applicability(raw, eligible)
    result = np.full((len(eligible), raw.shape[-1]), .5, dtype=np.float64)
    if eligible.any():
        result[eligible] = 0.0
        for fold in raw:
            result[eligible] += rsna_rank01(fold[eligible])
        result[eligible] /= raw.shape[0]
    return result

def _a5_blend_available(parent, a5_values, eligible, weight):
    """Parent-only on metadata-proven absence; no silent fallback on model error."""
    base = np.asarray(parent, dtype=np.float64)
    a5 = np.asarray(a5_values, dtype=np.float64)
    mask = np.asarray(eligible, dtype=bool)
    if base.shape != a5.shape or mask.shape != (len(base),):
        raise ValueError('A5 merge shape mismatch')
    rsna_finite(base, 'A5 parent probabilities', probability=True)
    if not mask.any() or weight == 0:
        return base.copy()
    rsna_finite(a5[mask], 'A5 applicable ranks', probability=True)
    base_rank = pd.DataFrame(base).rank(method='average', pct=True).to_numpy()
    a5_rank = pd.DataFrame(a5[mask]).rank(method='average', pct=True).to_numpy()
    result = base_rank.copy()
    result[mask] = (1-weight)*base_rank[mask] + weight*a5_rank
    rsna_finite(result, 'A5 applicability-aware merge', probability=True)
    return result

# Macro ROC-AUC depends on ordering, so combine fold orderings rather
# than allowing a fold's probability scale to dominate the mean.
preds = np.full((len(models), len(studies), len(LABELS)), np.nan, np.float32)
_a5_eligible = np.zeros(len(studies), dtype=bool)
t0, done = (time.time(), 0)
def submit_study_block(executor, start):
    """Keep one bounded CPU decode block ahead of GPU inference."""
    block = studies[start:start + CHUNK]
    futures = [
        executor.submit(
            build_study,
            (index, study, by.get(study, [])),
        )
        for index, study in enumerate(block)
    ]
    return start, block, futures

# CPU decode threads avoid forking a process after CUDA/model initialization.
with A5DecodePool(max_workers=WORKERS) as ex:
    pending_block = submit_study_block(ex, 0)
    while pending_block is not None:
        rsna_deadline('A5 full-cohort inference')
        c0, block, futs = pending_block
        next_start = c0 + len(block)
        pending_block = (
            submit_study_block(ex, next_start)
            if next_start < len(studies)
            else None
        )
        imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
        msks = np.zeros((len(block), N_SLOT), np.uint8)
        _fut_index = {f: i for i, f in enumerate(futs)}
        for f in as_completed(futs):
            try:
                i, a, k = f.result()
                imgs[i], msks[i] = (a, k)
            except Exception as e:
                i = _fut_index[f]
                rsna_event('a5_study_failed', study=str(block[i]), error=f'{type(e).__name__}: {str(e)[:500]}')
                print(f'[a5] {block[i]}: build failed, treated as A5-absent (flagged): {type(e).__name__}: {str(e)[:200]}', flush=True)
                imgs[i] = 0; msks[i] = 0
        _a5_eligible[c0:c0 + len(block)] = (msks > 0).any(axis=1)
        preds[:, c0:c0 + len(block)] = predict(imgs, msks)
        _a5_validate_applicability(preds[:, c0:c0 + len(block)], _a5_eligible[c0:c0 + len(block)])
        done += len(block)
        el = time.time() - t0
        print(f'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m', flush=True)
        del imgs, msks
        gc.collect()
print(f'\ninference done in {(time.time() - t0) / 60:.1f} min')
np.savez_compressed('/kaggle/working/speed_a5_raw.npz',
    study_uids=np.asarray(studies), raw_probabilities=preds, applicable=_a5_eligible)
_speed_a5_pool.shutdown(wait=True)
del _speed_a5_models, models, m, enc, z
gc.collect()
for _speed_device in range(torch.cuda.device_count()):
    with torch.cuda.device(_speed_device):
        torch.cuda.empty_cache()
A5_W = 0.52
A5_LABELS = list(LABELS)
_a5_ok = _a5_validate_applicability(preds, _a5_eligible)
_a5_rank_mean = _a5_rank_available(preds, _a5_eligible)
A5_PREDS = dict(zip(sub_df['StudyInstanceUID'].astype(str), _a5_rank_mean.astype(np.float32)))
A5_AVAILABLE = dict(zip(sub_df['StudyInstanceUID'].astype(str), _a5_eligible.tolist()))
rsna_save_predictions('a5_rank_mean', studies, _a5_rank_mean, LABELS)
rsna_json('/kaggle/working/diagnostics/a5_applicability.json', {
    'unavailable_uids':[u for u, ok in A5_AVAILABLE.items() if not ok],
    'available_count':int(_a5_eligible.sum()), 'model_failure_fallback_allowed':False,
    'rank_placeholder_for_unavailable':0.5, 'placeholder_used_in_blend':False})
rsna_phase('a5', 'COMPLETE', applicable=int(_a5_eligible.sum()), absent=int((~_a5_eligible).sum()))
for _a5k, _a5v in _A5_SAVED.items():
    globals()[_a5k] = _a5v
del _A5_SAVED, _a5k, _a5v
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_028, 28, 'f9c7c32140b051df0fab9511bffe97abd7ed806770cc320ee1ad2fd5def40e7f')


In [ ]:
GPU_STAGE_029 = r'''_a5_sub = pd.read_csv('/kaggle/working/_pipeline_stage.csv', dtype={'StudyInstanceUID':str})
assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'
if A5_W > 0:
    _a5_uid_order = _a5_sub.StudyInstanceUID.astype(str).tolist()
    _a5_ours = np.stack([A5_PREDS[u] for u in _a5_uid_order])
    _a5_presence = np.asarray([A5_AVAILABLE[u] for u in _a5_uid_order], dtype=bool)
    _a5_sub[A5_LABELS] = _a5_blend_available(
        _a5_sub[A5_LABELS].to_numpy(), _a5_ours, _a5_presence, A5_W)
    assert np.isfinite(_a5_sub[A5_LABELS].to_numpy()).all()
    _a5_sub.to_csv('/kaggle/working/_pipeline_stage.csv', index=False)
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_029, 29, '5c31acb1019dd9de97103d9a68cb0fc0db6aa1032fb00af5b483b3a67ed1b08b')


### Rad encoder, heads and inherited calibration


In [ ]:
GPU_STAGE_031 = r'''from __future__ import annotations
import base64 as _rad_b64
import zlib as _rad_zlib
# Inherited fitted parameters; no split/training provenance is supplied.
# Original decompressed SHA256: 8b616cd8462ef0808702000c4e9d9e29d58be1a9939b5787c246dc7bff26eab2
_RAD_CALIBRATOR = {'coef': [[0.054988985480198795,
           0.0,
           -0.0,
           0.0,
           -0.006905034746797682,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.007598718307712681,
           0.004775188808852552,
           -0.0,
           0.0,
           0.0,
           -0.004984520904425084,
           0.0,
           -0.006541710133337969,
           -0.0,
           -0.003752718730904078,
           0.0,
           -0.0,
           -0.0,
           0.0028342800789145934,
           -0.0,
           0.012842247615159527,
           0.0,
           -0.0,
           0.004720465029524873,
           -0.0,
           -0.0,
           -0.003820481837620495,
           0.021610275070444572,
           -0.0,
           -0.009703704271490837,
           0.0,
           -0.0007892533771826419,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.002190521845853738,
           -0.0,
           0.0,
           -0.0,
           0.0018799414367212483,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.008708677037306389,
           0.0,
           0.0,
           -0.0006364643665921704,
           -0.0,
           0.0,
           -0.0007433491597744793,
           -0.0,
           -0.0,
           -0.0022552694692117214,
           0.08996657756223722,
           0.0,
           -0.0,
           0.0027210612157672017,
           -0.0,
           0.0,
           -0.009935986032994056,
           0.0014937801726411662,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0041838897324036756,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0],
          [-0.0,
           0.05407410902522933,
           -0.0,
           -0.0,
           -0.00955325444490328,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.002741621763298323,
           0.0,
           0.0030651354435245284,
           0.0,
           0.005261907711238927,
           -0.0,
           -0.0,
           -2.246328164835325e-06,
           -0.0,
           -0.007235628651266947,
           0.0,
           -0.0,
           -0.0,
           0.002244318533778753,
           0.0009848822815680745,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0017347234462540013,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.001643286147786208,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.032048814954222495,
           -0.0,
           -0.0,
           -0.002201853534088179,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0],
          [-0.0,
           -0.0,
           0.0591118991425821,
           -0.006355044471722653,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.007699542581442222,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.048274735575991476,
           -0.0,
           -0.0,
           -0.008539297840185648,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.004472020944327244,
           0.0,
           -0.0,
           -0.00347437691921257,
           0.05861944258061976,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.006976785974184935,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.004010865327087766,
           0.0,
           0.0,
           0.0,
           -0.0005908157372598818,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.00500875228252724,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.12402375784783327,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0015649029295778055,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0027029555164037418,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0],
          [0.0,
           -0.0,
           -0.0,
           0.020022960936525995,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0017738172621121892,
           -0.0,
           -0.0023183591962150705,
           0.0,
           -0.007515949427429685,
           -0.010155250767449142,
           0.0004984806412923339,
           -0.0,
           0.0,
           -0.005354229345626853,
           0.0,
           0.0,
           -0.0007398489398975964,
           -0.0,
           -0.0,
           0.0,
           -0.004071983093585694,
           -0.0,
           0.01700346063545301,
           -0.0,
           0.0058490781650871355,
           -0.001005834776031484,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0009063914656936253,
           0.000214420591619211,
           0.0005584969828176026,
           -0.0,
           -0.0020443851541547286,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.007754847683752665,
           -0.0005033977078919417,
           0.000708682810146001,
           -0.0,
           0.0,
           -0.003502795873998033,
           0.0,
           0.0,
           -0.0028597700818967357,
           -0.0,
           0.11847279874112572,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0015607762716186155,
           -0.0,
           -0.005456887523612534,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0],
          [0.0,
           -0.0,
           0.011905299490447545,
           -0.0,
           0.051327525539440685,
           0.0,
           0.006490303077739882,
           0.0,
           0.009189655557113569,
           -0.0032555342419950226,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.015598360106187889,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0007355085953963836,
           0.0,
           0.0,
           0.0,
           -0.005692959829707796,
           0.008383567458721805,
           -0.0004342984045223054,
           0.040683776777621715,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.003946854023482639,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.003675523046569264,
           -0.0,
           -0.0,
           -0.006683593070962812,
           0.001598200174796656,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0009560338374132588,
           0.0,
           0.0014865501499527398,
           0.0,
           0.0,
           -0.006465356514571239,
           0.0,
           -0.0,
           0.08801358135090863,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0032578169698303334,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0022856634143569687,
           -0.0008211041163776929,
           0.0,
           0.0,
           0.001979434428479871,
           -0.0006904492760499002],
          [0.0,
           -0.0010201300673726335,
           -0.0,
           0.003330913654546868,
           0.0,
           0.03804589769802522,
           0.0036582135089306815,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0074698905145500535,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.004181180657534594,
           -0.0,
           0.0,
           -0.0014097935467749104,
           0.0,
           -0.008364014702270928,
           -0.0,
           0.0,
           0.0,
           0.031234909820757367,
           0.0,
           0.0,
           0.001761560363567976,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           -0.00010302276929695168,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0036722429881927794,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           -0.028070173174009682,
           -0.0,
           0.03932353957184419,
           0.0,
           0.052709270489857116,
           4.30670771118039e-05,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.005897795403501839,
           -0.0,
           -0.0017091642652794067,
           0.0,
           -0.0,
           -0.0,
           -0.0011445129464464573,
           0.0,
           -3.599923112263568e-05],
          [-0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.00398886886838279,
           0.024531745408578224,
           0.0,
           0.003472383623711957,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0024216642234806546,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.04441771278056577,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.04208386438405934,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -8.871585477522171e-05,
           -0.0,
           -0.0,
           -0.0,
           -0.01794873111002335,
           -0.010965712486455531,
           0.0,
           -0.0,
           -0.0003112348219671695,
           -0.0,
           0.00033734018896491223,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.002767382390020047,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.006553844259659735,
           -0.0,
           0.002859144951920367,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.1080204096279449,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.008286727880437668,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0016705366518583718,
           0.0,
           0.003966228970353484,
           0.0,
           -0.0,
           -0.0036942492702868837,
           0.0,
           2.0158865558700288e-05,
           -0.0,
           -0.003290955791230868,
           0.0,
           2.931550215656576e-05],
          [-0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.15721758161271948,
           0.0,
           -0.0009277854332745394,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0035078908538144857,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0029922774103666346,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0015069965777220378,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0034575286435870912,
           0.0,
           0.0,
           0.0,
           0.0,
           0.004879501088574802,
           0.0,
           -0.0,
           0.0,
           0.00107584083385987,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.07386105032184348,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.0006098366519528874,
           0.000710602647614561,
           0.0,
           0.00029648470600281623,
           0.0,
           0.0006669555200166979,
           0.0,
           0.0001247447196589612,
           0.0],
          [0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.010971916263113718,
           0.0,
           0.0005950879974343855,
           0.020934939880106,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.00716089007036752,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.012582689029033865,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.05233942478784284,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.003853509415892567,
           -0.009802578580289266,
           0.0,
           0.0,
           0.013526997046656204,
           0.0064884417795260645,
           -0.0,
           0.0002747792719802145,
           0.00794104167516065,
           0.0032425867028947433,
           -0.0],
          [0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.07766030485486214,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0016984387848328866,
           -0.02318816122584484,
           -0.0,
           0.0008232018620112374,
           -0.0,
           0.0,
           -0.0,
           0.0030093149725252876,
           0.0,
           0.015496588831608236,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.003985489123804257,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.08934738817249994,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0031358753239660443,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.002757717666564488,
           0.0,
           0.0,
           0.0,
           -0.001572020817838309],
          [0.0,
           0.0,
           -0.007645376480798367,
           -0.0,
           -0.013152846307009906,
           -0.0,
           -0.013405770463176186,
           0.0,
           -0.0,
           -0.0126185896490272,
           0.08456110280686102,
           0.0010498490952287943,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.026761069960320228,
           0.004881051675806471,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0029949120807847875,
           0.0,
           0.0,
           -0.0013947178093899313,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.00942848401064722,
           -0.0,
           -0.0,
           -0.00022519045066539146,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0457033966423892,
           0.027947704687921546,
           0.0,
           -0.0017566658004398543,
           -0.011670049510825595,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0007825317116991779,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0],
          [0.0,
           0.0,
           -0.0,
           -0.0,
           -0.001502844459484492,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0020872198960683284,
           0.009847417955540042,
           0.004860013798480458,
           0.0,
           -0.0,
           -0.0012632551037073973,
           0.0,
           -0.007111539097901386,
           -0.0,
           -0.00195199372867498,
           0.0,
           -0.0,
           -0.0,
           0.003484155321535836,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0005385599274495165,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.002005337879973018,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           -0.0,
           0.0,
           0.04784159508857366,
           0.0,
           -0.0,
           -0.0,
           -0.0,
           -0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0,
           0.0,
           0.0,
           -0.0,
           0.0]],
 'gate': ['ACL', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', "Baker's", 'Contusion'],
 'groups': [['ACL', 'MCL', 'Contusion', 'Fracture'],
            ['Medial Meniscus', 'Lateral Meniscus'],
            ['Medial OA', 'Lateral OA', 'PF OA'],
            ['Effusion', 'Synovitis', "Baker's"]],
 'intercept': [0.27594726240322665,
               0.19664147502522833,
               0.46103077516667196,
               0.26000148037766385,
               0.348880476077348,
               0.267855131253255,
               0.4114411126395088,
               0.4677221586179633,
               0.2775079245121116,
               0.3059966406870837,
               0.26281611268502275,
               0.1933670474221893],
 'mean': [0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          -8.169035821569329e-19,
          4.0845179107846643e-19,
          8.169035821569329e-19,
          0.0,
          -2.0422589553923322e-19,
          -7.147906343873162e-19,
          1.0211294776961661e-19,
          2.0422589553923322e-19,
          -6.1267768661769965e-19,
          -8.169035821569329e-19,
          3.0633884330884983e-19,
          -1.2253553732353993e-18,
          -4.0845179107846643e-19,
          3.0633884330884983e-19,
          6.1267768661769965e-19,
          -1.021129477696166e-18,
          -2.0422589553923322e-19,
          -9.190165299265494e-19,
          2.0422589553923322e-19,
          2.0422589553923322e-19,
          -2.0422589553923322e-19,
          -1.2253553732353993e-18,
          -1.0211294776961661e-19,
          0.0,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          0.5001149689583813,
          5.526557829386066,
          2.2368360542653485,
          1.9521729133134054,
          1.337548861807312,
          3.1772821338238675,
          1.0584042308576684,
          1.0482869625201197,
          1.0705909404460796,
          3.1772821338238675,
          1.0584042308576684,
          1.0482869625201197,
          1.0705909404460796],
 'protocol_columns': ['n_series',
                      'n_Sag',
                      'n_Cor',
                      'n_Axi',
                      'Fat',
                      'Fat_Sag',
                      'Fat_Cor',
                      'Fat_Axi',
                      'Flu',
                      'Flu_Sag',
                      'Flu_Cor',
                      'Flu_Axi'],
 'scale': [0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512695294856,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512695294856,
           0.28867512338382056,
           0.28867512338382056,
           0.28867512338382056,
           0.28867512338382056,
           0.28867512338382056,
           0.28867512338382056,
           0.288675123394349,
           0.288675123394349,
           0.288675123394349,
           0.288675123394349,
           0.28867512336276374,
           0.28867512338382056,
           0.28867512696347697,
           0.28867512695294856,
           0.28867512695294856,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512696347697,
           0.28867512695294856,
           0.2886751269529485,
           0.28867512695294856,
           0.28867512696347697,
           0.28867512694242015,
           0.23019299127577333,
           0.20945811170519504,
           0.18896664844677993,
           0.22034647621868486,
           0.15727493499021805,
           0.19246986757904713,
           0.18959582474964737,
           0.1500338770229888,
           0.17335524281933992,
           0.21499556517074797,
           0.18526691377924037,
           0.24722042176621192,
           0.23300663477492883,
           0.2351154922532429,
           0.19851192071060803,
           0.22614032204035772,
           0.16824789069383667,
           0.19316845808174274,
           0.19078421415473018,
           0.17455770295025105,
           0.1939184152151192,
           0.2380093052256761,
           0.2054846773727961,
           0.25238186441777205,
           0.2560260929356755,
           0.2563420389842119,
           0.2669423487470595,
           0.2578640024055912,
           0.2718501229846065,
           0.26593176523552875,
           0.26652763985208816,
           0.2716787563858938,
           0.26822049981618185,
           0.2551672579287264,
           0.26457231958493127,
           0.24970143393957778,
           0.21746552440992822,
           0.24075616342318434,
           0.2586368483721014,
           0.23201516104802145,
           1.392601296101108,
           0.5943047831182455,
           0.7471955639897663,
           0.6579157483703758,
           0.6336751078357071,
           0.41835483650318195,
           0.35622110896336706,
           0.26841408540232553,
           0.6336751078357071,
           0.41835483650318195,
           0.35622110896336706,
           0.26841408540232553]}
# BTKD Rad graph: preserve its E13/E11 calibrated feature contract; remove unused twin work.
#
# The pinned reference Rad family is fused with correct-contract E13, then
# the same E13 heads run on the E11 layout at 0.15. No twin/legacy wrapper
# follows it, matching the branch that produced V48's visible submission.

import contextlib as _rad_contextlib
import gc as _rad_gc
import hashlib as _rad_hashlib
import json as _rad_json
import os as _rad_os
import re as _rad_re
import time as _rad_time
from concurrent.futures import ThreadPoolExecutor as _RadThreadPool
from pathlib import Path as _RadPath

import numpy as _rad_np
import pandas as _rad_pd
import pydicom as _rad_pydicom
import torch as _rad_torch
import torch.nn as _rad_nn
import torch.nn.functional as _rad_F
from torchvision.models import resnet50 as _rad_resnet50

_RAD_LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
_RAD_ALPHA = 0.55
_RAD_EXCLUDE = ("Baker's", 'Fracture')
_RAD_HEADS_SHA256 = '54f657826b3458a7ba3d462e198ba380732f2b136246182312704929874a9a2c'
_RAD_REFERENCE_HEADS_SHA256 = '0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa'
_RAD_ENCODER_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'
_RAD_E13_HEADS_SHA256 = 'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a'
_RAD_E13_MEMBER_WEIGHT = 0.50
_RAD_V48_SECOND_ALPHA = 0.20
_RAD_TWIN_ALT_WEIGHT = 0.500001
_RAD_TOKEN_DIM, _RAD_HEAD_DIM = 2048, 512

_RAD_E11_SLOTS = [
    ('SAG_NOFS', 'Sagittal', None, False),
    ('COR_NOFS', 'Coronal', None, False),
    ('AX_NOFS', 'Axial', None, False),
    ('SAG_FS', 'Sagittal', None, True),
]
_RAD_E11_CROP_MM = 130.0
_RAD_E11_CACHE_SLICES = 8
_RAD_E11_IMG = 224

_RAD_E13_SLOTS = [
    ('SAG_FS', 'Sagittal', None, True),
    ('COR_FS', 'Coronal', None, True),
    ('AX_FS', 'Axial', None, True),
    ('SAG_NOFS', 'Sagittal', None, False),
]
_RAD_E13_CROP_MM = 130.0
_RAD_E13_CACHE_SLICES = 8
_RAD_E13_IMG = 224

# Our independently trained five-fold family.  Its preprocessing and estimator
# are preserved from V35: native DICOM geometry/fat-sat handling and a mean of
# per-fold percentile ranks (rather than v15's rank of the probability mean).
_OUR_N_SLOT, _OUR_N_SLICE, _OUR_IMG = 3, 8, 224

# Exact V40/E10 test representation: three fat-suppressed planes, eight
# acquired slices per plane, full frame, legacy ordering/laterality/fill.
SLOTS = [
    ('SAG_FS', 'Sagittal', None, True),
    ('COR_FS', 'Coronal', None, True),
    ('AX_FS', 'Axial', None, True),
]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8
IMG = CACHE_IMG = 224
CROP_MM = 10_000.0
SLICE_BAND = (0.2, 0.8)
RULES = dict(RULES_LEGACY)
TIME_BUDGET = 8.0 * 3600


def _rad_log(message):
    print(f'[Rad-dual5] {message}', flush=True)


def _rad_sha256(path, chunk=8 << 20):
    digest = _rad_hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()


def _rad_find_file(name, expected_sha=None, explicit_env=None):
    if explicit_env and _rad_os.environ.get(explicit_env):
        candidates = [_RadPath(_rad_os.environ[explicit_env])]
    else:
        by_name = {
            'ResNet50.pt': [
                '/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt',
                '/kaggle/input/resnet-50-radimagenet-marwan/ResNet50.pt',
            ],
            'v52_radimagenet_heads.pt': [
                '/kaggle/input/datasets/prvsiyan/rsna-knee-v52-radimagenet-heads-20260812/v52_radimagenet_heads.pt',
                '/kaggle/input/rsna-knee-v52-radimagenet-heads-20260812/v52_radimagenet_heads.pt',
                '/kaggle/input/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt',
                '/kaggle/input/rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt',
            ],
            'v52_e11_heads.pt': [
                '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e11-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/rsna-knee-e11-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20/v52_e11_heads.pt',
                '/kaggle/input/rsna-knee-e11-diverse-heads-v20/v52_e11_heads.pt',
            ],
        }
        if name not in by_name:
            raise FileNotFoundError(f'unpinned Rad artifact name: {name}')
        candidates = [_RadPath(path) for path in by_name[name]]
    existing = [path for path in candidates if path.is_file()]
    valid = [
        path for path in existing
        if expected_sha is None or _rad_sha256(path) == expected_sha
    ]
    if len(valid) != 1:
        raise RuntimeError(
            f'expected one verified Rad artifact {name}, found {valid}; existing={existing}'
        )
    return valid[0]


class _RadEncoder(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = _rad_nn.Sequential(
            *list(_rad_resnet50(weights=None).children())[:-2]
        )

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))


class _RadHead(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.project = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_TOKEN_DIM),
            _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM),
            _rad_nn.GELU(),
        )
        self.plane = _rad_nn.Parameter(_rad_torch.randn(N_SLOT, _RAD_HEAD_DIM) * .01)
        self.position = _rad_nn.Parameter(_rad_torch.randn(CACHE_SLICES, _RAD_HEAD_DIM) * .01)
        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02)
        self.attn = _rad_nn.MultiheadAttention(
            _RAD_HEAD_DIM, 8, dropout=.10, batch_first=True
        )
        self.fuse = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_HEAD_DIM * 4),
            _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM),
            _rad_nn.GELU(),
            _rad_nn.Dropout(.15),
        )
        self.weight = _rad_nn.Parameter(
            _rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02
        )
        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), self.plane.shape[0], self.position.shape[0], self.plane.shape[-1])
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(
            query, token, token, key_padding_mask=key_padding, need_weights=False
        )[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(_RAD_LABELS), -1)
        fused = self.fuse(_rad_torch.cat(
            [attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1
        ))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias


def _rad_load_public_heads(device, expected_sha):
    heads_path = _rad_find_file('v52_radimagenet_heads.pt', expected_sha)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=True)
    expected = {
        'version': 'v52-radimagenet-resnet50-official-1',
        'targets': _RAD_LABELS,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'encoder_source_commit': '0ce16f7375db4236e646829d1eca61cdb4282133',
        'img': 224,
        'slices_per_plane': 8,
        'feature': 'global_average_pool',
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'public-v15 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('public-v15 bundle requires exactly five heads')
    if sorted(int(record.get('fold', -1)) for record in folds) != list(range(5)):
        raise RuntimeError('public-v15 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return heads, str(heads_path)


def _rad_load_e13_heads(device):
    # V48 used an unqualified filename shared by E11 and E13. Resolve the
    # intended E13 bundle by content and validate its complete pixel contract.
    heads_path = _rad_find_file('v52_e11_heads.pt', _RAD_E13_HEADS_SHA256)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=False)
    expected = {
        'version': 'e11-radimagenet-resnet50-diverse-1',
        'targets': _RAD_LABELS,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'slots': [list(slot) for slot in _RAD_E13_SLOTS],
        'crop_mm': _RAD_E13_CROP_MM,
        'img': _RAD_E13_IMG,
        'slices_per_plane': _RAD_E13_CACHE_SLICES,
        'feature': 'global_average_pool',
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'E13 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('E13 bundle requires exactly five heads')
    if sorted(int(record.get('fold', -1)) for record in folds) != list(range(5)):
        raise RuntimeError('E13 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return heads, str(heads_path)


def _rad_load_models(device):
    """Two persistent encoder replicas; unused alternative heads are not loaded."""
    import copy
    path = _rad_find_file('ResNet50.pt', _RAD_ENCODER_SHA256, explicit_env='RSNA_RAD_WEIGHT_PATH')
    encoder = _RadEncoder().eval()
    rsna_strict_load(encoder, _rad_torch.load(path,map_location='cpu',weights_only=True),'Rad encoder')
    if sum(p.numel() for p in encoder.parameters())!=23508032:
        raise RuntimeError('Rad encoder parameter count changed')
    replicas=[encoder.to('cuda:0'), copy.deepcopy(encoder).to('cuda:1')] if torch.cuda.device_count() >= 2 else [encoder.to('cuda:0')]
    for m in replicas:
        m.requires_grad_(False)
    heads,hpath=_rad_load_public_heads(device,_RAD_REFERENCE_HEADS_SHA256)
    return replicas,heads,str(path),hpath


@_rad_torch.inference_mode()
def _rad_encode(encoder, pixels, slot_mask, device):
    """Preserve parent's <=96 images/device chunks without DataParallel replication."""
    n,slots,slices,height,width=pixels.shape
    token_mask=_rad_np.repeat(slot_mask[:,:,None],slices,axis=2).reshape(n,-1)
    _rad_empty=int((token_mask.sum(1)==0).sum())
    if _rad_empty:
        print(f'[rad] {_rad_empty} study(ies) with no slot evidence encoded as all-zero rows (parent policy)', flush=True)
    valid=_rad_np.flatnonzero(token_mask.reshape(-1)>0)
    flat=pixels.reshape(-1,height,width)
    features=_rad_np.zeros((n,slots*slices,_RAD_TOKEN_DIM),_rad_np.float16)
    def run(indices,d):
        dev=_rad_torch.device(f'cuda:{d}')
        def _rad_forward(_amp):
            with _rad_torch.cuda.device(dev), _rad_torch.inference_mode(), _rad_torch.autocast('cuda',dtype=_rad_torch.float16,enabled=_amp):
                image=_rad_torch.from_numpy(flat[indices]).to(dev).float().div_(127.5).sub_(1.)
                image=image.unsqueeze(1).expand(-1,3,-1,-1).contiguous()
                return encoder[d](image).float().cpu().numpy()
        value=_rad_forward(True)
        half=value.astype(_rad_np.float16)
        if not (_rad_np.isfinite(value).all() and _rad_np.isfinite(half).all()):
            rsna_event('rad_fp16_nonfinite_retry_fp32',tokens=int(len(indices)))
            value=_rad_forward(False); half=_rad_np.clip(value,-65000,65000).astype(_rad_np.float16)
        if not _rad_np.isfinite(half).all():
            rsna_event('rad_nonfinite_zeroed',tokens=int((~_rad_np.isfinite(half).reshape(len(half),-1).all(axis=1)).sum()))
            half=_rad_np.where(_rad_np.isfinite(half),half,0).astype(_rad_np.float16)
        return indices,half
    _dev_count = len(encoder)
    with _RadThreadPool(max_workers=_dev_count) as pool:
        for start in range(0, len(valid), 96 * _dev_count):
            rsna_deadline('Rad full-cohort inference')
            block = valid[start:start + 96 * _dev_count]
            if _dev_count == 1:
                splits = (block,)
            else:
                cut = (len(block) + 1) // 2
                splits = (block[:cut], block[cut:])
            jobs = [pool.submit(run, ii, d) for d, ii in enumerate(splits) if len(ii)]
            for job in jobs:
                indices, value = job.result()
                features.reshape(-1, _RAD_TOKEN_DIM)[indices] = value
    return features,token_mask.astype(_rad_np.float32)


@_rad_torch.inference_mode()
def _rad_predict_head(head, features, masks, device, batch=64):
    predictions = []
    for start in range(0, len(features), batch):
        image = _rad_torch.from_numpy(features[start:start + batch]).to(device)
        mask = _rad_torch.from_numpy(masks[start:start + batch]).to(device)
        def _head_forward(_amp):
            amp = (_rad_torch.autocast('cuda', enabled=_amp)
                   if device.type == 'cuda' else _rad_contextlib.nullcontext())
            with amp:
                return _rad_torch.sigmoid(head(image if _amp else image.float(), mask)).float().cpu()
        block = _head_forward(True)
        if not bool(_rad_torch.isfinite(block).all()):
            rsna_event('rad_head_fp16_nonfinite_retry_fp32', rows=int(len(image)))
            block = _head_forward(False)
        if not bool(_rad_torch.isfinite(block).all()):
            rsna_event('rad_head_nonfinite_neutral', rows=int((~_rad_torch.isfinite(block)).any(dim=1).sum()))
            block = _rad_torch.where(_rad_torch.isfinite(block), block, _rad_torch.full_like(block, 0.5))
        predictions.append(block)
    return _rad_torch.cat(predictions).numpy()


def _rad_rank_columns(values):
    return rsna_rankpct(values)


def _rad_validate(frame, expected_ids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_RAD_LABELS]:
        raise RuntimeError('V36 submission schema drift')
    ids = frame['StudyInstanceUID'].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError('V36 submission study identity/order drift')
    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)
    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        rsna_event('rad_invalid_values_repaired', nonfinite=int((~_rad_np.isfinite(values)).sum()), out_of_range=int(((values < 0) | (values > 1)).sum()))
        frame[_RAD_LABELS] = _rad_np.clip(_rad_np.where(_rad_np.isfinite(values), values, 0.5), 0, 1)


class _RadLayoutPrefetch:
    """One CPU layout builder overlaps GPU work; GPU head shapes are model-owned."""
    def __init__(self, prepare, configurations, release):
        from concurrent.futures import ThreadPoolExecutor
        self.prepare = prepare
        self.configurations = list(configurations)
        self.release = release
        self.executor = ThreadPoolExecutor(max_workers=1)
        self.pending = None
        self.next_index = 0
        self.acquired = None

    def _submit(self):
        if self.next_index < len(self.configurations):
            config = self.configurations[self.next_index]
            self.next_index += 1
            self.pending = self.executor.submit(self.prepare, *config)

    def __enter__(self):
        self._submit()
        return self

    def take(self):
        if self.acquired is not None:
            raise RuntimeError('release the current Rad layout before taking another')
        if self.pending is None:
            raise StopIteration('all Rad layouts consumed')
        future, self.pending = self.pending, None
        try:
            self.acquired = future.result()
        except BaseException:
            self.executor.shutdown(wait=True, cancel_futures=True)
            raise
        self._submit()
        return self.acquired

    def release_current(self):
        if self.acquired is not None:
            self.release(self.acquired[1])  # (study IDs, pixels, masks)
            self.acquired = None

    def __exit__(self, exc_type, exc, tb):
        self.release_current()
        pending, self.pending = self.pending, None
        self.executor.shutdown(wait=True, cancel_futures=True)
        if pending is not None and not pending.cancelled():
            try:
                extra = pending.result()
                self.release(extra[1])
            except BaseException:
                if exc_type is None:
                    raise
        return False


def _rad_main():
    """BTKD Rad/calibrator graph unchanged; diagnostic-only twin work removed.

    BTKD's E13-on-E11 second input is intentional in its fitted calibrator.
    Keep that pair together; do not claim standalone E13 train/serve parity.
    """
    started=_rad_time.time();work=_RadPath('/kaggle/working');primary=work/'_pipeline_stage.csv'
    test=_rad_pd.read_csv(ROOT/'test.csv',dtype={'StudyInstanceUID':str})
    ids=test.StudyInstanceUID.tolist()
    baseline=rsna_frame(_rad_pd.read_csv(primary,dtype={'StudyInstanceUID':str}),ids,_RAD_LABELS,'Rad parent')
    device=_rad_torch.device('cuda:0')
    encoder,reference_heads,encoder_path,reference_heads_path=_rad_load_models(device)
    tser=_rad_pd.read_csv(ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
    plane=dict(zip(tser.SeriesInstanceUID,tser.Anatomical_Plane))
    # One annotation pass. All three layouts use the identical legacy annotation rules.
    headers=annotate(walk('test_series'))
    side_map=lat_of(headers,'rad common')
    def prepare_layout(layout,crop,tag):
        globals().update(SLOTS=list(layout),N_SLOT=len(layout),CACHE_SLICES=8,IMG=224,CACHE_IMG=224,CROP_MM=float(crop),RULES=dict(RULES_LEGACY))
        value=build_cache(pick_slots(headers,plane),plane,side_map,tag)
        if value[0]!=ids:
            rsna_release_pixels(value[1])
            raise RuntimeError('Rad cache order changed')
        return value
    def encode_layout(prefetch):
        studies,pixels,mask=prefetch.take()
        try:
            f,tm=_rad_encode(encoder,pixels,mask,device)
            return f,tm,int(tm.sum())
        finally:
            prefetch.release_current()
    def predict_heads(heads,f,mask,tag):
        if len(heads)!=5: raise RuntimeError(tag+': five heads required')
        probs=_rad_np.stack([_rad_predict_head(h,f,mask,device) for h in heads])
        rsna_finite(probs,tag,probability=True);rsna_save_predictions(tag,ids,probs,_RAD_LABELS)
        return probs.mean(0)
    first=[('SAG_FS','Sagittal',None,True),('COR_FS','Coronal',None,True),('AX_FS','Axial',None,True)]
    # E13 has four slots; construct its model before the producer mutates globals.
    globals().update(SLOTS=list(_RAD_E13_SLOTS),N_SLOT=4,CACHE_SLICES=8)
    e13_heads,e13_path=_rad_load_e13_heads(device)
    configurations=[(first,10000.,'rad_e10'),(_RAD_E13_SLOTS,130.,'rad_e13'),(_RAD_E11_SLOTS,130.,'rad_legacy_second')]
    with _RadLayoutPrefetch(prepare_layout,configurations,rsna_release_pixels) as prefetch:
        f,m,tokens0=encode_layout(prefetch)
        reference_rank=_rad_rank_columns(predict_heads(reference_heads,f,m,'rad_e10_raw'))
        del f,m,reference_heads
        f,m,tokens1=encode_layout(prefetch)
        e13_rank=_rad_rank_columns(predict_heads(e13_heads,f,m,'rad_e13_raw'))
        reference_rank=_rad_rank_columns((1-_RAD_E13_MEMBER_WEIGHT)*reference_rank+_RAD_E13_MEMBER_WEIGHT*e13_rank)
        del f,m,e13_rank
        baseline_rank=_rad_rank_columns(baseline[_RAD_LABELS].to_numpy())
        candidate=baseline.copy()
        for j,label in enumerate(_RAD_LABELS):
            if label not in _RAD_EXCLUDE:
                candidate[label]=(1-_RAD_ALPHA)*baseline_rank[:,j]+_RAD_ALPHA*reference_rank[:,j]
        _rad_validate(candidate,ids)
        f,m,tokens2=encode_layout(prefetch)
        second_rank=_rad_rank_columns(predict_heads(e13_heads,f,m,'rad_legacy_second_raw'))
        del f,m,e13_heads
    branch=candidate.copy()
    branch[_RAD_LABELS]=_rad_rank_columns((1-_RAD_V48_SECOND_ALPHA)*_rad_rank_columns(candidate[_RAD_LABELS].to_numpy())+_RAD_V48_SECOND_ALPHA*second_rank)
    rsna_save_predictions('transformer_no_calibrator', ids, branch[_RAD_LABELS].to_numpy(), _RAD_LABELS)
    cal=_RAD_CALIBRATOR
    protocol=_rad_pd.DataFrame(index=_rad_pd.Index(ids,name='StudyInstanceUID'))
    protocol['n_series']=tser.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    for pl in ('Sagittal','Coronal','Axial'):
        part=tser[tser.Anatomical_Plane.astype(str)==pl]
        protocol[f'n_{pl[:3]}']=part.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    for flag in ('Fat_Suppression','Fluid_Sensitive'):
        marked=tser[_rad_pd.to_numeric(tser[flag],errors='coerce').fillna(0)>0]
        protocol[flag[:3]]=marked.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
        for pl in ('Sagittal','Coronal','Axial'):
            part=marked[marked.Anatomical_Plane.astype(str)==pl]
            protocol[f'{flag[:3]}_{pl[:3]}']=part.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    if list(protocol.columns)!=list(cal['protocol_columns']): raise RuntimeError('calibrator protocol drift')
    mean=(baseline_rank+reference_rank+second_rank)/3.
    blocks=[baseline_rank,reference_rank,second_rank,reference_rank-baseline_rank,second_rank-baseline_rank,mean]
    for group in cal['groups']:
        blocks.append(mean[:,[_RAD_LABELS.index(t) for t in group]].mean(axis=1,keepdims=True))
    blocks.append(protocol.to_numpy(_rad_np.float64))
    x=_rad_np.concatenate(blocks,axis=1); center=_rad_np.asarray(cal['mean']);spread=_rad_np.asarray(cal['scale']);coef=_rad_np.asarray(cal['coef']);bias=_rad_np.asarray(cal['intercept'])
    if x.shape[1]!=coef.shape[1] or not bool((spread>0).all()): raise RuntimeError('calibrator shape/scale drift')
    adjusted=_rad_rank_columns(((x-center)/spread)@coef.T+bias)
    values=branch[_RAD_LABELS].to_numpy(_rad_np.float64).copy()
    for j,label in enumerate(_RAD_LABELS):
        if label in set(cal['gate']): values[:,j]=.6*values[:,j]+.4*adjusted[:,j]
    final=branch.copy();final[_RAD_LABELS]=_rad_rank_columns(values)
    # Public fracture-only ablation: add the existing Rad main contribution
    # for Fracture. Its author reported silver-OOF gain but gold58 regression.
    # No model rerun, no default change, and no effect on other labels.
    fj = _RAD_LABELS.index('Fracture')
    fracture_candidate = candidate.copy()
    fracture_candidate['Fracture'] = (1-_RAD_ALPHA)*baseline_rank[:,fj] + _RAD_ALPHA*reference_rank[:,fj]
    fracture_branch = _rad_rank_columns((1-_RAD_V48_SECOND_ALPHA)*_rad_rank_columns(fracture_candidate[_RAD_LABELS].to_numpy()) + _RAD_V48_SECOND_ALPHA*second_rank)
    fracture_final = final.copy()
    fracture_final['Fracture'] = fracture_branch[:,fj]
    rsna_save_predictions('transformer_rad_fracture', ids, fracture_final[_RAD_LABELS].to_numpy(), _RAD_LABELS)
    _rad_validate(final,ids);rsna_save_predictions('btkd_parent_after_rad',ids,final[_RAD_LABELS].to_numpy(),_RAD_LABELS)
    tmp=primary.with_suffix('.tmp');final.to_csv(tmp,index=False);_rad_os.replace(tmp,primary)
    globals()['V18_CALIBRATOR_APPLIED']=True
    rsna_json(work/'v558_rad_receipt.json',{'status':'COMPLETE','graph':'BTKD legacy Rad plus its matched calibrator preserved','reference_heads_sha256':_RAD_REFERENCE_HEADS_SHA256,'e13_sha256':_RAD_E13_HEADS_SHA256,'encoder_sha256':_RAD_ENCODER_SHA256,'legacy_cross_layout_pass_preserved':True,'standalone_cross_layout_train_parity_claimed':False,'unused_alternate_family_removed':True,'encoder_replicas':2,'layout_cpu_prefetch':True,'model_owned_head_shape':True,'tokens':[tokens0,tokens1,tokens2],'seconds':_rad_time.time()-started})
    del encoder;_rad_gc.collect()
    for d in range(2):
        with _rad_torch.cuda.device(d): _rad_torch.cuda.empty_cache()
    _rad_log('BTKD reference Rad/calibrator complete; unused alternate-head branch removed')


rsna_phase('radimagenet', 'START')
_rad_main()
rsna_phase('radimagenet', 'COMPLETE')
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_031, 31, 'cebe98aefa3d4bd90770a2711f93b48012e1fb56ea401e9b1ba3e58863097bf6')


### Raptor, residual CoAt and D4


In [ ]:
GPU_STAGE_033 = '''if globals().get('_DINOV2_MATCHED_MEMBERS') != 20:
    raise RuntimeError('DINOv2 20/20 fingerprint gate did not pass')
import gc as _ke_gc
import os as _ke_os
import time as _ke_time
from concurrent.futures import ThreadPoolExecutor as _KeThreadPool
import numpy as _ke_np
import pandas as _ke_pd
from pathlib import Path as _KePath

_ke_primary = _KePath('/kaggle/working/_pipeline_stage.csv')
_ke_ours = _ke_pd.read_csv(_ke_primary, dtype={'StudyInstanceUID': str})
_KE_LAB = [c for c in _ke_ours.columns if c != 'StudyInstanceUID']

_KE_SRC = r\'\'\'
import os, glob, time, gc, hashlib
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
torch.backends.cudnn.benchmark = False  # ragged 94-window chunks: no repeated autotuning
torch.backends.cuda.matmul.allow_tf32 = True
IMG = 336
CROP_MM = 140.0
SPAN_LO, SPAN_HI = 0.02, 0.98
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14),
         ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(slot[2] for slot in SLOTS)
K_EVAL = 62
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
       "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
       "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
_SLOTS64 = [("Sagittal", 1, 18), ("Sagittal", 0, 14),
            ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12)]
_SLOTS44 = [("Sagittal", 1, 12), ("Sagittal", 0, 10),
            ("Coronal", 1, 8), ("Coronal", 0, 6), ("Axial", -1, 8)]
ARMS = [
    {"name": "maxspan-v5", "file": "raptor_ft_coatnet_v5_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": False, "w": 0.60},
    {"name": "native384dense-v10", "file": "raptor_ft_coatnet_v10_full.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 384, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": False, "w": 0.10},
    {"name": "maxspan-v5-reverse", "file": "raptor_ft_coatnet_v5_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": True, "w": 0.10},
    {"name": "native384-v8", "file": "raptor_ft_coatnet_v8_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 384, "slots": _SLOTS44, "span": (0.06, 0.94), "k_eval": 42,
     "reverse": False, "w": 0.20},
]

def build_backbone(arch, pretrained=False):
    hybrid = arch.startswith(('maxvit', 'maxxvit', 'coatnet', 'coat_', 'convnext'))
    is_vit = not hybrid and any((k in arch for k in ('vit', 'deit', 'dinov2', 'eva', 'beit')))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool='token', dynamic_img_size=True)
    else:
        kw.update(global_pool='avg')
    return timm.create_model(arch, **kw)

class RaptorClassifier(nn.Module):

    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))

def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location='cpu', weights_only=False)
    arch = ck.get('arch', arch_default)
    if arch != arch_default:
        raise RuntimeError(f'Raptor architecture drift: {arch} != {arch_default}')
    ck_res = int(ck.get('res', res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    rsna_strict_load(model,ck['model'],str(pt_path))
    rsna_event('raptor_checkpoint',file=str(pt_path),sha256=rsna_sha(pt_path),arch=arch,res=ck_res)
    model.eval().to(device)
    del ck
    gc.collect()
    return (model, ck_res)



def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = (int(valid.min()), int(valid.max()))
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]

def eval_windows(vol, mask, k, res, norm=NORM):
    """Resize each selected source plane once, then gather the unchanged RGB triplets."""
    volume = np.asarray(vol)
    depth = int(volume.shape[0])
    centers = np.asarray(_eval_centers(mask, depth, k), dtype=np.int64)
    centers = np.clip(centers, 1, depth - 2)
    if tuple(volume.shape[-2:]) == (res, res):
        # Native384 needs no interpolation; the existing small per-triplet copy
        # avoids a slower large advanced-index gather and preserves its exact math.
        wins = np.empty((len(centers), 3, res, res), np.float32)
        for j, center in enumerate(centers):
            wins[j] = np.stack([volume[center - 1], volume[center],
                                volume[center + 1]], axis=0).astype(np.float32) / 255.0
        x = torch.from_numpy(wins)
        if norm == 'imagenet':
            x = (x - _MEAN) / _STD
        return x
    triplets = centers[:, None] + np.asarray([-1, 0, 1], dtype=np.int64)
    # Use only planes referenced by the original eval-center recipe.
    unique, inverse = np.unique(triplets.reshape(-1), return_inverse=True)
    source = volume[unique].astype(np.float32) / 255.0
    planes = torch.from_numpy(source)
    if tuple(planes.shape[-2:]) != (res, res):
        resized = torch.empty((len(planes), res, res), dtype=torch.float32)
        for start in range(0, len(planes), 16):
            resized[start:start+16] = F.interpolate(
                planes[start:start+16, None], size=(res, res),
                mode='bilinear', align_corners=False)[:, 0]
        planes = resized
    x = planes[torch.from_numpy(inverse.reshape(-1, 3))]
    if norm == 'imagenet':
        x = (x - _MEAN) / _STD
    return x





def rankpct(x):
    return rsna_rank01(x)

def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + '/*.dcm')
        recs = []
        ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and (len(iop) == 6):
                    r = np.array(iop[:3], float)
                    c = np.array(iop[3:], float)
                    n = np.cross(r, c)
                    pos = float(np.dot(np.array(ipp, float), n))
                else:
                    instance = getattr(h, 'InstanceNumber', None)
                    if instance is None:
                        rsna_event('raptor_order_index_fallback', file=str(f)); instance = len(recs)
                    pos = float(instance)
                if not np.isfinite(pos):
                    rsna_event('raptor_order_nonfinite_fallback', file=str(f)); pos = float(len(recs))
                ps = getattr(h, 'PixelSpacing', None)
                ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps)
                recs.append((pos, f, ps))
            except Exception as exc:
                rsna_event('raptor_order_unreadable_skipped', file=str(f), error=f'{type(exc).__name__}: {exc}'); continue
        recs.sort(key=lambda x: (x[0], x[1]))
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return ([(f, ps) for _, f, ps in recs], med_ps)

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape
        cpx = int(round(CROP_MM / max(ps, 0.001)))
        cpx = min(cpx, min(h, w))
        y0 = (h - cpx) // 2
        x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)
    return (order_and_meta, read_px, mm_crop_resize)

def _pick_series_for_slot(rows, plane, fluid, used):
    """An unknown protocol flag is not false and must not reach int(NaN)."""
    def flag(value):
        if value is None:
            return None
        if isinstance(value, str) and value.strip().lower() in ('', 'nan', 'none', '<na>', 'unknown'):
            return None
        try:
            number = float(value)
        except (TypeError, ValueError):
            text = str(value).strip().lower()
            if text in ('true', 'yes', 'y', 't'):
                return 1
            if text in ('false', 'no', 'n', 'f'):
                return 0
            return None  # unknown code: neither false nor fatal; the plane-level fallback applies
        if not np.isfinite(number):
            return None
        if number not in (0.0, 1.0):
            return None
        return int(number)
    candidates = [r for r in rows if r['Anatomical_Plane'] == plane
                  and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        preferred = [r for r in candidates if flag(r.get('Fluid_Sensitive')) == fluid]
        if preferred:
            return preferred[0]
    return candidates[0] if candidates else None



def find_test_root():
    from pathlib import Path
    for value in [os.environ.get('RSNA_COMP_ROOT', ''),
                  '/kaggle/input/competitions/rsna-knee-abnormality-detection',
                  '/kaggle/input/rsna-knee-abnormality-detection']:
        if value and (Path(value)/'test.csv').is_file():
            return value
    raise FileNotFoundError('explicit competition root absent')

def find_weight_file(fname):
    return str(_asset_find_asset(fname))
\'\'\'
_KE_NS = {'__name__': '_ke_raptor', '_asset_find_asset': _asset_find_asset, 'rsna_rank01':rsna_rank01, 'rsna_strict_load':rsna_strict_load, 'rsna_event':rsna_event, 'rsna_sha':rsna_sha}
exec(compile(_KE_SRC, '<raptor>', 'exec'), _KE_NS)
from concurrent.futures import Future as _KeFuture
import threading as _ke_threading

_ke_base_make_reader = _KE_NS['_make_reader']
_ke_order_cache = {}

# DINO already read these immutable tags. Pre-resolved Futures plug directly
# into Raptor's existing single-flight cache; ambiguous series are absent.
for _ke_series_dir, _ke_header_value in globals().get(
    'RAPTOR_HEADER_CACHE', {}
).items():
    _ke_future = _KeFuture()
    _ke_future.set_result(_ke_header_value)
    _ke_order_cache[str(_ke_series_dir)] = _ke_future
print(
    f"[raptor-fast] seeded {len(_ke_order_cache)} DINO header records",
    flush=True,
)

from collections import OrderedDict as _KeOrderedDict
_ke_pixel_cache = _KeOrderedDict()
_ke_order_cache_lock = _ke_threading.Lock()


def _ke_make_cached_reader():
    """Share immutable DICOM ordering metadata across Raptor views."""
    base_order, read_pixel, crop_resize = _ke_base_make_reader()

    def cached_order(series_dir):
        key = str(series_dir)
        owner = False
        with _ke_order_cache_lock:
            future = _ke_order_cache.get(key)
            if future is None:
                future = _KeFuture()
                _ke_order_cache[key] = future
                owner = True
        if owner:
            try:
                future.set_result(base_order(series_dir))
            except BaseException as error:
                future.set_exception(error)
                with _ke_order_cache_lock:
                    _ke_order_cache.pop(key, None)
                raise
        return future.result()

    def cached_pixel(path):
        global _ke_pixel_cache_bytes
        from collections import OrderedDict
        key=str(path)
        with _ke_order_cache_lock:
            got=_ke_pixel_cache.get(key)
            if got is not None:
                _ke_pixel_cache.move_to_end(key);return got
        value=read_pixel(path)
        if value.ndim!=2 or not _ke_np.isfinite(value).all():
            raise RuntimeError(f'Raptor invalid pixels {path}')
        value.setflags(write=False)
        with _ke_order_cache_lock:
            previous = _ke_pixel_cache.pop(key, None)
            if previous is not None:
                _ke_pixel_cache_bytes -= previous.nbytes
            _ke_pixel_cache[key] = value
            _ke_pixel_cache_bytes += value.nbytes
            while _ke_pixel_cache_bytes > (192 << 20):
                _, removed = _ke_pixel_cache.popitem(last=False)
                _ke_pixel_cache_bytes -= removed.nbytes
        return value
    return cached_order, cached_pixel, crop_resize


_KE_NS['_make_reader'] = _ke_make_cached_reader
if _ke_os.environ.get('RSNA_COMP_ROOT'):
    _KE_NS['find_test_root'] = lambda: _ke_os.environ['RSNA_COMP_ROOT']


def _ke_prepare_windows(arm, study_uid, series, series_root, reader):







    order_and_meta, read_px, _ = reader
    image_size = int(arm['img'])
    slots = list(arm['slots'])
    span_lo, span_hi = map(float, arm['span'])
    used, pools = set(), []
    rows = series.get(study_uid, [])
    for plane, fluid, count in slots:
        record = _KE_NS['_pick_series_for_slot'](rows, plane, fluid, used)
        if record is None:
            pools.append(None)
            continue
        used.add(record['SeriesInstanceUID'])
        files, median_spacing = order_and_meta(
            f"{series_root}/{study_uid}/{record['SeriesInstanceUID']}")
        if not files:
            rsna_event('raptor_acquisition_empty', study=str(study_uid), series=str(record['SeriesInstanceUID'])); pools.append(None); continue
        lo = int(len(files) * span_lo)
        hi = max(int(len(files) * span_hi) - 1, lo)
        original = _ke_np.linspace(lo, hi, count).round().astype(int)
        pools.append((files, median_spacing, lo, hi, original))
    capacities = [0 if p is None else p[3] - p[2] + 1 for p in pools]
    quotas = _dense_allocate(capacities, [s[2] for s in slots], 96)
    volume, sources = [], []
    import cv2
    for pool, quota in zip(pools, quotas):
        if pool is None or not quota:
            continue
        files, median_spacing, lo, hi, original = pool
        picks = lo + _dense_unique_linspace(hi - lo + 1, int(quota))
        arrays = {}
        # Decode union once. Failure is explicit, not an unreported 0.5 score.
        for i in sorted(set(original.tolist()) | set(picks.tolist())):
            arrays[i] = read_px(files[i][0])
        all_pixels = _ke_np.concatenate([arrays[int(i)].ravel() for i in original])
        low, high = _ke_np.percentile(all_pixels, [2.0, 98.0])
        for i in picks:
            path, spacing = files[int(i)]
            a = arrays[int(i)]
            a = _ke_np.clip((a-low)/(high-low+1e-6), 0, 1)
            spacing = spacing if spacing > 0 else median_spacing
            h, w = a.shape
            c = min(int(round(140.0/max(spacing, .001))), min(h, w))
            y, x = (h-c)//2, (w-c)//2
            a = cv2.resize(a[y:y+c, x:x+c], (image_size,image_size),
                           interpolation=cv2.INTER_AREA)
            volume.append((a*255).astype(_ke_np.uint8))
            sources.append(str(path))
    if len(sources) != len(set(sources)):
        raise RuntimeError('capacity-aware sampling selected duplicate source files')
    if len(volume) < 3:
        raise RuntimeError(f'{study_uid}: fewer than three unique source slices')
    volume = _ke_np.stack(volume)
    # Presence means an acquired source, not an intensity test. A genuinely
    # black acquired slice is not interchangeable with padding.
    mask = _ke_np.ones(len(volume), dtype=_ke_np.uint8)
    windows = _KE_NS['eval_windows'](
        volume, mask, k=len(volume)-2, res=int(arm['res']), norm=_KE_NS['NORM'])
    if len(windows) != min(96, sum(capacities))-2:
        raise RuntimeError('capacity-aware sampling Raptor cardinality drift')
    import hashlib, json
    signature = hashlib.sha256(json.dumps(sources, ensure_ascii=False,
                                separators=(',', ':')).encode()).hexdigest()
    audit = {'arm':arm['name'], 'uid':str(study_uid),
             'capacities':capacities, 'quotas':quotas.tolist(),
             'source_count':len(sources), 'source_list_sha256':signature,
             'windows':len(windows), 'duplicate_sources':0}
    identity = (arm['name'], str(study_uid))
    with _ke_input_lock:
        if identity in _ke_input_ids:
            raise RuntimeError(f'Duplicate Raptor preparation: {identity}')
        _ke_input_ids.add(identity)
        with _ke_input_audit_path.open('a') as handle:
            handle.write(json.dumps(audit, sort_keys=True, allow_nan=False) + '\\n')
    return windows


def _ke_infer_input(model, xwins, device):
    """Bound backbone workspace; keep all window features for one head call."""
    torch = _KE_NS['torch']
    def _forward(_amp):
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16, enabled=_amp):
            features = [model.backbone(xwins[i:i+8].to(device))
                        for i in range(0,len(xwins),8)]
            feats = torch.cat(features,dim=0).unsqueeze(0)
            return torch.sigmoid(model.head(feats).float())[0].cpu().numpy()
    probabilities = _forward(True)
    if not _ke_np.isfinite(probabilities).all():
        rsna_event('raptor_fp16_nonfinite_retry_fp32')
        probabilities = _forward(False)
    if not _ke_np.isfinite(probabilities).all():
        raise RuntimeError('nonfinite Raptor prediction')
    return probabilities


def _ke_prefetched_windows(arm, test_ids, series, series_root, reader):
    """Bound host memory while decoding one study ahead of its GPU forward."""
    depth = int(_ke_os.environ.get('RSNA_RAPTOR_PREFETCH', '2'))
    if not 1 <= depth <= 4: raise ValueError('Raptor prefetch must be 1..4')
    with _KeThreadPool(max_workers=1) as executor:
        pending = {}
        submit_at = 0
        while submit_at < min(depth, len(test_ids)):
            pending[submit_at] = executor.submit(
                _ke_prepare_windows,
                arm,
                test_ids[submit_at],
                series,
                series_root,
                reader,
            )
            submit_at += 1
        for study_index, study_uid in enumerate(test_ids):
            future = pending.pop(study_index)
            if submit_at < len(test_ids):
                pending[submit_at] = executor.submit(
                    _ke_prepare_windows,
                    arm,
                    test_ids[submit_at],
                    series,
                    series_root,
                    reader,
                )
                submit_at += 1
            yield study_index, study_uid, future


def _ke_run_raptor_arms():
    """Run the four unchanged Raptor arms across exactly two GPUs."""
    torch = _KE_NS['torch']
    if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
        raise RuntimeError(
            f"optimized Raptor requires at least one GPU, got {torch.cuda.device_count()}"
        )
    started = _ke_time.time()
    root = _KE_NS['find_test_root']()
    series_root = root + '/test_series'
    if not _ke_os.path.isdir(series_root):
        series_root = root + '/test_images'
    test = _ke_pd.read_csv(root + '/test.csv')
    test['StudyInstanceUID'] = test['StudyInstanceUID'].astype(str)
    test_ids = test['StudyInstanceUID'].tolist()
    test_series = _ke_pd.read_csv(root + '/test_series.csv')
    test_series['StudyInstanceUID'] = test_series['StudyInstanceUID'].astype(str)
    test_series['SeriesInstanceUID'] = test_series['SeriesInstanceUID'].astype(str)
    series = {
        key: frame.to_dict('records')
        for key, frame in test_series.groupby('StudyInstanceUID')
    }
    sample = root + '/sample_submission.csv'
    columns = ['StudyInstanceUID', *_KE_NS['LAB']]
    if _ke_os.path.exists(sample):
        columns = list(_ke_pd.read_csv(sample, nrows=1).columns)
    arms = list(_KE_NS['ARMS'])
    outputs = [
        _ke_np.full((len(test_ids), len(_KE_NS['LAB'])), np.nan, _ke_np.float32)
        for _ in arms
    ]
    print(
        f"[raptor-fast] {len(test_ids)} studies; balanced arm groups "
        f"cuda:0=[0,2], cuda:1=[1,3]",
        flush=True,
    )

    def run_single(arm_index, device):
        arm = arms[arm_index]
        reader = _KE_NS['_make_reader']()
        weight_path = _KE_NS['find_weight_file'](arm['file'])
        model, resolution = _KE_NS['load_model'](
            weight_path, arm['arch'], arm['res'], device
        )
        if int(resolution) != int(arm['res']):
            raise RuntimeError(
                f"{arm['name']} checkpoint resolution {resolution} != {arm['res']}"
            )
        for study_index, study_uid, future in _ke_prefetched_windows(
            arm, test_ids, series, series_root, reader
        ):
            rsna_deadline('Raptor full-cohort inference')
            try:
                windows = future.result()
                outputs[arm_index][study_index] = _KE_NS['infer_probs'](
                    model, windows, device
                )
                del windows
            except Exception as error:
                rsna_event('raptor_study_failed', arm=str(arm['name']), study=str(study_uid), error=f'{type(error).__name__}: {str(error)[:500]}')
                with _ke_input_lock:
                    _ke_input_ids.add((arm['name'], str(study_uid)))
        del model
        _ke_gc.collect()
        with torch.cuda.device(device):
            torch.cuda.empty_cache()

    def run_shared_maxspan(device):
        first, reverse = arms[0], arms[2]
        comparable = ('file', 'arch', 'res', 'img', 'slots', 'span', 'k_eval')
        if any(first[key] != reverse[key] for key in comparable):
            raise RuntimeError('MaxSpan forward/reverse arms no longer share preprocessing')
        reader = _KE_NS['_make_reader']()
        weight_path = _KE_NS['find_weight_file'](first['file'])
        model, resolution = _KE_NS['load_model'](
            weight_path, first['arch'], first['res'], device
        )
        if int(resolution) != int(first['res']):
            raise RuntimeError(
                f"MaxSpan checkpoint resolution {resolution} != {first['res']}"
            )
        for study_index, study_uid, future in _ke_prefetched_windows(
            first, test_ids, series, series_root, reader
        ):
            rsna_deadline('Raptor full-cohort inference')
            try:
                windows = future.result()
                outputs[0][study_index] = _KE_NS['infer_probs'](
                    model, windows, device
                )
                outputs[2][study_index] = _KE_NS['infer_probs'](
                    model, windows.flip(1).contiguous(), device
                )
                del windows
            except Exception as error:
                rsna_event('raptor_study_failed', arm=str(first['name']), study=str(study_uid), error=f'{type(error).__name__}: {str(error)[:500]}')
                with _ke_input_lock:
                    _ke_input_ids.add((first['name'], str(study_uid))); _ke_input_ids.add((reverse['name'], str(study_uid)))
        del model
        _ke_gc.collect()
        with torch.cuda.device(device):
            torch.cuda.empty_cache()

    if torch.cuda.device_count() >= 2:
        def gpu_zero():
            with torch.cuda.device(0):
                run_shared_maxspan(torch.device('cuda:0'))

        def gpu_one():
            with torch.cuda.device(1):
                run_single(1, torch.device('cuda:1'))
                run_single(3, torch.device('cuda:1'))

        with _KeThreadPool(max_workers=2) as executor:
            workers = [executor.submit(gpu_zero), executor.submit(gpu_one)]
            for w in workers:
                w.result()
    else:
        with torch.cuda.device(0):
            run_shared_maxspan(torch.device('cuda:0'))
            run_single(1, torch.device('cuda:0'))
            run_single(3, torch.device('cuda:0'))

    _ke_np.savez_compressed('/kaggle/working/raptor_raw.npz',
        study_uids=_ke_np.asarray(test_ids), raw_probabilities=_ke_np.stack(outputs))
    for _ai, _arr in enumerate(outputs):
        _bad = ~_ke_np.isfinite(_arr).all(axis=1)
        if _bad.any():
            _fill = _ke_np.nanmean(_ke_np.where(_ke_np.isfinite(_arr), _arr, _ke_np.nan), axis=0) if (~_bad).any() else _ke_np.full(_arr.shape[1], .5, _arr.dtype)
            _arr[_bad] = _ke_np.where(_ke_np.isfinite(_fill), _fill, .5)
            rsna_event('raptor_neutral_fill', arm=str(arms[_ai]['name']), studies=int(_bad.sum()))
    rsna_finite(_ke_np.stack(outputs),'all four Raptor views',probability=True)
    weights = _ke_np.asarray([float(arm['w']) for arm in arms], _ke_np.float64)
    weights /= weights.sum()
    probability_blend = _ke_np.tensordot(
        weights,
        _ke_np.stack([_ke_np.clip(value, 0, 1) for value in outputs]),
        axes=(0, 0),
    )
    ranks = _KE_NS['rankpct'](probability_blend)
    rsna_finite(ranks,'Raptor ranked ensemble',probability=True)
    submission = _ke_pd.DataFrame(ranks.astype(_ke_np.float32), columns=_KE_NS['LAB'])
    submission.insert(0, 'StudyInstanceUID', test_ids)
    submission = submission[columns]
    if submission['StudyInstanceUID'].tolist() != test_ids:
        raise RuntimeError('Raptor study order drift')
    if not _ke_np.isfinite(submission[_KE_NS['LAB']].to_numpy()).all():
        raise RuntimeError('Raptor produced non-finite predictions')
    output = _KePath('/kaggle/working/_raptor.csv')
    submission.to_csv(output, index=False)
    print(
        f"[raptor-fast] wrote {output}; elapsed {_ke_time.time() - started:.1f}s",
        flush=True,
    )


_ke_input_lock = _ke_threading.Lock()
_ke_input_ids = set()
_ke_pixel_cache_bytes = 0
_ke_input_audit_path = _KePath('/kaggle/working/diagnostics/raptor_inputs.jsonl')
_ke_input_audit_path.unlink(missing_ok=True)
_KE_NS['infer_probs'] = _ke_infer_input
for _arm in _KE_NS['ARMS']:
    _arm['parent_k_eval'] = _arm['k_eval']
    _arm['k_eval'] = 94
rsna_phase('public_raptor', 'START')
_ke_run_raptor_arms()
rsna_phase('public_raptor', 'COMPLETE')
_ke_pixel_cache.clear()
_ke_order_cache.clear()
_ke_gc.collect()
_KePath('/kaggle/working/raptor_input_before_coat.csv').write_bytes(_KePath('/kaggle/working/_raptor.csv').read_bytes())
import json as _asset_json
_KePath('/kaggle/working/raptor_input_summary.json').write_text(
    _asset_json.dumps({'studies':len(_RSNA_TEST_IDS), 'preparations':len(_ke_input_ids),
                    'view_names':[a['name'] for a in _KE_NS['ARMS']],
                    'shared_preparation_views':['maxspan-v5','maxspan-v5-reverse'],
                    'records':str(_ke_input_audit_path)},indent=2))



def _coat_substitute():
    import hashlib as _h, os as _o, subprocess as _sp, sys as _sy
    from pathlib import Path as _P
    import pandas as _pd

    import numpy as _np
    _P('/kaggle/working/_raptor_public.csv').write_bytes(
        _P('/kaggle/working/_raptor.csv').read_bytes())
    _np.savez_compressed('/kaggle/working/btk_v32_rest_input.npz',
        study_uids=_ke_ours.StudyInstanceUID.to_numpy().astype(str),
        labels=_np.asarray(_KE_LAB), values=_ke_ours[_KE_LAB].to_numpy())
    MAN_SHA = '98511a8fdeb9da0e6e70c78d013ff636e1476f31c80b5dc134d294b18c3f284e'
    WHL_SHA = '236c8df54a90f4d02076e6f9c1cc763d794542e886c576a6fee46ec8ff75a7a9'
    raptor = _P('/kaggle/working/_raptor.csv')

    def sha(p):
        d = _h.sha256()
        with _P(p).open('rb') as f:
            for b in iter(lambda: f.read(8 << 20), b''):
                d.update(b)
        return d.hexdigest()

    def find(name, want):
        return _asset_find_asset(name, want)

    man = find('coat_resgated_ep10_top3_manifest.json', MAN_SHA)
    if man is None:
        raise RuntimeError('coat manifest absent or hash mismatch')

    art = man.parent

    whl = find('opencv_python_headless-4.12.0.88-*.whl', WHL_SHA)
    envd = _P('/kaggle/working/_coat_env')
    _sp.run(
        [
            _sy.executable,
            '-m',
            'pip',
            'install',
            '--no-deps',
            '--quiet',
            '--target',
            str(envd),
            str(whl),
        ],
        check=True,
    )

    out = _P('/kaggle/working/_coat_arm.csv')

    import inspect
    runtime_path = _P('/kaggle/working/input_resgated_runtime.py')
    helper_source = 'import numpy as np\\n' + '\\n'.join(inspect.getsource(f) for f in
        (_dense_allocate, _dense_unique_linspace, _dense_coat_specs))
    runtime_sha = _asset_write_coat_runtime(art,runtime_path,helper_source)

    child = (
        "import sys, json, os\\n"
        f"sys.path.insert(0, {str(envd)!r})\\n"
        f"sys.path.insert(0, {str(art)!r})\\n"
        "import cv2; assert cv2.__version__ == '4.12.0', cv2.__version__\\n"
        "import torch; assert torch.cuda.device_count() >= 1\\n"
        "sys.path.insert(0, '/kaggle/working')\\n"
        "import input_resgated_runtime as rt\\n"
        "assert rt.base.cv2.__version__ == '4.12.0'\\n"
        "from pathlib import Path\\n"
        "r = rt.run_submission("
        "competition_root=Path(os.environ['RSNA_COMP_ROOT']) "
        "if os.environ.get('RSNA_COMP_ROOT') "
        "else rt.base.find_competition_root(),\\n"
        f"    artifact_root=Path({str(art)!r}), "
        f"output_path=Path({str(out)!r}),\\n"
        "    gpu_batch_studies=2, backbone_micro_images=8)\\n"
        "assert r['status'] == rt.SUBMISSION_STATUS\\n"
        "assert r['models'] == 3\\n"
        "assert [i['epoch'] for i in r['checkpoints']] == [4, 6, 8]\\n"
        "r['delivery_contract_verified'] = True\\n"
        "print('[coat-child] fallback_studies', r['fallback_studies'], flush=True)\\n"
        "Path('/kaggle/working/_coat_arm_receipt.json').write_text("
        "json.dumps(r, indent=2))\\n"
    )

    env = dict(_o.environ)
    env['CUDNN_CONV_WSCAP_DBG'] = '1024'
    env['RSNA_COMP_ROOT'] = _KE_NS['find_test_root']()
    env['PYTHONPATH'] = f"{envd}:{art}:" + env.get('PYTHONPATH', '')

    import json as _j
    d4_out = _P('/kaggle/working/d4_input')/'d4.csv'

    def _run_residual_reader():
        _run_required_child([_sy.executable, '-c', child], env,
                            _P('/kaggle/working/resgated_runtime.log'))
        return _j.loads(_P('/kaggle/working/_coat_arm_receipt.json').read_text())

    def _run_d4_reader():
        d4_manifest = _asset_find_asset('coatnet_pairfilm_manifest.json',
            '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d')
        timm_whl = d4_manifest.parent/'timm-1.0.22-py3-none-any.whl'
        if sha(timm_whl) != '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0':
            raise RuntimeError('D4 timm wheel changed')
        d4_env = _P('/kaggle/working/_d4_env')
        _sp.run([_sy.executable,'-m','pip','install','--no-deps','--quiet',
            '--target',str(d4_env),str(timm_whl)],check=True)
        d4_dir = _P('/kaggle/working/d4_input')
        d4_dir.mkdir(exist_ok=True)
        return _asset_run_d4(
            d4_manifest.parent, d4_env, _KE_NS['find_test_root'](), d4_out
        )

    _parallel_coat_readers = _o.environ.get(
        'RSNA_PARALLEL_COAT_READERS', '1'
    ) == '1'
    _coat_receipt = None
    _d4_receipt = None
    _coat_exc = None
    _d4_exc = None
    if _parallel_coat_readers:
        rsna_phase('coat_readers', 'START', parallel=True)
        with _KeThreadPool(max_workers=2) as _coat_pool:
            _coat_future = _coat_pool.submit(_run_residual_reader)
            _d4_future = _coat_pool.submit(_run_d4_reader)
            try:
                _coat_receipt = _coat_future.result()
            except Exception as _error:
                _coat_exc = _error
            try:
                _d4_receipt = _d4_future.result()
            except Exception as _error:
                _d4_exc = _error
        rsna_phase(
            'coat_readers', 'COMPLETE', parallel=True,
            residual_ok=_coat_exc is None, d4_ok=_d4_exc is None,
        )
    else:
        rsna_phase('residual_coat', 'START')
        try:
            _coat_receipt = _run_residual_reader()
        except Exception as _error:
            _coat_exc = _error
        rsna_phase('residual_coat', 'COMPLETE', ok=_coat_exc is None)
        rsna_phase('d4', 'START')
        try:
            _d4_receipt = _run_d4_reader()
        except Exception as _error:
            _d4_exc = _error
        rsna_phase('d4', 'COMPLETE', ok=_d4_exc is None)

    _coat_resgated_ok = _coat_exc is None
    if _coat_resgated_ok:
        if int(_coat_receipt.get('fallback_studies', 0) or 0):
            rsna_event('coat_resgated_fallback_studies', count=int(_coat_receipt['fallback_studies']), failures=list(_coat_receipt.get('failures', []))[:20])
    else:
        rsna_event('coat_resgated_child_failed', error=f'{type(_coat_exc).__name__}: {str(_coat_exc)[:1500]}')
        print(f'[coat-arm] residual CoAt child FAILED; continuing without it (flagged): {type(_coat_exc).__name__}', flush=True)

    _coat_d4_ok = _d4_exc is None
    if not _coat_d4_ok:
        rsna_event('coat_d4_child_failed', error=f'{type(_d4_exc).__name__}: {str(_d4_exc)[:1500]}')
        print(f'[coat-arm] D4 CoAt child FAILED; continuing without it (flagged): {type(_d4_exc).__name__}', flush=True)

    pub = _pd.read_csv(raptor, dtype={'StudyInstanceUID': str})
    pub = rsna_frame(pub,_RSNA_TEST_IDS,_KE_LAB,'public Raptor')
    lab = [c for c in pub.columns if c != 'StudyInstanceUID']
    import numpy as _np
    # Retain BTKD inner blend coefficients; input/new family scores are unvalidated.
    private_alpha = 0.4
    public_rank = pub[lab].rank(method='average', pct=True)
    _members = []
    if _coat_resgated_ok:
        try:
            ours = _pd.read_csv(out, dtype={'StudyInstanceUID': str})
            ours = rsna_frame(ours,_RSNA_TEST_IDS,_KE_LAB,'resgated CoAt')
            if list(ours.columns) != list(pub.columns):
                raise RuntimeError('coat arm column drift')
            ours = ours.set_index('StudyInstanceUID').reindex(pub.StudyInstanceUID.astype(str).tolist()).reset_index()
            if ours[lab].isna().any().any():
                raise RuntimeError('coat arm does not cover every study')
            rsna_save_predictions('coat_resgated_rank',_RSNA_TEST_IDS,ours[lab].to_numpy(),lab)
            _members.append(('resgated_top3', ours[lab].rank(method='average', pct=True)))
        except Exception as _exc:
            rsna_event('coat_resgated_output_rejected', error=f'{type(_exc).__name__}: {str(_exc)[:500]}')
    if _coat_d4_ok:
        try:
            d4 = _pd.read_csv(d4_out,dtype={'StudyInstanceUID':str})
            d4 = rsna_frame(d4,_RSNA_TEST_IDS,_KE_LAB,'D4 CoAt')
            rsna_save_predictions('coat_d4_rank',_RSNA_TEST_IDS,d4[lab].to_numpy(),lab)
            if d4.StudyInstanceUID.duplicated().any() or list(d4.columns)!=list(pub.columns):
                raise RuntimeError('D4 schema/UID drift')
            d4 = d4.set_index('StudyInstanceUID').reindex(pub.StudyInstanceUID.astype(str))
            if d4[lab].isna().any().any():
                raise RuntimeError('D4 missing predictions')
            _members.append(('d4_swa3', d4[lab].reset_index(drop=True).rank(method='average',pct=True)))
        except Exception as _exc:
            rsna_event('coat_d4_output_rejected', error=f'{type(_exc).__name__}: {str(_exc)[:500]}')
    if not _members:
        rsna_event('coat_family_unavailable', note='public Raptor kept as the CoAt/Raptor input (flagged)')
        print('[coat-arm] no CoAt family member available; public Raptor retained (flagged)', flush=True)
        return 0
    if len(_members) == 2:
        private_rank = _asset_half_rank_mix(_members[0][1], _members[1][1])
    else:
        rsna_event('coat_family_partial', member=_members[0][0])
        private_rank = _members[0][1]
    family = pub.copy()
    family[lab] = private_rank
    family.to_csv('/kaggle/working/_coat_family_rank.csv', index=False)
    hybrid = pub.copy()
    hybrid[lab] = (1.0 - private_alpha) * public_rank + private_alpha * private_rank
    if not _np.isfinite(hybrid[lab].to_numpy(_np.float64)).all():
        raise RuntimeError('CoAt/Raptor hybrid contains non-finite values')
    tmp = raptor.with_name('.raptor_coat_hybrid.csv')
    hybrid.to_csv(tmp, index=False)
    _o.replace(tmp, raptor)
    raptor.with_name('_coat_raptor_blend_receipt.json').write_text(_j.dumps({
        'contract': 'public_raptor_private_residual_coat_v32_no_inner_rerank_v1', 'inner_rerank': False, 'private_alpha': private_alpha,
        'within_coat': {m: 0.5 for m, _ in _members} if len(_members) == 2 else {_members[0][0]: 1.0}, 'public_raptor_alpha': 1.0 - private_alpha,
        'study_count': len(pub), 'finding_specific_weights': False, 'members': [m for m, _ in _members]}, indent=2, sort_keys=True) + '\\n')
    return len(pub)

_coat_n = _coat_substitute()
print(f'[coat-arm] residual top3 / D4 SWA3 equal rank family; BTKD32 no-inner-rerank; {_coat_n} studies',flush=True)

_ke_theirs = _ke_pd.read_csv('/kaggle/working/_raptor.csv',
                             dtype={'StudyInstanceUID': str})
assert list(_ke_theirs.columns) == list(_ke_ours.columns), 'column drift'
_ke_theirs = _ke_theirs.set_index('StudyInstanceUID').reindex(
    _ke_ours['StudyInstanceUID']).reset_index()
assert _ke_theirs[_KE_LAB].notna().all().all(), 'study identity drift'


_ke_tr = _ke_ours[_KE_LAB].rank(method='average', pct=True)
_ke_cr = _ke_theirs[_KE_LAB].copy()
_blend_transformer = _ke_ours.copy()
_blend_coatnet = _ke_theirs.copy()
_blend_labels = list(_KE_LAB)
_blend_tr = _ke_tr.copy()
_blend_cr = _ke_cr.copy()
_coatnet_weight = {label: 0.60 for label in _blend_labels}
_coatnet_weight.update({
    'ACL': 0.75,
    'Medial Meniscus': 0.80,
    'Lateral Meniscus': 1.00,
    'Lateral OA': 0.75,
    'Fracture': 0.75,
})
_blend_output = _blend_transformer.copy()
for _blend_label in _blend_labels:
    _blend_w = float(_coatnet_weight[_blend_label])
    _blend_output[_blend_label] = (
        (1.0 - _blend_w) * _blend_tr[_blend_label]
        + _blend_w * _blend_cr[_blend_label]
    )
_blend_output[_blend_labels] = _blend_output[_blend_labels].rank(
    method='average', pct=True
)
assert _ke_np.isfinite(
    _blend_output[_blend_labels].to_numpy(_ke_np.float64)
).all()
_blend_output.to_csv(_ke_primary, index=False)
'''
if FIXED_RUN['mode'] == 'inference':
    execute_pinned_stage(GPU_STAGE_033, 33, 'feba3fd71eb0f6a45f2c996ce3958228ba0b15c6935e447418cbebf9642668d3')


## 3. Frozen candidate set — no extra model calls

| Name | Change relative to reference | Evidence status |
|---|---|---|
| `reference` | Preserved final reference scores | Public source family displays 0.942; this file is unscored |
| `reference_probability_family` | Rank the equal mean of available residual-CoAt and D4 probability means, then keep the original Raptor/outer blend | New, unvalidated hypothesis |
| `reference_no_calibrator` | Reuse the saved transformer output before inherited calibration | Unvalidated ablation |
| `reference_rad_fracture` | Reuse the already computed Rad main contribution for Fracture | Public author reported weak-label gain and Gold58 regression; unvalidated here |

The probability candidate averages the saved family means in **float64**, then
uses average-tie percentile ranks. It preserves 0.60 Raptor / 0.40 CoAt-family
mixing, no additional rank inside that mixture, and the inherited label-specific
outer weights. Residual CSV checkpoint reduction is not established locally;
using its stored probability mean may change that reduction as well as pooling.
This candidate does **not** reproduce V32, which also adds Global96.

No candidate is selected from the test-score distribution. Differences between
unlabeled predictions are reported as changed scores, never as measured AUC.


In [ ]:
REPLAY_MODULE_SOURCE = r'''"""CPU-only replay of an already-computed RSNA Knee prediction cohort.

This module saves model outputs, their exact study/label identities, and source
file hashes in an immutable NPZ + JSON pair. Replaying that pair performs no
image decoding, model inference, model training, or automatic candidate choice.
The existing reference scores are copied without reranking or recalibration.

The only new fixed candidate takes the float64 probability mean of the two EXISTING
CoAtNet families (Residual and D4), ranks that mean once, and preserves the
existing 0.60/0.40 Raptor/family mixture and per-finding outer weights. It is an
unvalidated ablation, not a demonstrated score improvement. No additional
model, GPU pass, cross-finding clinical rule, or pseudo-label is used.

Hashes detect accidental changes and mismatched files; they are not signatures
and cannot prove truthful checkpoint-training provenance. Current input CSVs
must be supplied on load. A public preview cannot be replayed for a hidden-test
cohort, and preview bundles cannot be used with purpose='submission'.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from dataclasses import dataclass
import csv
import hashlib
import json
import os
from pathlib import Path
import re
import tempfile
from types import MappingProxyType

import numpy as np
import pandas as pd


ID_COLUMN = "StudyInstanceUID"
LABELS = (
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
)
SCHEMA_VERSION = 1
REQUIRED_METADATA = ("test.csv", "test_series.csv", "sample_submission.csv")
REQUIRED_FRAMES = ("reference", "transformer", "raptor_public")
REQUIRED_PROBABILITY_FAMILIES = ("resgated_top3", "d4_swa3")
OUTER_WEIGHTS = (.75, .60, .80, 1.0, .60, .75, .60, .60, .60, .60, .60, .75)
FIXED_CANDIDATES = (
    "reference", "reference_probability_family", "reference_no_calibrator",
    "reference_rad_fracture",
)
_NAME = re.compile(r"[A-Za-z][A-Za-z0-9_]{0,63}\Z")
_HEX = re.compile(r"[0-9a-f]{64}\Z")
_MAX_ARCHIVE_BYTES = 256 << 20


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def _canonical(obj) -> bytes:
    return json.dumps(obj, ensure_ascii=False, allow_nan=False, sort_keys=True,
                      separators=(",", ":")).encode("utf-8")


def _fingerprint(obj) -> str:
    return hashlib.sha256(_canonical(obj)).hexdigest()


def _identities(study_ids, labels):
    if isinstance(study_ids, (str, bytes)):
        raise ValueError("study_ids must be an ordered collection of strings")
    ids = list(study_ids)
    if (not ids or any(not isinstance(uid, str) or not uid.strip() for uid in ids)
            or len(ids) != len(set(ids))):
        raise ValueError("Study IDs must be nonempty unique strings; preserve leading zeroes")
    if list(labels) != list(LABELS):
        raise ValueError("Labels/order must exactly match the 12 competition findings")
    return ids, list(LABELS)


def _align(frame: pd.DataFrame, ids: Sequence[str], name: str) -> pd.DataFrame:
    if not isinstance(frame, pd.DataFrame) or frame.columns.tolist() != [ID_COLUMN, *LABELS]:
        raise ValueError(f"{name}: expected exact official ID and label columns/order")
    values = frame[ID_COLUMN].tolist()
    if (any(not isinstance(uid, str) or not uid.strip() for uid in values)
            or len(values) != len(set(values)) or set(values) != set(ids)):
        raise ValueError(f"{name}: missing, extra, duplicate, or nonstring study IDs")
    result = frame.set_index(ID_COLUMN).loc[list(ids)].reset_index().copy(deep=True)
    numeric = result[list(LABELS)].to_numpy(dtype=np.float64)
    if not np.isfinite(numeric).all() or np.any((numeric < 0) | (numeric > 1)):
        raise ValueError(f"{name}: scores must be finite and in [0, 1]")
    # Canonical float64 retains the exact numerical value of every float32 or
    # float64 input. No rank, clipping, rounding, or calibration occurs here.
    result[list(LABELS)] = numeric
    return result


def _csv(path: Path) -> pd.DataFrame:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        header = next(csv.reader(handle), [])
    if not header or len(header) != len(set(header)):
        raise ValueError(f"{path.name}: empty or duplicate CSV header")
    return pd.read_csv(path, dtype={ID_COLUMN: str, "SeriesInstanceUID": str},
                       float_precision="round_trip")


def _file_record(path: str | Path) -> dict:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing provenance artifact: {path}")
    return {"path": str(path.resolve()), "bytes": path.stat().st_size,
            "sha256": sha256_file(path)}


def _metadata_snapshot(paths: Mapping[str, str | Path], ids: Sequence[str]) -> dict:
    if set(paths) != set(REQUIRED_METADATA):
        raise ValueError(f"metadata_paths must contain exactly {REQUIRED_METADATA}")
    records = {name: _file_record(paths[name]) for name in REQUIRED_METADATA}
    tables = {name: _csv(Path(paths[name])) for name in REQUIRED_METADATA}
    test = tables["test.csv"]
    if ID_COLUMN not in test or test[ID_COLUMN].tolist() != list(ids):
        raise ValueError("Current test.csv study order/cohort differs from expected study_ids")
    _align(tables["sample_submission.csv"], ids, "current sample_submission.csv")
    series = tables["test_series.csv"]
    if not {ID_COLUMN, "SeriesInstanceUID"}.issubset(series.columns):
        raise ValueError("Current test_series.csv lacks study/series identifiers")
    if (series[[ID_COLUMN, "SeriesInstanceUID"]].isna().any().any()
            or series[ID_COLUMN].str.strip().eq("").any()
            or series.SeriesInstanceUID.str.strip().eq("").any()
            or set(series[ID_COLUMN]) != set(ids)
            or series.duplicated([ID_COLUMN, "SeriesInstanceUID"]).any()):
        raise ValueError("Current test_series.csv has missing, duplicate, or wrong-cohort series")
    for name, record in records.items():
        if sha256_file(paths[name]) != record["sha256"]:
            raise RuntimeError(f"Metadata changed while being read: {name}")
    return records


def _array_record(array: np.ndarray) -> dict:
    array = np.ascontiguousarray(array)
    header = {"shape": list(array.shape), "dtype": array.dtype.str}
    digest = hashlib.sha256(_canonical(header))
    digest.update(array.tobytes(order="C"))
    return {**header, "sha256": digest.hexdigest()}


def _paths(path: str | Path) -> tuple[Path, Path]:
    path = Path(path)
    if path.suffix != ".npz":
        raise ValueError("Prediction bundle path must end in .npz")
    return path, path.with_suffix(".json")


def _freeze(value):
    if isinstance(value, dict):
        return MappingProxyType({key: _freeze(item) for key, item in value.items()})
    if isinstance(value, list):
        return tuple(_freeze(item) for item in value)
    return value


@dataclass(frozen=True)
class PredictionBundle:
    """Loaded immutable predictions; frame() always returns a writable copy."""

    study_ids: tuple[str, ...]
    labels: tuple[str, ...]
    scores: Mapping[str, np.ndarray]
    probabilities: Mapping[str, np.ndarray]
    manifest: Mapping
    path: Path

    def frame(self, name: str, *, probability: bool = False) -> pd.DataFrame:
        source = self.probabilities if probability else self.scores
        values = np.array(source[name], dtype=np.float64, copy=True)
        result = pd.DataFrame(values, columns=list(self.labels))
        result.insert(0, ID_COLUMN, list(self.study_ids))
        return result


def export_prediction_bundle(
    path: str | Path, *, study_ids: Sequence[str], labels: Sequence[str],
    frames: Mapping[str, pd.DataFrame],
    probability_families: Mapping[str, pd.DataFrame],
    metadata_paths: Mapping[str, str | Path], artifact_paths: Mapping[str, str | Path],
    provenance: Mapping, run_id: str, cohort_role: str = "test",
) -> dict:
    """Write a new immutable NPZ + JSON pair, refusing any existing target.

    Call only after successful normal inference and full-ensemble verification.
    Required frames: reference, transformer, raptor_public. Raw probability
    families must be the actual means for resgated_top3 and d4_swa3, with
    explicit IDs/labels. Rank CSVs are not a substitute for those probabilities.
    Additional already-computed branch frames and the two named ablations may
    be stored. Artifact paths record actual file hashes (receipts, runtime,
    prediction diagnostics, optional checkpoint files), not asserted hashes.
    """
    archive_path, sidecar_path = _paths(path)
    ids, labels = _identities(study_ids, labels)
    if not isinstance(run_id, str) or not run_id.strip():
        raise ValueError("A nonempty run_id is required")
    if cohort_role not in {"test", "audit_train", "independent_validation"}:
        raise ValueError("Unsupported cohort_role")
    if not set(REQUIRED_FRAMES).issubset(frames):
        raise ValueError(f"frames must contain {REQUIRED_FRAMES}")
    if set(probability_families) != set(REQUIRED_PROBABILITY_FAMILIES):
        raise ValueError("Exactly the Residual and D4 raw probability families are required")
    if any(not isinstance(name, str) or not _NAME.fullmatch(name) for name in frames):
        raise ValueError("Prediction branch names must be safe unique identifiers")
    if not artifact_paths or not isinstance(provenance, Mapping):
        raise ValueError("Actual artifact_paths and explicit provenance are required")
    for name in ("gpu_stage_fingerprint", "preprocessing_fingerprint"):
        if not isinstance(provenance.get(name), str) or not _HEX.fullmatch(provenance[name]):
            raise ValueError(f"provenance.{name} must be an actual SHA256 source fingerprint")
    if archive_path.exists() or sidecar_path.exists():
        raise FileExistsError("Prediction bundle is immutable; use a new run-specific path")
    metadata = _metadata_snapshot(metadata_paths, ids)
    artifacts = {str(name): _file_record(value) for name, value in artifact_paths.items()}
    # A JSON round trip rejects unserializable or nonfinite provenance before
    # any file is published and prevents mutation of the caller's object.
    declared_provenance = json.loads(_canonical(dict(provenance)))
    arrays = {"study_ids": np.asarray(ids, dtype=np.str_),
              "labels": np.asarray(labels, dtype=np.str_)}
    for name, frame in frames.items():
        arrays["score__" + name] = _align(frame, ids, name)[labels].to_numpy(np.float64)
    for name, frame in probability_families.items():
        arrays["probability__" + name] = _align(frame, ids, name)[labels].to_numpy(np.float64)
    identity = {"study_ids": ids, "labels": labels, "cohort_role": cohort_role}
    metadata_identity = {name: {key: record[key] for key in ("sha256", "bytes")}
                         for name, record in metadata.items()}
    core = {
        "schema_version": SCHEMA_VERSION, "format": "rsna_fixed_compute_prediction_bundle",
        "run_id": run_id, "cohort_role": cohort_role, "study_count": len(ids),
        "public_preview_sized": cohort_role == "test" and len(ids) <= 3,
        "study_ids": ids, "labels": labels, "identity_sha256": _fingerprint(identity),
        "metadata": metadata, "metadata_sha256": _fingerprint(metadata_identity),
        "artifacts": artifacts, "provenance": declared_provenance,
        "arrays": {name: _array_record(array) for name, array in arrays.items()},
        "score_branches": list(frames), "probability_families": list(probability_families),
        "candidate_policy": {
            "fixed_candidates": list(FIXED_CANDIDATES), "auto_selection": False,
            "new_gpu_passes": 0, "uses_truth": False,
            "reference_policy": "copy_exact_scores_without_reranking",
            "probability_family_rule": "rank((float64(resgated_top3_probability)+float64(d4_swa3_probability))/2.0)",
            "family_arithmetic": "preserve stored means, cast each to float64, add, divide by float64 2.0",
            "public_raptor_weight": .60, "coatnet_family_weight": .40,
            "outer_weights": list(OUTER_WEIGHTS),
        },
        "score_claim": "UNVALIDATED_ABLATIONS_NO_KAGGLE_SCORE_CLAIM",
    }
    core_bytes = _canonical(core)
    arrays["manifest_core_json"] = np.frombuffer(core_bytes, dtype=np.uint8)
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    made_archive = made_sidecar = False
    with tempfile.TemporaryDirectory(prefix=".rsna-bundle-", dir=archive_path.parent) as temporary:
        staged_archive = Path(temporary) / "bundle.npz"
        staged_sidecar = Path(temporary) / "bundle.json"
        with staged_archive.open("wb") as handle:
            np.savez_compressed(handle, **arrays)
            handle.flush()
            os.fsync(handle.fileno())
        envelope = {"core": core, "core_sha256": hashlib.sha256(core_bytes).hexdigest(),
                    "archive_sha256": sha256_file(staged_archive),
                    "archive_bytes": staged_archive.stat().st_size}
        with staged_sidecar.open("wb") as handle:
            handle.write(_canonical(envelope) + b"\n")
            handle.flush()
            os.fsync(handle.fileno())
        try:
            # Hard links publish without overwriting a concurrently-created run.
            os.link(staged_archive, archive_path)
            made_archive = True
            os.link(staged_sidecar, sidecar_path)
            made_sidecar = True
            archive_path.chmod(0o444)
            sidecar_path.chmod(0o444)
        except BaseException:
            if made_archive:
                archive_path.unlink(missing_ok=True)
            if made_sidecar:
                sidecar_path.unlink(missing_ok=True)
            raise
    return envelope


def load_prediction_bundle(
    path: str | Path, *, expected_study_ids: Sequence[str], expected_labels: Sequence[str],
    expected_metadata_paths: Mapping[str, str | Path], expected_cohort_role: str = "test",
    expected_run_id: str | None = None, purpose: str = "same_cohort",
    expected_artifact_paths: Mapping[str, str | Path] | None = None,
) -> PredictionBundle:
    """Verify a saved bundle against the live metadata for this exact cohort.

    Paths may move; byte hashes, ordered IDs, labels, and cohort role must match.
    No subsetting or concatenation of preview/other-cohort predictions is allowed.
    purpose='analysis' permits training-audit bundles but never selects a winner.
    purpose='submission' requires a non-preview test cohort. On Kaggle's three-row
    preview, use same_cohort solely to examine that current preview's candidates.
    """
    archive_path, sidecar_path = _paths(path)
    ids, labels = _identities(expected_study_ids, expected_labels)
    if purpose not in {"same_cohort", "submission", "analysis"}:
        raise ValueError("Unsupported replay purpose")
    if not archive_path.is_file() or not sidecar_path.is_file():
        raise FileNotFoundError("Prediction replay requires both the NPZ and its JSON sidecar")
    if sidecar_path.stat().st_size > (16 << 20):
        raise ValueError("Prediction bundle sidecar is unexpectedly large")
    envelope = json.loads(sidecar_path.read_text())
    if set(envelope) != {"core", "core_sha256", "archive_sha256", "archive_bytes"}:
        raise ValueError("Prediction bundle envelope schema changed")
    core = envelope["core"]
    if (core.get("schema_version") != SCHEMA_VERSION
            or core.get("format") != "rsna_fixed_compute_prediction_bundle"):
        raise ValueError("Unsupported prediction bundle schema")
    if _fingerprint(core) != envelope["core_sha256"]:
        raise ValueError("Prediction manifest SHA256 mismatch")
    if (archive_path.stat().st_size != envelope["archive_bytes"]
            or sha256_file(archive_path) != envelope["archive_sha256"]):
        raise ValueError("Prediction archive SHA256/size mismatch")
    if (core["study_ids"] != ids or core["labels"] != labels
            or core["study_count"] != len(ids) or core["cohort_role"] != expected_cohort_role):
        raise ValueError("Prediction bundle cohort, label order, or role mismatch")
    if expected_cohort_role not in {"test", "audit_train", "independent_validation"}:
        raise ValueError("Unsupported expected_cohort_role")
    if expected_run_id is not None and core["run_id"] != expected_run_id:
        raise ValueError("Prediction bundle belongs to a different inference run")
    identity = {"study_ids": ids, "labels": labels, "cohort_role": expected_cohort_role}
    if _fingerprint(identity) != core["identity_sha256"]:
        raise ValueError("Prediction identity fingerprint mismatch")
    preview_sized = expected_cohort_role == "test" and len(ids) <= 3
    if core["public_preview_sized"] is not preview_sized:
        raise ValueError("Inconsistent preview declaration")
    if purpose == "submission" and (expected_cohort_role != "test" or preview_sized):
        raise ValueError("Preview or training/validation bundles cannot replay a scored submission")
    metadata = _metadata_snapshot(expected_metadata_paths, ids)
    current_identity = {name: {key: record[key] for key in ("sha256", "bytes")}
                        for name, record in metadata.items()}
    stored_identity = {name: {key: record[key] for key in ("sha256", "bytes")}
                       for name, record in core["metadata"].items()}
    if (current_identity != stored_identity
            or _fingerprint(current_identity) != core["metadata_sha256"]):
        raise ValueError("Current metadata differs from the saved prediction cohort; stale bundle rejected")
    for name, record in core["artifacts"].items():
        if (not isinstance(record.get("sha256"), str) or not _HEX.fullmatch(record["sha256"])
                or not isinstance(record.get("bytes"), int) or record["bytes"] < 0):
            raise ValueError(f"Invalid stored artifact fingerprint: {name}")
    for name in ("gpu_stage_fingerprint", "preprocessing_fingerprint"):
        value = core["provenance"].get(name)
        if not isinstance(value, str) or not _HEX.fullmatch(value):
            raise ValueError(f"Missing or invalid saved source fingerprint: {name}")
    if expected_artifact_paths is not None:
        if set(expected_artifact_paths) != set(core["artifacts"]):
            raise ValueError("Expected artifact inventory differs")
        for name, artifact in expected_artifact_paths.items():
            actual = _file_record(artifact)
            if any(actual[key] != core["artifacts"][name][key] for key in ("bytes", "sha256")):
                raise ValueError(f"Current artifact differs from saved provenance: {name}")
    score_names = core["score_branches"]
    probability_names = core["probability_families"]
    if (len(score_names) != len(set(score_names))
            or not set(REQUIRED_FRAMES).issubset(score_names)
            or any(not isinstance(name, str) or not _NAME.fullmatch(name) for name in score_names)
            or len(probability_names) != 2
            or set(probability_names) != set(REQUIRED_PROBABILITY_FAMILIES)):
        raise ValueError("Prediction branch inventory changed")
    expected_keys = {"study_ids", "labels", "manifest_core_json",
                     *("score__" + name for name in score_names),
                     *("probability__" + name for name in probability_names)}
    import math
    import zipfile
    with zipfile.ZipFile(archive_path) as zipped:
        members = zipped.infolist()
        if (sum(item.file_size for item in members) > _MAX_ARCHIVE_BYTES
                or len(members) != len(expected_keys)
                or {item.filename for item in members} != {key + ".npy" for key in expected_keys}):
            raise ValueError("Prediction archive content size/inventory is invalid")
        # A forged .npy header can advertise billions of entries despite a tiny
        # compressed member. Check the actual header before np.load allocates.
        for member in members:
            key = member.filename[:-4]
            if key == "study_ids":
                wanted = np.asarray(ids, dtype=np.str_)
                wanted_shape, wanted_dtype = wanted.shape, wanted.dtype
            elif key == "labels":
                wanted = np.asarray(labels, dtype=np.str_)
                wanted_shape, wanted_dtype = wanted.shape, wanted.dtype
            elif key == "manifest_core_json":
                wanted_shape, wanted_dtype = (len(_canonical(core)),), np.dtype(np.uint8)
            else:
                wanted_shape, wanted_dtype = (len(ids), len(labels)), np.dtype(np.float64)
            with zipped.open(member) as handle:
                version = np.lib.format.read_magic(handle)
                if version == (1, 0):
                    shape, _, dtype = np.lib.format.read_array_header_1_0(handle)
                elif version == (2, 0):
                    shape, _, dtype = np.lib.format.read_array_header_2_0(handle)
                else:
                    raise ValueError("Unsupported NPY member format")
                if (shape != wanted_shape or dtype != wanted_dtype
                        or member.file_size != handle.tell() + math.prod(shape) * dtype.itemsize):
                    raise ValueError(f"Prediction array header shape/dtype/size mismatch: {key}")
    scores, probabilities = {}, {}
    with np.load(archive_path, allow_pickle=False) as archive:
        if set(archive.files) != expected_keys:
            raise ValueError("Prediction archive array inventory changed")
        encoded = archive["manifest_core_json"]
        if encoded.dtype != np.uint8 or encoded.ndim != 1 or encoded.tobytes() != _canonical(core):
            raise ValueError("Embedded and sidecar manifests differ")
        for name in expected_keys - {"manifest_core_json"}:
            array = archive[name]
            if name not in core["arrays"] or _array_record(array) != core["arrays"][name]:
                raise ValueError(f"Prediction array fingerprint mismatch: {name}")
            if name == "study_ids":
                if array.ndim != 1 or array.dtype.kind != "U" or array.tolist() != ids:
                    raise ValueError("Archive study identity differs")
            elif name == "labels":
                if array.ndim != 1 or array.dtype.kind != "U" or array.tolist() != labels:
                    raise ValueError("Archive label identity differs")
            else:
                if (array.dtype != np.float64 or array.shape != (len(ids), len(labels))
                        or not np.isfinite(array).all() or np.any((array < 0) | (array > 1))):
                    raise ValueError(f"Invalid prediction score/probability array: {name}")
                # frombuffer(bytes) prevents callers from re-enabling WRITEABLE.
                immutable = np.frombuffer(array.tobytes(), dtype=np.float64).reshape(array.shape)
                kind, branch = name.split("__", 1)
                (scores if kind == "score" else probabilities)[branch] = immutable
    return PredictionBundle(tuple(ids), tuple(labels), MappingProxyType(scores),
                            MappingProxyType(probabilities), _freeze(envelope), archive_path)


def read_probability_family(
    raw_path: str | Path, companion_csv: str | Path, *, study_ids: Sequence[str],
    labels: Sequence[str], expected_members: int, source_contract: str,
) -> tuple[pd.DataFrame, dict]:
    """Read public raw-NPZ means; never silently substitute a rank matrix.

    Preserve a stored probability_mean exactly, including source float32
    rounding. If it is absent, convert raw_probabilities to float64 and mean
    over the validated [member, study, finding] axis. For released NPZs lacking
    a labels key, source_contract must identify the independently verified
    runtime whose label order is recorded by the companion CSV. This function
    does not itself authenticate that runtime; the caller's inference gate must.
    """
    ids, labels = _identities(study_ids, labels)
    if (isinstance(expected_members, bool) or not isinstance(expected_members, int)
            or expected_members < 1 or not isinstance(source_contract, str)
            or not source_contract.strip()):
        raise ValueError("Positive member count and verified source_contract description required")
    raw_path, companion_csv = Path(raw_path), Path(companion_csv)
    raw_record, companion_record = _file_record(raw_path), _file_record(companion_csv)
    companion = _align(_csv(companion_csv), ids, "raw-probability companion")
    consistency = {"status": "not_applicable_no_stored_mean_and_members_pair"}
    companion_rank_verified = False
    with np.load(raw_path, allow_pickle=False) as archive:
        if "study_uids" not in archive.files:
            raise ValueError("Raw probability archive lacks study_uids")
        raw_ids = archive["study_uids"]
        if raw_ids.ndim != 1 or raw_ids.dtype.kind not in {"U", "S"}:
            raise ValueError("Raw study_uids must be a one-dimensional string array")
        actual_ids, _ = _identities(raw_ids.astype(str).tolist(), labels)
        if set(actual_ids) != set(ids):
            raise ValueError("Raw probability archive cohort mismatch")
        present_labels = [key for key in ("labels", "targets") if key in archive.files]
        for key in present_labels:
            if archive[key].ndim != 1 or archive[key].astype(str).tolist() != labels:
                raise ValueError("Raw probability label order mismatch")
        # Validate a member array whenever present, even if its stored mean is
        # preferred, so a wrong checkpoint count cannot hide behind that mean.
        raw = source_raw = None
        if "raw_probabilities" in archive.files:
            source_raw = archive["raw_probabilities"]
            if source_raw.dtype.kind != "f":
                raise ValueError("Raw probabilities must have floating-point dtype")
            raw = np.asarray(source_raw, dtype=np.float64)
            if (raw.shape != (expected_members, len(ids), len(labels))
                    or not np.isfinite(raw).all() or np.any((raw < 0) | (raw > 1))):
                raise ValueError("Raw probabilities have wrong member axis/shape or invalid values")
        if "probability_mean" in archive.files:
            stored_mean = archive["probability_mean"]
            if stored_mean.dtype.kind != "f" or stored_mean.shape != (len(ids), len(labels)):
                raise ValueError("Stored probability mean has invalid shape/dtype")
            values = np.asarray(stored_mean, dtype=np.float64)
            reduction = "stored_probability_mean"
            if raw is not None:
                mean64 = raw.mean(axis=0)
                native_mean = source_raw.mean(axis=0)
                exact64 = np.array_equal(stored_mean, mean64.astype(stored_mean.dtype))
                exact_native = np.array_equal(stored_mean, native_mean.astype(stored_mean.dtype))
                error64 = float(np.max(np.abs(values - mean64)))
                error_native = float(np.max(np.abs(values - native_mean.astype(np.float64))))
                tolerance = float(4 * np.finfo(np.float32).eps)
                if not (exact64 or exact_native or min(error64, error_native) <= tolerance):
                    raise ValueError("Stored probability mean disagrees with raw members beyond float32 rounding")
                consistency = {
                    "status": "exact_float64_mean_cast" if exact64 else
                              "exact_source_dtype_mean_cast" if exact_native else "within_float32_rounding_tolerance",
                    "raw_dtype": source_raw.dtype.str, "stored_mean_dtype": stored_mean.dtype.str,
                    "exact_float64_mean_cast": bool(exact64), "exact_source_dtype_mean_cast": bool(exact_native),
                    "max_abs_error_float64_mean": error64, "max_abs_error_native_mean": error_native,
                    "absolute_tolerance": tolerance, "relative_tolerance": 0.0,
                }
        elif raw is not None:
            values = raw.mean(axis=0)
            reduction = "float64_raw_member_mean_axis0"
        else:
            raise ValueError("Raw archive lacks probability_mean and raw_probabilities; rank fallback forbidden")
        if "submission_percentile_rank" in archive.files:
            ranks = np.asarray(archive["submission_percentile_rank"], dtype=np.float64)
            if ranks.shape != (len(ids), len(labels)):
                raise ValueError("Stored submission percentile-rank shape mismatch")
            rank_frame = pd.DataFrame(ranks, columns=labels)
            rank_frame.insert(0, ID_COLUMN, actual_ids)
            rank_frame = _align(rank_frame, ids, "stored submission percentile ranks")
            if not np.array_equal(rank_frame[labels].to_numpy(), companion[labels].to_numpy()):
                raise ValueError("Raw archive submission ranks disagree with companion CSV")
            companion_rank_verified = True
    if values.shape != (len(ids), len(labels)):
        raise ValueError("Raw probability mean shape mismatch")
    frame = pd.DataFrame(values, columns=labels)
    frame.insert(0, ID_COLUMN, actual_ids)
    frame = _align(frame, ids, "raw probability mean")
    if (sha256_file(raw_path) != raw_record["sha256"]
            or sha256_file(companion_csv) != companion_record["sha256"]):
        raise RuntimeError("Raw probability inputs changed while being read")
    return frame, {"raw_archive": raw_record, "companion_csv": companion_record,
                   "reduction": reduction, "expected_members": expected_members,
                   "source_contract": source_contract,
                   "mean_consistency": consistency,
                   "stored_submission_ranks_match_companion": companion_rank_verified,
                   "label_order_evidence": "npz_and_companion" if present_labels else "verified_runtime_and_companion"}


def replay_candidates(bundle: PredictionBundle, *, validated_recipe=None) -> dict[str, pd.DataFrame]:
    """Return at most four fixed candidates, plus an explicitly validated recipe.

    No winner is selected and no labels, study reports, or leaderboard values
    are read. The probability-mean candidate keeps the full public Raptor and
    existing outer blend. Optional already-computed ablations are copied from
    the bundle. No grid, automatic search, or data-dependent weight is used.
    """
    if not isinstance(bundle, PredictionBundle):
        raise TypeError("Use load_prediction_bundle to verify an actual prediction bundle first")
    ids, labels = list(bundle.study_ids), list(bundle.labels)
    # Immutable source arrays are never passed as writable DataFrame views.
    result = {"reference": bundle.frame("reference")}
    transformer_rank = bundle.frame("transformer")[labels].rank(method="average", pct=True)
    public_rank = bundle.frame("raptor_public")[labels].rank(method="average", pct=True)
    residual = bundle.frame("resgated_top3", probability=True)[labels]
    d4 = bundle.frame("d4_swa3", probability=True)[labels]
    family_rank = ((residual + d4) / 2.0).rank(method="average", pct=True)
    # Do not rerank hybrid here: an extra rank changes the public outer blend.
    hybrid = .60 * public_rank + .40 * family_rank
    combined = (1.0 - np.asarray(OUTER_WEIGHTS)) * transformer_rank + np.asarray(OUTER_WEIGHTS) * hybrid
    candidate = combined.rank(method="average", pct=True)
    candidate.insert(0, ID_COLUMN, ids)
    result["reference_probability_family"] = _align(candidate, ids, "probability-family candidate")
    for name in FIXED_CANDIDATES[2:]:
        if name in bundle.scores:
            result[name] = bundle.frame(name)
    if validated_recipe is not None:
        # The existing utility enforces same-finding weights, expert-only truth,
        # entire-cohort model exclusion, an untouched release gate, exact model
        # coverage, and unchanged reference for rejected findings. No callback
        # supplied by a caller can bypass those checks in this module.
        from validation_tools import apply_recipe
        available = {name: bundle.frame(name) for name in bundle.scores}
        available.update(result)
        result["validated_recipe"] = _align(
            apply_recipe(available, validated_recipe, ids), ids, "validated recipe")
    return result
'''
import importlib.util as _fixed_importlib
import sys as _fixed_sys
from pathlib import Path
_fixed_module_path = Path(FIXED_RUN['output_root']) / 'fixed_compute_replay.py'
_fixed_module_path.parent.mkdir(parents=True, exist_ok=True)
_fixed_module_path.write_text(REPLAY_MODULE_SOURCE)
_fixed_spec = _fixed_importlib.spec_from_file_location('fixed_compute_replay', _fixed_module_path)
_fixed_module = _fixed_importlib.module_from_spec(_fixed_spec)
_fixed_sys.modules[_fixed_spec.name] = _fixed_module
_fixed_spec.loader.exec_module(_fixed_module)


In [ ]:
import fixed_compute_replay as replay
import shutil
if FIXED_RUN['mode'] == 'inference':
    if len(EXECUTED_REFERENCE_STAGES) != 22 or _DINOV2_MATCHED_MEMBERS != 20:
        raise RuntimeError('Reference model stages are incomplete')
    if not globals().get('V18_CALIBRATOR_APPLIED'):
        raise RuntimeError('The original transformer branch is incomplete')
    delivery_require_full_ensemble(_RSNA_AUDIT, '/kaggle/working', _RSNA_TEST_IDS, _ke_input_ids)
    ALL_REFERENCE_FRAMES = delivery_export_candidates('/kaggle/working', _RSNA_TEST_IDS, _coatnet_weight)
    # Retain only the frozen experiment sources; no arbitrary weight grid.
    BUNDLE_FRAMES = {name: ALL_REFERENCE_FRAMES[name] for name in (
        'reference', 'transformer', 'raptor_public',
        'reference_no_calibrator', 'reference_rad_fracture')}
    PROBABILITY_FAMILIES, RAW_EVIDENCE = {}, {}
    for name, raw, csv, members, contract in (
        ('resgated_top3', '/kaggle/working/coat_resgated_ep10_top3_predictions.npz',
         '/kaggle/working/_coat_arm.csv', 3,
         'Pinned residual runtime b11e58f8d7cabe9e264ac70b01e811dc6a846aa01248ace1f0e6536d0d4094d9; verified receipt models=3 epochs=[4,6,8]'),
        ('d4_swa3', '/kaggle/working/d4_input/coatnet_d4_depthzone_swa3_predictions.npz',
         '/kaggle/working/d4_input/d4.csv', 1,
         'Pinned D4 wrapper validates one SWA model, exact raw mean, ranks and CSV hash'),
    ):
        frame, evidence = replay.read_probability_family(
            raw, csv, study_ids=_RSNA_TEST_IDS, labels=RSNA_TARGETS,
            expected_members=members, source_contract=contract)
        PROBABILITY_FAMILIES[name], RAW_EVIDENCE[name] = frame, evidence
    delivery_json(FIXED_OUTPUT / 'raw_probability_checks.json', RAW_EVIDENCE)
    delivery_json(FIXED_OUTPUT / 'reference_stage_execution.json', {
        'reference_budget': FIXED_BUDGET, 'executed_stages': EXECUTED_REFERENCE_STAGES,
        'extra_model_forward_passes': 0, 'runtime_equality_measured': False,
    })
    delivery_json(FIXED_OUTPUT / 'runtime_audit.json', _RSNA_AUDIT)
    metadata_copy = FIXED_OUTPUT / 'metadata'
    metadata_copy.mkdir()
    for name, path in metadata_paths(FIXED_METADATA_ROOT).items():
        shutil.copyfile(path, metadata_copy / name)
    _artifact_paths = {
        'stage_execution': FIXED_OUTPUT / 'reference_stage_execution.json',
        'runtime_audit': FIXED_OUTPUT / 'runtime_audit.json',
        'probability_checks': FIXED_OUTPUT / 'raw_probability_checks.json',
        'residual_receipt': Path('/kaggle/working/_coat_arm_receipt.json'),
        'd4_receipt': Path('/kaggle/working/d4_input/d4.receipt.json'),
        'family_receipt': Path('/kaggle/working/_coat_raptor_blend_receipt.json'),
    }
    for name, path in list(_artifact_paths.items()):
        if path.parent != FIXED_OUTPUT:
            destination = FIXED_OUTPUT / (name + '.json')
            shutil.copyfile(path, destination)
            _artifact_paths[name] = destination
    _bundle_path = FIXED_OUTPUT / 'predictions.npz'
    replay.export_prediction_bundle(
        _bundle_path, study_ids=_RSNA_TEST_IDS, labels=RSNA_TARGETS,
        frames=BUNDLE_FRAMES, probability_families=PROBABILITY_FAMILIES,
        metadata_paths=metadata_paths(metadata_copy), artifact_paths=_artifact_paths,
        provenance={
            'gpu_stage_fingerprint': FIXED_BUDGET['ordered_cell_hashes_sha256'],
            'preprocessing_fingerprint': FIXED_BUDGET['ordered_cell_hashes_sha256'],
            'preprocessing_fingerprint_scope': 'all pinned source stages include their original preprocessing',
            'base_notebook_sha256': FIXED_BUDGET['reference_notebook_sha256'],
            'extra_model_forward_passes': 0, 'model_training_overlap': 'unknown',
            'auc_measured': None, 'raw_probability_checks': RAW_EVIDENCE,
        }, run_id=FIXED_OUTPUT.name, cohort_role='test')
else:
    _bundle_path = Path(FIXED_RUN['replay_bundle'])
    _RSNA_TEST_IDS = pd.read_csv(FIXED_METADATA_ROOT / 'test.csv',
                               dtype={RSNA_ID: str})[RSNA_ID].tolist()

PREDICTION_BUNDLE = replay.load_prediction_bundle(
    _bundle_path, expected_study_ids=_RSNA_TEST_IDS, expected_labels=RSNA_TARGETS,
    expected_metadata_paths=metadata_paths(FIXED_METADATA_ROOT),
    expected_cohort_role='test', purpose='same_cohort')
if PREDICTION_BUNDLE.manifest['core']['provenance']['gpu_stage_fingerprint'] != FIXED_BUDGET['ordered_cell_hashes_sha256']:
    raise RuntimeError('The saved bundle belongs to a different model-stage source contract')
CANDIDATE_FRAMES = replay.replay_candidates(PREDICTION_BUNDLE)
for name, frame in CANDIDATE_FRAMES.items():
    frame.to_csv(FIXED_OUTPUT / (name + '.csv'), index=False)
delivery_json(FIXED_OUTPUT / 'candidate_differences.json',
              candidate_differences(CANDIDATE_FRAMES, RSNA_TARGETS))
print('Computed candidates:', list(CANDIDATE_FRAMES))
print('AUC: not measured. No labels or score-based candidate selection were used.')


## 4. Optional independently validated recipe

The CPU validation tools below can evaluate already saved predictions with
expert labels and explicit training/selection exclusion records. They do not
run automatically or launch models. A supplied recipe is accepted only if its
declared held-out gate passes, required candidate sources exist, and labels,
cohort identity and weights satisfy the contract. Hashes do not authenticate
truthfulness of a provenance declaration.

The inspected DINO/Rad OOF receipts lack matching full-ensemble validation
outputs. E13 chooses checkpoints using its validation-fold AUC; its resulting
OOF is not an untouched final evaluation. Do not label those diagnostics as
independent evidence for a 0.990 final ensemble. See the accompanying plan.


In [ ]:
VALIDATION_MODULE_SOURCE = r'''"""Patient-disjoint validation and conservative blending for RSNA Knee 2026.

Official metric: mean of the 12 per-finding ROC AUCs. Missing training labels
are masked here; this is a validation choice, not a claim that the organizer's
hidden labels are missing. Never replace an unknown label with a negative.

This utility uses ONLY a labeled development cohort. It does not read reports,
leaderboard scores, test labels, or test identifiers. Before interpreting its
report as independent evidence, every underlying model (including any report
labeler, supervised preprocessor, or calibration stage) must have been trained
without the ENTIRE development cohort. Ordinary precomputed OOF predictions
are useful for exploratory stacking, but splitting those predictions again
does not make their base-model training independent of the new outer split.
An OOF bundle is therefore blocked by default and cannot export a deployable
calibration manifest. See the example manifest and run_validation_bundle().

Dependencies: numpy, pandas, scipy, scikit-learn. Runs on CPU.
Metric reference: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection
Data reference: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data
"""

from __future__ import annotations

import argparse
import csv
from dataclasses import dataclass
import hashlib
import json
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.model_selection import GroupKFold, GroupShuffleSplit


TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]


def sha256(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _check_y(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float64)
    if y.ndim != 2 or not y.shape[0] or not y.shape[1]:
        raise ValueError("Labels must be nonempty [study, finding].")
    if np.isinf(y).any() or np.any(np.isfinite(y) & ~np.isin(y, [-1, 0, 1])):
        raise ValueError("Labels must be 0, 1, NaN, or -1 (unknown).")
    return np.where(y == -1, np.nan, y)


def _check_predictions(p: np.ndarray, shape: tuple | None = None) -> np.ndarray:
    p = np.asarray(p, dtype=np.float64)
    if shape is not None and p.shape != shape:
        raise ValueError(f"Prediction shape {p.shape} != {shape}.")
    if not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError("Predictions must be finite probabilities/scores in [0, 1].")
    return p


def _groups(groups: Sequence, n: int) -> np.ndarray:
    s = pd.Series(groups)
    if len(s) != n or s.isna().any() or s.astype(str).str.strip().eq("").any():
        raise ValueError("Every study needs a nonempty, stable patient/group identifier.")
    return s.astype(str).to_numpy()


@dataclass
class _AUCCache:
    """Pre-sort once; cluster bootstrap multiplies study weights, not array size."""

    indices: np.ndarray
    starts: np.ndarray
    positives: np.ndarray

    @classmethod
    def build(cls, y: np.ndarray, p: np.ndarray) -> "_AUCCache":
        indices = np.flatnonzero(np.isfinite(y))
        indices = indices[np.argsort(p[indices], kind="stable")]
        if len(indices):
            starts = np.r_[0, np.flatnonzero(np.diff(p[indices]) != 0) + 1]
        else:
            starts = np.array([], dtype=int)
        return cls(indices, starts, y[indices] == 1)

    def auc(self, weights: np.ndarray) -> float:
        if not len(self.indices):
            return float("nan")
        w = weights[self.indices]
        positives = np.add.reduceat(w * self.positives, self.starts)
        negatives = np.add.reduceat(w * ~self.positives, self.starts)
        denominator = positives.sum() * negatives.sum()
        if denominator <= 0:
            return float("nan")
        # Identical prediction scores contribute half of their tied pairs.
        numerator = np.sum(positives * (np.cumsum(negatives) - .5 * negatives))
        return float(numerator / denominator)


def masked_macro_auc(y, p, labels: Sequence[str] | None = None) -> dict:
    """Return per-class counts/AUC and full macro only if every class is evaluable.

    supported_macro_auc is diagnostic when a class lacks positives or negatives;
    it must not be presented as the official 12-class metric in that case.
    """
    y = _check_y(y)
    p = _check_predictions(p, y.shape)
    labels = list(labels or TARGETS)
    if len(labels) != y.shape[1] or len(set(labels)) != len(labels):
        raise ValueError("Label names must be unique and match the matrix width.")
    per_class, aucs = {}, []
    for j, label in enumerate(labels):
        score = _AUCCache.build(y[:, j], p[:, j]).auc(np.ones(len(y)))
        known = np.isfinite(y[:, j])
        per_class[label] = {
            "auc": float(score) if np.isfinite(score) else None,
            "observed": int(known.sum()),
            "positive": int(np.sum(y[:, j] == 1)),
            "negative": int(np.sum(y[:, j] == 0)),
            "unknown": int((~known).sum()),
        }
        aucs.append(score)
    finite = np.isfinite(aucs)
    return {
        "macro_auc": float(np.mean(aucs)) if finite.all() else None,
        "supported_macro_auc": float(np.nanmean(aucs)) if finite.any() else None,
        "evaluable_classes": int(finite.sum()),
        "required_classes": len(labels),
        "per_class": per_class,
    }


def compare_cluster_bootstrap(
    y, baseline, candidate, groups, *, labels=None, n_bootstrap=2000,
    seed=2026, confidence=.95, minimum_valid_fraction=.90,
) -> dict:
    """Paired patient-cluster bootstrap of candidate minus baseline AUC.

    A replicate samples patients with replacement and retains every study of
    each sampled patient. Both predictions use exactly the same samples.
    Macro replicates require ALL original target classes to remain evaluable;
    they never average a changing subset of classes. Percentile intervals are
    approximate uncertainty summaries conditional on the supplied fixed
    predictions, not a distribution-free guarantee. When predictions contain
    cohort ranks, those ranks are not refitted inside each bootstrap replicate.
    """
    y = _check_y(y)
    a = _check_predictions(baseline, y.shape)
    b = _check_predictions(candidate, y.shape)
    labels = list(labels or TARGETS)
    if len(labels) != y.shape[1] or len(set(labels)) != len(labels):
        raise ValueError("Label names must be unique and match the matrix width.")
    group_ids = _groups(groups, len(y))
    unique, inverse = np.unique(group_ids, return_inverse=True)
    if len(unique) < 2 or n_bootstrap < 100:
        raise ValueError("Need at least two groups and at least 100 bootstrap replicates.")
    if not 0 < confidence < 1 or not 0 < minimum_valid_fraction <= 1:
        raise ValueError("Invalid confidence or minimum_valid_fraction.")
    a_cache = [_AUCCache.build(y[:, j], a[:, j]) for j in range(y.shape[1])]
    b_cache = [_AUCCache.build(y[:, j], b[:, j]) for j in range(y.shape[1])]
    rng = np.random.default_rng(seed)
    delta = np.full((n_bootstrap, y.shape[1]), np.nan)
    for k in range(n_bootstrap):
        multiplicity = rng.multinomial(len(unique), np.full(len(unique), 1 / len(unique)))
        weights = multiplicity[inverse].astype(float)
        delta[k] = [cb.auc(weights) - ca.auc(weights) for ca, cb in zip(a_cache, b_cache)]
    point_a = masked_macro_auc(y, a, labels)
    point_b = masked_macro_auc(y, b, labels)
    alpha = 1 - confidence

    def summarize(samples, point):
        valid = samples[np.isfinite(samples)]
        sufficient = len(valid) / n_bootstrap >= minimum_valid_fraction
        return {
            "delta_auc": point,
            "valid_replicates": int(len(valid)),
            "valid_fraction": float(len(valid) / n_bootstrap),
            "interval_reliable_support": bool(sufficient),
            "ci_low": float(np.quantile(valid, alpha / 2)) if sufficient else None,
            "ci_high": float(np.quantile(valid, 1 - alpha / 2)) if sufficient else None,
            "one_sided_lower": float(np.quantile(valid, alpha)) if sufficient else None,
            "bootstrap_fraction_nonpositive": float(np.mean(valid <= 0)) if sufficient else None,
        }

    per_class = {}
    for j, label in enumerate(labels):
        pa, pb = point_a["per_class"][label]["auc"], point_b["per_class"][label]["auc"]
        per_class[label] = summarize(delta[:, j], None if pa is None or pb is None else pb - pa)
    complete = np.isfinite(delta).all(axis=1)
    macro_samples = np.full(n_bootstrap, np.nan)
    macro_samples[complete] = delta[complete].mean(axis=1)
    pa, pb = point_a["macro_auc"], point_b["macro_auc"]
    return {
        "confidence": confidence, "n_bootstrap": int(n_bootstrap),
        "patient_groups": int(len(unique)), "studies": len(y),
        "baseline": point_a, "candidate": point_b,
        "macro": summarize(macro_samples, None if pa is None or pb is None else pb - pa),
        "per_class": per_class,
    }


@dataclass
class ValidationBundle:
    ids: np.ndarray
    y: np.ndarray
    predictions: np.ndarray  # [study, model, finding]
    groups: np.ndarray
    folds: np.ndarray | None
    labels: list[str]
    model_names: list[str]
    independent_cohort: bool
    expert_truth: bool
    provenance: dict


def _read_frame(path: Path, string_columns: Sequence[str]) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        with path.open(newline="", encoding="utf-8-sig") as handle:
            header = next(csv.reader(handle), [])
        if len(header) != len(set(header)):
            raise ValueError(f"Duplicate CSV header columns in {path}.")
        # Explicit string dtype prevents irreversible conversion of e.g. 00123.
        frame = pd.read_csv(path, dtype={column: "string" for column in string_columns})
    elif path.suffix.lower() in {".parquet", ".pq"}:
        frame = pd.read_parquet(path)
        for column in string_columns:
            if column in frame:
                frame[column] = frame[column].astype("string")
    else:
        raise ValueError("Prediction and label tables must be CSV or Parquet.")
    if frame.columns.duplicated().any():
        raise ValueError(f"Duplicate columns in {path}.")
    return frame


def load_validation_bundle(manifest_path: str | Path) -> ValidationBundle:
    """Strictly align studies and audit DECLARED training-group provenance.

    Required manifest fields: schema_version=1, truth_csv, id_column,
    group_column, labels, models. Each model needs name, prediction_csv,
    prediction_role, training_groups_complete=true, and training_groups for
    heldout predictions OR training_groups_by_fold for out_of_fold predictions.
    OOF mode additionally needs fold_column in manifest and every data table.
    truth_label_source must be expert_manual for a deployable recipe; values
    report_derived, mixed, or unknown permit descriptive analysis only. Supply
    expert-only labels and mask all other cells to NaN to evaluate expert truth.
    The declarations must include all groups that influenced training, weak
    labeling, tuning, and learned preprocessing. The code cannot verify that a
    declared training ledger truthfully describes the actual checkpoint.
    """
    manifest_path = Path(manifest_path).resolve()
    spec = json.loads(manifest_path.read_text())
    if spec.get("schema_version") != 1:
        raise ValueError("Unsupported validation manifest schema_version.")
    labels = spec.get("labels", TARGETS)
    if labels != TARGETS:
        raise ValueError("Manifest labels/order must exactly match the 12 competition labels.")
    truth_source = spec.get("truth_label_source", "unknown")
    if truth_source not in {"expert_manual", "report_derived", "mixed", "unknown"}:
        raise ValueError("Unsupported truth_label_source; use expert_manual, report_derived, mixed, or unknown.")
    id_col, group_col = spec["id_column"], spec["group_column"]
    fold_col = spec.get("fold_column")
    if group_col == id_col:
        raise ValueError("Use verified patient groups, not StudyInstanceUID as a patient proxy.")

    def locate(value):
        path = Path(value)
        return path if path.is_absolute() else manifest_path.parent / path

    def training_group_set(values, name):
        if not isinstance(values, list) or any(value is None for value in values):
            raise ValueError(f"{name}: training-group ledger entries must be lists of group IDs.")
        result = {str(value) for value in values}
        if any(not value.strip() or value.startswith("REPLACE_WITH_") for value in result):
            raise ValueError(f"{name}: replace every training-ledger placeholder with actual group IDs.")
        return result

    truth_path = locate(spec["truth_csv"])
    truth = _read_frame(truth_path, [id_col, group_col, *([fold_col] if fold_col else [])])
    required = [id_col, group_col, *labels, *([fold_col] if fold_col else [])]
    if not set(required).issubset(truth.columns):
        raise ValueError(f"Truth table missing {sorted(set(required) - set(truth.columns))}.")
    if truth[id_col].isna().any() or truth[id_col].str.strip().eq("").any() or truth[id_col].duplicated().any():
        raise ValueError("Truth StudyInstanceUID values must be nonempty and unique.")
    ids = truth[id_col].astype(str).to_numpy()
    groups = _groups(truth[group_col], len(truth))
    y = _check_y(truth[labels].to_numpy(dtype=float, na_value=np.nan))
    folds = None
    if fold_col:
        if truth[fold_col].isna().any():
            raise ValueError("Missing truth fold ID.")
        folds = truth[fold_col].astype(str).to_numpy()
        if pd.DataFrame({"group": groups, "fold": folds}).groupby("group").fold.nunique().gt(1).any():
            raise ValueError("A patient appears in more than one base-model OOF fold.")
    models = spec.get("models", [])
    names = [m["name"] for m in models]
    if not names or len(names) != len(set(names)):
        raise ValueError("At least one uniquely named model is required.")
    arrays, records, independent = [], [], True
    for model in models:
        name = model["name"]
        if model.get("training_groups_complete") is not True:
            raise ValueError(f"{name}: a complete declared training-group ledger is required.")
        path = locate(model["prediction_csv"])
        pred = _read_frame(path, [id_col, *([fold_col] if fold_col else [])])
        if not {id_col, *labels}.issubset(pred.columns):
            raise ValueError(f"{name}: missing identity or target columns.")
        if pred[id_col].isna().any() or pred[id_col].duplicated().any():
            raise ValueError(f"{name}: missing or duplicate study identifier.")
        pred[id_col] = pred[id_col].astype(str)
        if set(pred[id_col]) != set(ids):
            raise ValueError(f"{name}: prediction IDs do not exactly match the validation cohort.")
        pred = pred.set_index(id_col).loc[ids]
        arrays.append(_check_predictions(pred[labels].to_numpy(float), y.shape))
        role = model["prediction_role"]
        if role == "heldout":
            training = training_group_set(model["training_groups"], name)
            if training.intersection(groups):
                raise ValueError(f"{name}: declared training groups overlap the held-out cohort.")
        elif role == "out_of_fold":
            independent = False
            if folds is None or fold_col not in pred or pred[fold_col].isna().any():
                raise ValueError(f"{name}: OOF predictions require an explicit per-study fold.")
            if not np.array_equal(pred[fold_col].astype(str).to_numpy(), folds):
                raise ValueError(f"{name}: prediction folds do not match the truth ledger.")
            ledger = model["training_groups_by_fold"]
            if set(ledger) != set(folds):
                raise ValueError(f"{name}: training ledger must cover every OOF fold exactly.")
            for fold in np.unique(folds):
                training = training_group_set(ledger[fold], name)
                if training.intersection(groups[folds == fold]):
                    raise ValueError(f"{name}: patient leakage in OOF fold {fold}.")
        else:
            raise ValueError(f"{name}: prediction_role must be heldout or out_of_fold.")
        records.append({"name": name, "prediction_role": role, "sha256": sha256(path)})
    provenance = {
        "manifest_sha256": sha256(manifest_path), "truth_sha256": sha256(truth_path),
        "models": records, "training_provenance_is_user_declared": True,
        "all_base_models_exclude_entire_cohort": independent,
        "truth_label_source": truth_source,
    }
    return ValidationBundle(ids, y, np.stack(arrays, axis=1), groups, folds,
                            list(labels), names, independent, truth_source == "expert_manual", provenance)


def blend_predictions(predictions, weights) -> np.ndarray:
    """Apply identity-space, nonnegative model-by-finding weights summing to 1."""
    p = _check_predictions(predictions)
    if p.ndim != 3 or not all(p.shape):
        raise ValueError("Predictions must be nonempty [study, model, finding].")
    w = np.asarray(weights, dtype=float)
    if w.shape != p.shape[1:] or not np.isfinite(w).all() or (w < 0).any():
        raise ValueError("Weights must be finite, nonnegative [model, finding].")
    if not np.allclose(w.sum(axis=0), 1, rtol=0, atol=1e-10):
        raise ValueError("Weights must sum to one per finding.")
    return np.clip(np.einsum("nmc,mc->nc", p, w), 0, 1)


def cohort_rank01(predictions) -> np.ndarray:
    """Rank each model/finding across THIS cohort; tied scores share a rank.

    Validation ranks are recomputed within each fit/inner/gate/report cohort.
    Inference ranks are computed once across the entire aligned test cohort.
    Do not apply this function independently to inference batches/shards.
    """
    p = _check_predictions(predictions)
    if p.ndim not in (2, 3) or not all(p.shape):
        raise ValueError("Expected nonempty [study, finding] or [study, model, finding].")
    if len(p) == 1:
        return np.full_like(p, .5)
    flat = p.reshape(len(p), -1)
    ranks = pd.DataFrame(flat).rank(axis=0, method="average").to_numpy()
    return ((ranks - 1) / (len(p) - 1)).reshape(p.shape)


def _transform_predictions(predictions, mode):
    if mode == "cohort_rank01":
        return cohort_rank01(predictions)
    if mode == "identity":
        return _check_predictions(predictions)
    raise ValueError("prediction_transform must be identity or cohort_rank01.")


def _fit_pairwise_weights(y, x, prior, *, seed, ridge=.02, max_pairs=20000):
    """Convex pairwise logistic surrogate with simplex constraint and shrinkage."""
    positives, negatives = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if len(positives) < 2 or len(negatives) < 2 or x.shape[1] == 1:
        return prior.copy()
    rng = np.random.default_rng(seed)
    total_pairs = len(positives) * len(negatives)
    if total_pairs <= max_pairs:
        pi, ni = np.meshgrid(positives, negatives, indexing="ij")
        diff = x[pi.ravel()] - x[ni.ravel()]
    else:
        diff = x[rng.choice(positives, max_pairs)] - x[rng.choice(negatives, max_pairs)]
    scale = 8.0  # fixed before validation, never tuned on the gate/report cohorts

    def objective(w):
        z = scale * (diff @ w)
        loss = np.logaddexp(0, -z).mean() + ridge * np.square(w - prior).sum()
        grad = -scale * np.mean(expit(-z)[:, None] * diff, axis=0) + 2 * ridge * (w - prior)
        return float(loss), grad

    result = minimize(objective, prior, jac=True, method="SLSQP",
                      bounds=[(0, 1)] * len(prior),
                      constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1,
                                    "jac": lambda w: np.ones_like(w)}],
                      options={"ftol": 1e-9, "maxiter": 200})
    if not result.success or not np.isfinite(result.x).all():
        return prior.copy()
    w = np.maximum(result.x, 0)
    return w / w.sum()


def _select_weights(y, p, groups, prior, *, seed, inner_folds=3, min_class_count=10,
                    prediction_transform="cohort_rank01"):
    """Only fit-cohort labels choose class weights and shrinkage strength."""
    n_groups = len(np.unique(groups))
    n_splits = min(inner_folds, n_groups)
    if n_splits < 2:
        return prior.copy(), []
    # GroupKFold partitions whole patients; no studies cross a validation fold.
    folds = list(GroupKFold(n_splits=n_splits).split(np.zeros(len(y)), groups=groups))
    transformed_folds = [(_transform_predictions(p[tr], prediction_transform),
                          _transform_predictions(p[va], prediction_transform)) for tr, va in folds]
    full_fit = _transform_predictions(p, prediction_transform)
    strengths = np.array([0., .25, .50, .75])  # >=25% prior weight always retained
    chosen = prior.copy()
    details = []
    for j in range(y.shape[1]):
        if min(np.sum(y[:, j] == 0), np.sum(y[:, j] == 1)) < min_class_count:
            details.append({"finding_index": j, "shrinkage_strength": 0., "reason": "insufficient_fit_labels"})
            continue
        scores = np.full((n_splits, len(strengths)), np.nan)
        for fold, (tr, va) in enumerate(folds):
            xtr, xva = transformed_folds[fold]
            learned = _fit_pairwise_weights(y[tr, j], xtr[:, :, j], prior[:, j], seed=seed + 100*j + fold)
            for k, strength in enumerate(strengths):
                w = (1-strength) * prior[:, j] + strength * learned
                pred = xva[:, :, j] @ w
                scores[fold, k] = _AUCCache.build(y[va, j], pred).auc(np.ones(len(va)))
        # Every inner fold must support the class. Prefer smaller deviations on ties.
        if not np.isfinite(scores).all():
            details.append({"finding_index": j, "shrinkage_strength": 0., "reason": "unevaluable_inner_fold"})
            continue
        means = scores.mean(axis=0)
        best = int(np.argmax(means))
        strength = float(strengths[best])
        learned = _fit_pairwise_weights(y[:, j], full_fit[:, :, j], prior[:, j], seed=seed + 100*j + 99)
        chosen[:, j] = (1-strength) * prior[:, j] + strength * learned
        details.append({"finding_index": j, "shrinkage_strength": strength,
                        "inner_mean_auc": means.tolist(), "reason": "inner_group_validation"})
    return chosen, details


def make_group_partitions(groups, *, seed=2026, report_fraction=.25, gate_fraction=.25):
    """Pre-register one split using patient IDs, independently of any scores.

    Fractions refer to patient groups, not exact study counts. If rare targets
    are unevaluable, retain the prior and collect more labeled validation data;
    do not rerun seeds until a more favorable score appears.
    """
    groups = _groups(groups, len(groups))
    if not (0 < report_fraction < 1 and 0 < gate_fraction < 1 - report_fraction):
        raise ValueError("Need positive fit, gate, and report fractions.")
    if len(np.unique(groups)) < 12:
        raise ValueError("At least 12 patient groups are needed for fit/gate/report splits.")
    splitter = GroupShuffleSplit(n_splits=1, test_size=report_fraction, random_state=seed)
    development, report = next(splitter.split(np.zeros(len(groups)), groups=groups))
    splitter = GroupShuffleSplit(n_splits=1, test_size=gate_fraction/(1-report_fraction), random_state=seed+1)
    a, b = next(splitter.split(np.zeros(len(development)), groups=groups[development]))
    result = {"fit": development[a], "gate": development[b], "report": report}
    for name, idx in result.items():
        if not len(idx):
            raise ValueError(f"Empty {name} partition.")
    if any(set(groups[result[a]]) & set(groups[result[b]]) for a, b in [("fit","gate"),("fit","report"),("gate","report")]):
        raise AssertionError("Patient partition overlap.")
    return result


def fit_validate_blend(
    y, predictions, groups, *, labels=None, model_names=None, prior_weights=None,
    seed=2026, n_bootstrap=5000, familywise_alpha=.05, min_gate_gain=.0005,
    minimum_class_count=20, prediction_transform="cohort_rank01", baseline_model=None,
) -> dict:
    """Fit -> one-time per-class acceptance gate -> untouched final report.

    Assumes all base-model training excludes this entire cohort. Use
    run_validation_bundle() for provenance checks. Inner group folds select
    shrinkage on fit data. A candidate must beat the prior on its disjoint gate
    and have a positive one-sided cluster-bootstrap lower bound, Bonferroni
    adjusted across all findings. The report cohort is evaluated exactly once;
    it NEVER influences weights or per-finding acceptance, and its result may
    be negative. It controls the final release decision in export_results().

    Repeatedly comparing new designs against this same gate/report eventually
    overfits them. Version/freeze these cohorts and reserve a fresh outer test
    set for substantial new model families. Confidence limits are approximate.
    """
    y = _check_y(y)
    p = _check_predictions(predictions)
    if p.ndim != 3 or p.shape[0] != len(y) or p.shape[2] != y.shape[1] or p.shape[1] < 1:
        raise ValueError("Predictions must have shape [study, model, finding].")
    groups = _groups(groups, len(y))
    labels = list(labels or TARGETS)
    names = list(model_names or [f"model_{i}" for i in range(p.shape[1])])
    if len(labels) != y.shape[1] or len(names) != p.shape[1] or len(set(names)) != len(names):
        raise ValueError("Invalid label/model-name dimensions or duplicate model names.")
    if not 0 < familywise_alpha < 1 or min_gate_gain < 0 or minimum_class_count < 2:
        raise ValueError("Invalid acceptance-gate settings.")
    if baseline_model is None and "reference" in names:
        baseline_model = "reference"
    if baseline_model is not None and baseline_model not in names:
        raise ValueError("baseline_model must name one of the supplied models.")
    if prior_weights is None:
        prior = np.full(p.shape[1:], 1/p.shape[1])
        if baseline_model is not None:
            prior[:] = 0
            prior[names.index(baseline_model)] = 1
    else:
        prior = np.asarray(prior_weights, float)
    if baseline_model is not None:
        expected_prior = np.zeros(p.shape[1:])
        expected_prior[names.index(baseline_model)] = 1
        if not np.array_equal(prior, expected_prior):
            raise ValueError("An exact baseline_model requires a one-hot prior for that model.")
    blend_predictions(p, prior)  # Validate the prior before any fitting.
    parts = make_group_partitions(groups, seed=seed)
    tr, gate, report = (parts[k] for k in ["fit", "gate", "report"])
    candidate_weights, tuning = _select_weights(y[tr], p[tr], groups[tr], prior, seed=seed,
                                               prediction_transform=prediction_transform)
    gate_x = _transform_predictions(p[gate], prediction_transform)
    gate_baseline = blend_predictions(gate_x, prior)
    gate_candidate = blend_predictions(gate_x, candidate_weights)
    # One-sided lower tail alpha / C; CI remains approximate, not a formal
    # finite-sample error guarantee. Report CI below remains ordinary 95%.
    gate_result = compare_cluster_bootstrap(
        y[gate], gate_baseline, gate_candidate, groups[gate], labels=labels,
        n_bootstrap=n_bootstrap, seed=seed+2, confidence=1-familywise_alpha/len(labels),
    )
    tail_support = n_bootstrap * familywise_alpha / len(labels)
    accepted, weights, reasons = [], prior.copy(), {}
    for j, label in enumerate(labels):
        delta = gate_result["per_class"][label]
        counts = gate_result["baseline"]["per_class"][label]
        enough = min(counts["positive"], counts["negative"]) >= minimum_class_count
        lower = delta["one_sided_lower"]
        gain = delta["delta_auc"]
        accept = bool(enough and tail_support >= 20 and lower is not None and lower > 0
                      and gain is not None and gain >= min_gate_gain)
        if accept:
            accepted.append(label)
            weights[:, j] = candidate_weights[:, j]
        reasons[label] = {"accepted": accept, "enough_gate_labels": bool(enough),
                          "enough_bootstrap_tail_samples": bool(tail_support >= 20),
                          "gate_delta_auc": gain, "gate_one_sided_lower": lower}
    report_x = _transform_predictions(p[report], prediction_transform)
    report_baseline = blend_predictions(report_x, prior)
    report_final = blend_predictions(report_x, weights)
    report_result = compare_cluster_bootstrap(
        y[report], report_baseline, report_final, groups[report], labels=labels,
        n_bootstrap=n_bootstrap, seed=seed+3, confidence=.95,
    )
    partition_records = {
        name: {"studies": len(idx), "patient_groups": len(np.unique(groups[idx])),
               "row_indices": idx.tolist()} for name, idx in parts.items()
    }
    report_macro = report_result["macro"]
    return {
        "schema_version": 1, "labels": labels, "model_names": names,
        "weight_layout": "model_by_label", "prediction_transform": prediction_transform,
        "baseline_model": baseline_model,
        "prior_weights": prior.tolist(), "candidate_weights": candidate_weights.tolist(),
        "weights": weights.tolist(), "accepted_findings": accepted,
        "seed": seed, "partitions": partition_records,
        "tuning": tuning, "acceptance": reasons, "gate_comparison": gate_result,
        "untouched_report": report_result,
        "report_supports_macro_improvement": bool(
            report_macro["ci_low"] is not None and report_macro["ci_low"] > 0),
        "report_was_not_used_to_select_weights": True,
        "report_used_for_final_release_decision": True,
        "warning": "No leaderboard or private-test score is guaranteed. Do not reuse the report to tune further variants.",
    }


def run_validation_bundle(bundle: ValidationBundle, *, allow_exploratory_oof=False, **kwargs) -> dict:
    """Block optimistic OOF-as-outer-holdout claims unless explicitly exploratory."""
    if not bundle.independent_cohort and not allow_exploratory_oof:
        raise ValueError(
            "This bundle contains OOF predictions. A new split of OOF rows is not a fully "
            "independent outer model validation. Supply predictions for a cohort excluded "
            "from every base model's training, or explicitly allow exploratory OOF analysis."
        )
    result = fit_validate_blend(bundle.y, bundle.predictions, bundle.groups,
                                labels=bundle.labels, model_names=bundle.model_names, **kwargs)
    result["provenance"] = bundle.provenance
    result["validation_status"] = "heldout_cohort" if bundle.independent_cohort else "exploratory_oof"
    result["deployable_calibration"] = bool(bundle.independent_cohort and bundle.expert_truth
                                             and result["accepted_findings"]
                                             and result["report_supports_macro_improvement"])
    if not bundle.expert_truth:
        result["validation_status"] = "exploratory_labels" if bundle.independent_cohort else "exploratory_oof"
        result["report_supports_macro_improvement"] = False
        result["warning"] = (
            "Truth is not declared expert-only. Report-derived or mixed labels measure agreement "
            "with the labeling process, not necessarily the expert hidden-test target. "
            "No deployable calibration is exported."
        )
    if not bundle.independent_cohort:
        result["report_supports_macro_improvement"] = False
        result["warning"] = (
            "Exploratory OOF stacking only: base-model training crosses the new meta-split. "
            "These intervals do not certify independent generalization. No deployable "
            "calibration is exported; regenerate a fully excluded validation cohort."
        )
    for record in result["partitions"].values():
        record["study_ids"] = bundle.ids[record["row_indices"]].tolist()
    return result


def export_results(result: dict, output_dir: str | Path) -> list[Path]:
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    report = output / "validation_report.json"
    report.write_text(json.dumps(result, indent=2, allow_nan=False) + "\n")
    paths = [report]
    calibration_path = output / "calibration.json"
    if result.get("deployable_calibration"):
        calibration = {key: result[key] for key in [
            "schema_version", "labels", "model_names", "weight_layout", "prediction_transform",
            "weights", "accepted_findings", "validation_status", "provenance",
            "report_supports_macro_improvement", "warning", "baseline_model",
        ]}
        calibration_path.write_text(json.dumps(calibration, indent=2, allow_nan=False) + "\n")
        paths.append(calibration_path)
    else:
        # Reusing an output directory must not leave an obsolete passed recipe
        # beside a new failed/exploratory report.
        calibration_path.unlink(missing_ok=True)
    return paths


def apply_recipe(frames: dict[str, pd.DataFrame], recipe: dict | str | Path,
                 ids: Sequence[str]) -> pd.DataFrame:
    """Apply an accepted same-finding recipe to whole-cohort candidate CSVs.

    No cross-finding formula is allowed. Every named model must cover exactly
    the expected unique study IDs with finite [0,1] values in all 12 columns.
    With baseline_model='reference', rejected findings are copied bit-for-bit
    from that reference DataFrame. An absent optional source is an error;
    dropping a model or renormalizing weights would change the validated rule.
    """
    if isinstance(recipe, (str, Path)):
        recipe = json.loads(Path(recipe).read_text())
    if recipe.get("schema_version") != 1 or recipe.get("labels") != TARGETS:
        raise ValueError("Recipe schema or label order mismatch.")
    if recipe.get("validation_status") != "heldout_cohort" or recipe.get("report_supports_macro_improvement") is not True:
        raise ValueError("Recipe did not pass independent held-out release checks.")
    provenance = recipe.get("provenance", {})
    if (provenance.get("all_base_models_exclude_entire_cohort") is not True
            or provenance.get("truth_label_source") != "expert_manual"):
        raise ValueError("Recipe lacks declared complete model exclusion and expert-label provenance.")
    if recipe.get("weight_layout") != "model_by_label":
        raise ValueError("Unsupported recipe weight layout.")
    ids = [str(uid) for uid in ids]
    if not ids or len(ids) != len(set(ids)) or any(not uid.strip() for uid in ids):
        raise ValueError("Expected study IDs must be unique and nonempty.")
    names = recipe["model_names"]
    if not names or len(names) != len(set(names)):
        raise ValueError("Recipe model names must be nonempty and unique.")
    arrays = []
    for name in names:
        if name not in frames:
            raise ValueError(f"Missing required candidate source: {name}.")
        frame = frames[name].copy()
        if frame.columns.tolist() != ["StudyInstanceUID", *TARGETS]:
            raise ValueError(f"{name}: candidate columns/order mismatch.")
        if frame.StudyInstanceUID.isna().any():
            raise ValueError(f"{name}: missing study ID.")
        frame["StudyInstanceUID"] = frame.StudyInstanceUID.astype(str)
        if frame.StudyInstanceUID.duplicated().any() or set(frame.StudyInstanceUID) != set(ids):
            raise ValueError(f"{name}: extra, missing, or duplicate study IDs.")
        arrays.append(_check_predictions(frame.set_index("StudyInstanceUID").loc[ids, TARGETS].to_numpy(float)))
    predictions = np.stack(arrays, axis=1)
    output = blend_predictions(_transform_predictions(predictions, recipe["prediction_transform"]), recipe["weights"])
    baseline = recipe.get("baseline_model")
    accepted = recipe.get("accepted_findings", [])
    if len(accepted) != len(set(accepted)) or not set(accepted).issubset(TARGETS):
        raise ValueError("Invalid accepted_findings list.")
    if baseline is not None:
        if baseline not in names:
            raise ValueError("Recipe baseline source is absent.")
        baseline_idx = names.index(baseline)
        weights = np.asarray(recipe["weights"])
        for j, label in enumerate(TARGETS):
            if label not in accepted:
                expected = np.zeros(len(names)); expected[baseline_idx] = 1
                if not np.array_equal(weights[:, j], expected):
                    raise ValueError("Rejected findings must preserve the exact baseline weights.")
                output[:, j] = predictions[:, baseline_idx, j]
    return pd.DataFrame({"StudyInstanceUID": ids, **{label: output[:, j] for j, label in enumerate(TARGETS)}})


def example_manifest() -> dict:
    """Relative table paths resolve against the manifest's own directory."""
    return {
        "schema_version": 1,
        "truth_csv": "validation_labels.csv",
        "truth_label_source": "expert_manual",
        "id_column": "StudyInstanceUID",
        "group_column": "PatientID",
        "labels": TARGETS,
        "models": [
            {"name": "reference", "prediction_csv": "reference_heldout.csv",
             "prediction_role": "heldout", "training_groups_complete": True,
             "training_groups": ["REPLACE_WITH_EVERY_GROUP_THAT_INFLUENCED_MODEL_TRAINING"]},
            {"name": "new_model", "prediction_csv": "new_model_heldout.csv",
             "prediction_role": "heldout", "training_groups_complete": True,
             "training_groups": ["REPLACE_WITH_EVERY_GROUP_THAT_INFLUENCED_MODEL_TRAINING"]},
        ],
    }


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("manifest", nargs="?")
    parser.add_argument("--write-example", type=Path)
    parser.add_argument("--output-dir", type=Path, default=Path("validation_output"))
    parser.add_argument("--n-bootstrap", type=int, default=5000)
    parser.add_argument("--seed", type=int, default=2026)
    parser.add_argument("--allow-exploratory-oof", action="store_true")
    args = parser.parse_args()
    if args.write_example:
        args.write_example.write_text(json.dumps(example_manifest(), indent=2) + "\n")
        print(f"Wrote editable example: {args.write_example}")
        return
    if not args.manifest:
        parser.error("Provide a manifest or --write-example PATH.")
    bundle = load_validation_bundle(args.manifest)
    result = run_validation_bundle(bundle, n_bootstrap=args.n_bootstrap, seed=args.seed,
                                   allow_exploratory_oof=args.allow_exploratory_oof)
    paths = export_results(result, args.output_dir)
    print(json.dumps({"status": result["validation_status"],
                      "accepted_findings": result["accepted_findings"],
                      "untouched_report_macro": result["untouched_report"]["macro"],
                      "files": [str(path) for path in paths]}, indent=2, allow_nan=False))


if __name__ == "__main__":
    main()
'''
import importlib.util as _fixed_importlib
import sys as _fixed_sys
from pathlib import Path
_fixed_module_path = Path(FIXED_RUN['output_root']) / 'validation_tools.py'
_fixed_module_path.parent.mkdir(parents=True, exist_ok=True)
_fixed_module_path.write_text(VALIDATION_MODULE_SOURCE)
_fixed_spec = _fixed_importlib.spec_from_file_location('validation_tools', _fixed_module_path)
_fixed_module = _fixed_importlib.module_from_spec(_fixed_spec)
_fixed_sys.modules[_fixed_spec.name] = _fixed_module
_fixed_spec.loader.exec_module(_fixed_module)


In [ ]:
from validation_tools import apply_recipe
_selected_name = FIXED_RUN['submission_candidate']
if _selected_name not in CANDIDATE_FRAMES:
    raise ValueError('The requested candidate is missing from this prediction bundle')
SELECTED_PREDICTIONS = CANDIDATE_FRAMES[_selected_name].copy()
SELECTED_RECIPE = {'candidate': _selected_name, 'score_status': 'UNMEASURED'}
if FIXED_RUN['fusion_recipe']:
    SELECTED_RECIPE = json.loads(Path(FIXED_RUN['fusion_recipe']).read_text())
    SELECTED_PREDICTIONS = apply_recipe(CANDIDATE_FRAMES, SELECTED_RECIPE, _RSNA_TEST_IDS)
SELECTED_PREDICTIONS.to_csv(FIXED_OUTPUT / 'selected_candidate.csv', index=False)


## 5. Final publication or replay receipt


In [ ]:
FINAL_RECEIPT = {
    'status': 'COMPLETE', 'notebook': 'RSNA_Knee_Fixed_GPU_v2',
    'mode': FIXED_RUN['mode'], 'selected_recipe': SELECTED_RECIPE,
    'target_auc': 0.990, 'measured_auc': None,
    'score_claim': 'NO_KAGGLE_SCORE_FOR_THIS_NOTEBOOK',
    'extra_model_forward_passes': 0, 'runtime_equality_measured': False,
    'ordered_reference_stage_sha256': FIXED_BUDGET['ordered_cell_hashes_sha256'],
    'prediction_bundle_sha256': replay.sha256_file(_bundle_path),
    'public_preview_sized': bool(PREDICTION_BUNDLE.manifest['core']['public_preview_sized']),
    'selected_csv_sha256': delivery_sha(FIXED_OUTPUT / 'selected_candidate.csv'),
}
if FIXED_RUN['mode'] == 'inference':
    for cache_file in list(_RSNA_CACHE_FILES):
        Path(cache_file).unlink(missing_ok=True)
    FINAL_RECEIPT['elapsed_seconds'] = time.time() - T0
    FINAL_PATH = delivery_publish(SELECTED_PREDICTIONS, _RSNA_TEST_IDS,
                                  DELIVERY_CONTEXT, FINAL_RECEIPT)
    shutil.copyfile('/kaggle/working/run_complete.json', FIXED_OUTPUT / 'run_complete.json')
    print('Competition file:', FINAL_PATH)
else:
    FINAL_RECEIPT['model_stages_executed'] = len(EXECUTED_REFERENCE_STAGES)
    delivery_json(FIXED_OUTPUT / 'replay_complete.json', FINAL_RECEIPT)
    print('CPU replay complete. Selected historical-cohort CSV:', FIXED_OUTPUT / 'selected_candidate.csv')
print('Save this output directory:', FIXED_OUTPUT)


## Input checklist and interpretation

Inference uses the same attached assets as the audited fast reference:

- Competition input and `rsna-knee-weights` (20 checked DINO members).
- Meta Research DINOv2 PyTorch/small/1 model files.
- `knee-mri-fold-weights`, folds `m_f0.pt` through `m_f4.pt`.
- `raptor-knee-maxspan`, `raptor-knee-native384dense`, `raptor-knee-native384`.
- The inherited hash-resolved RadImageNet encoder and E10/E13 head bundles.
- Residual CoAt `coat_resgated_ep10_top3_manifest.json`, three checkpoints/runtime.
- D4 `coatnet_pairfilm_manifest.json`, parent checkpoint, adapter and runtime.
- Pinned local OpenCV 4.12.0.88 and timm 1.0.22 wheels.

No Global96 input is used. The inherited input IDs remain in notebook metadata;
importing an IPYNB does not guarantee Kaggle attaches all datasets automatically.

**Why 0.990 is not promised:** the official score averages 12 ROC AUCs. Raising
an actual 0.942 to 0.990 would remove about 82.8% of the remaining mean pairwise
ranking error, counting ties by the usual half-credit convention. The supplied
files and inspected public receipts do not demonstrate that correction. This
is an experimental notebook designed to test a precise hypothesis at the same
model workload, with failed-family and wrong-cohort protections.

At the verified September 21, 2026 snapshot, displayed leaderboard scores were
0.958 first, 0.956 second and 0.955 for ranks 3–7. Those rounded, changing public
scores cannot guarantee future or private rank. See the
[live leaderboard](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/leaderboard).

Credits: the supplied notebooks and public work by Mattia Angeli, maverickss26,
Jiwei Liu, Berat Erol Celik, Dread Development, Pilkwang Kim, Sofia Anjenje,
Antoine G., prvsiyan, Marwan Mahmoud, renta.k and Anvith Pothula.
Upstream data/model licenses remain applicable.
